# Teacher Model — Notebook Overview

This notebook trains and evaluates the **teacher model** of the teacher-student knowledge
distillation pipeline for the 10-participant multimodal VR emotion study.

---

## Main Purpose

The teacher is a **signal-rich** model: it consumes the full 70-channel facial Action Unit (AU)
stream available from the Meta Quest Pro face tracker. Its role is twofold:

1. Establish the upper bound of emotion-recognition performance achievable from facial AUs alone.
2. Produce **soft labels** (class-probability vectors) that are exported and consumed by the
   student notebook for RQ2 knowledge distillation.

The student model, by contrast, is signal-sparse (eye gaze, head pose, upper-face AUs only) and is
trained in a separate notebook.

---

## Model

A **Mixture-of-Experts Temporal Convolutional Network (MoE-TCN)**:

- 3 TCN experts (natural / acted / shared), each a stack of dilated causal residual blocks
- A Top-2 gate with temperature annealing, load-balancing loss, and entropy regularisation
- Input shape `(SEQ_LENGTH=5, 70)` — 0.5 s of AU signal at 10 Hz
- Output: 7-class Ekman emotion softmax

---

## Notebook Structure

### Part 1: Setup and Environment
Google Drive mount, dependency installation, imports, and global random seeds.

### Part 2: Self-Report Validity Audit
Compares each participant's free-text self-report against the emotion that was actually elicited,
and produces the exclusion lists used later to filter unreliable recordings.

### Part 3: Labels, Emotion Classes, and the AU Feature Set
Emotion enum, data-source enum, label encoder, and the canonical 70-AU channel list.

### Part 4: Paths and Data-Loading Helpers
Output directories plus the windowing, thresholding, and participant-exclusive splitting helpers.

### Part 5: Dataset Construction
Loads the three data sources (our natural recordings, our acted recordings, EmojiHeroVR), applies the
self-report filter, builds participant-exclusive 70/10/20 splits, and stacks everything into tensors.

### Part 6: Training and Evaluation Helpers
Scoring and training wrappers shared by all teacher variants.

### Part 7: Teacher Model — MoE-TCN
Architecture definition, hyperparameters, training run, and result export.

### Part 8: Fine-Tuning Variant
Loads the original teacher and fine-tunes it on the new participants only, as an alternative to
joint retraining.

### Part 9: Teacher Variant Comparisons
Three-way and five-way comparisons across all teacher variants and all test sets.

### Part 10: Teacher Soft Labels for Distillation
Exports the soft labels and hard labels consumed by the student notebook (RQ2).

### Part 11: Original vs. +10-Participant Teacher (head-to-head)
Full metric suite, confusion matrices, per-class F1 plots, and reproducible raw predictions.

---

# Part 1: Setup and Environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:







%pip install keras-tcn

In [ ]:
from enum import Enum
from functools import reduce
from glob import glob
from typing import Generator, Callable

import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.calibration import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import shuffle as sk_shuffle


import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras import layers



from tcn import TCN

import matplotlib.pyplot as plt

import os
import random
import tempfile

RANDOM_STATE = 42
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Part 2: Self-Report Validity Audit

Participants reported, in a Google Form, which emotion they felt after each video. This section
compares those free-text self-reports against the emotion that the stimulus was designed to elicit.

Recordings where the intended emotion never appears in the self-report are treated as invalid
supervision and are excluded downstream (see Section 5.5). This is a data-validity step, not a
modelling step.

## 2.1 Self-Report vs. Elicited Emotion

Maps each participant folder to their Google Form submission, reads the per-video emotion order from
`emotion_flow.txt`, and parses the free-text self-report, keeping only the seven Ekman labels.

The cell prints a participant x emotion match grid and writes two files:

- `all_pairs.csv` — every (participant, video, intended emotion, reported emotion) pair
- `records_to_remove.csv` — the pairs where the intended emotion was never reported

The exclusion lists in Section 5.5 are transcribed from this output.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
NEW_DATA_ROOT = Path("/content/drive/MyDrive/Multi-modal Detection for Meta Quest 3/Data Collection (New)")
FORMS_PATH = "/content/drive/MyDrive/Multi-modal Detection for Meta Quest 3/Forms/Forms-15_participants.csv"

In [ ]:


NEW_DATA_ROOT = Path("/content/drive/MyDrive/Data Collection (New)")
#NEW_DATA_ROOT = Path("/content/drive/MyDrive/Multi-modal Detection for Meta Quest 3/Data Collection (New)")
FORMS_PATH    = "/content/drive/MyDrive/Forms/Forms-15_participants.csv"
#FORMS_PATH = "/content/drive/MyDrive/Multi-modal Detection for Meta Quest 3/Forms/Forms-15_participants.csv"

PARTICIPANT_TO_FORM_TIMESTAMP = {
    'P35': '6/3/2026 10:00:43',
    'P36': '6/3/2026 12:26:47',
    'P37': '6/4/2026 11:44:02',
    'P38': '6/5/2026 11:25:10',
    'P39': '6/5/2026 12:45:23',
    'P40': '6/10/2026 12:53:50',
    'P41': '6/10/2026 13:15:20',
    'P42': '6/11/2026 11:02:32',
    'P43': '6/11/2026 13:11:16',
    'P44': '6/16/2026 13:49:57',
    'P45': '7/01/2026 14:11:49',
    'P46': '7/02/2026 12:06:41',
    'P47': '7/15/2026 10:45:26',
    'P48': '7/17/2026 15:02:24',
    'P49': '7/24/2026 15:36:10',
}

TOLERANCE_SEC = 5  # CSV timestamps drift a second or two from the mapping

forms = pd.read_csv(FORMS_PATH)

# CSV side: strip the "AST" tz suffix, then parse
csv_raw = forms['Timestamp'].astype(str).str.replace(r'\s*[A-Z]{3}\s*$', '', regex=True)
csv_dt  = pd.to_datetime(csv_raw, errors='coerce')  # handles "2026/06/03 10:00:42 AM"

vid_root = NEW_DATA_ROOT / "VideoRecordings"

header = f"{'Participant':<18} {'folder':<8} {'flow.txt':<10} {'form_ts':<9} {'matched CSV time'}"
print(header)
print("-" * len(header))

missing = []
for p, form_ts in PARTICIPANT_TO_FORM_TIMESTAMP.items():
    folder_ok = (vid_root / p).is_dir()
    flow_ok   = (vid_root / p / "emotion_flow.txt").is_file()

    target = pd.to_datetime(form_ts, errors='coerce')  # mapping side, dayfirst not needed (M/D/Y)
    if pd.isna(target):
        ts_ok, matched_str = False, "(bad mapping ts)"
    else:
        deltas = (csv_dt - target).abs()
        nearest = deltas.min()
        ts_ok = pd.notna(nearest) and nearest.total_seconds() <= TOLERANCE_SEC
        matched_str = str(csv_dt[deltas.idxmin()]) if ts_ok else f"(nearest off by {nearest.total_seconds():.0f}s)"

    print(f"{p:<18} {'✓' if folder_ok else '✗':<8} "
          f"{'✓' if flow_ok else '✗':<10} {'✓' if ts_ok else '✗':<9} {matched_str}")

    if not (folder_ok and flow_ok and ts_ok):
        missing.append(p)

print("-" * len(header))
print(f"{len(PARTICIPANT_TO_FORM_TIMESTAMP)} checked, "
      f"{len(PARTICIPANT_TO_FORM_TIMESTAMP) - len(missing)} fully present, {len(missing)} with issues")
if not missing:
    print(f"\nAll {len(PARTICIPANT_TO_FORM_TIMESTAMP)} participants verified — safe to proceed.")

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ── Participant → Form timestamp mapping (user-verified) ─────────────────────
PARTICIPANT_TO_FORM_TIMESTAMP = {
    'P35': '6/3/2026 10:00:43',
    'P36': '6/3/2026 12:26:47',
    'P37': '6/4/2026 11:44:02',
    'P38': '6/5/2026 11:25:10',
    'P39': '6/5/2026 12:45:23',
    'P40': '6/10/2026 12:53:50',
    'P41': '6/10/2026 13:15:20',
    'P42': '6/11/2026 11:02:32',
    'P43': '6/11/2026 13:11:16',
    'P44': '6/16/2026 13:49:57',
    'P45': '7/01/2026 14:11:49',
    'P46': '7/02/2026 12:06:41',
    'P47': '7/15/2026 10:45:26',
    'P48': '7/17/2026 15:02:24',
    'P49': '7/24/2026 15:36:10',
}

# ── Only 7 Ekman emotions — no aliases, no mapping of non-Ekman labels ────────
EKMAN = {e.lower(): e for e in ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']}

def parse_reported(raw):
    """Extract ALL Ekman emotions mentioned in free-text self-report."""
    if pd.isna(raw) or str(raw).strip().lower() in ('n/a', 'n/a video failed', ''):
        return []
    text = str(raw).lower()
    return list({v for k, v in EKMAN.items() if k in text})

# ── Load emotion_flow files ───────────────────────────────────────────────────
NEW_DATA_ROOT = Path("/content/drive/MyDrive/Data Collection (New)")
FORMS_PATH    = "/content/drive/MyDrive/Forms/Forms-15_participants.csv"

flow_data = {}
for flow_file in sorted((NEW_DATA_ROOT / "VideoRecordings").glob("*/emotion_flow.txt")):
    participant = flow_file.parent.name
    lines = flow_file.read_text(encoding='utf-8-sig').strip().splitlines()
    video_map = {}
    for line in lines[1:]:
        m = re.match(r'(\d+)\. Video - (\w+) -', line.strip())
        if m:
            video_map[int(m.group(1))] = m.group(2)
    flow_data[participant] = video_map

# ── Load Forms.csv ────────────────────────────────────────────────────────────
forms    = pd.read_csv(FORMS_PATH)
emo_cols = [c for c in forms.columns if c.startswith('What emotion')]

# CSV timestamps look like '2026/06/03 10:00:42 AM AST' — strip tz, parse to datetime
_csv_dt = pd.to_datetime(
    forms['Timestamp'].astype(str).str.replace(r'\s*[A-Z]{3}\s*$', '', regex=True),
    errors='coerce',
)
TOLERANCE_SEC = 5  # mapping timestamps drift a second or two from the CSV

def find_form_idx(form_ts):
    target = pd.to_datetime(form_ts, errors='coerce')
    if pd.isna(target):
        return None
    deltas = (_csv_dt - target).abs()
    if pd.isna(deltas.min()) or deltas.min().total_seconds() > TOLERANCE_SEC:
        return None
    return int(deltas.idxmin())

# ── Build results ─────────────────────────────────────────────────────────────
results = []
for participant, form_ts in PARTICIPANT_TO_FORM_TIMESTAMP.items():
    form_idx = find_form_idx(form_ts)
    if form_idx is None:
        print(f"WARNING: form timestamp not found for {participant}: {form_ts}")
        continue
    form_row = forms.iloc[form_idx]
    video_map = flow_data.get(participant, {})
    for video_idx in range(1, 8):
        actual = video_map.get(video_idx)
        if actual is None:
            continue
        reported_raw = form_row[emo_cols[video_idx - 1]]
        reported     = parse_reported(reported_raw)
        match        = (actual in reported) if reported else False
        results.append({
            'participant'   : participant,
            'video_idx'     : video_idx,
            'actual_emotion': actual,
            'reported_raw'  : str(reported_raw),
            'reported_parsed': reported,
            'match'         : match,
        })

res_df = pd.DataFrame(results)

# ── Grid table ────────────────────────────────────────────────────────────────
EMOTIONS = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
col_w    = 11
header   = f"{'Participant':<20} " + "".join(f"{e:<{col_w}}" for e in EMOTIONS) + "Match"
sep      = "-" * len(header)

print("=" * len(header))
print("SELF-REPORT vs ACTUAL EMOTION")
print("✓ = match   ✗ = mismatch   · = not shown to participant")
print("=" * len(header))
print(header)
print(sep)

mismatch_records = []

for participant in sorted(PARTICIPANT_TO_FORM_TIMESTAMP.keys()):
    sub = res_df[res_df['participant'] == participant]
    emotion_marks = {}
    for _, r in sub.iterrows():
        emotion_marks[r['actual_emotion']] = '✓' if r['match'] else '✗'
        if not r['match']:
            mismatch_records.append({
                'participant'   : participant,
                'actual_emotion': r['actual_emotion'],
                'reported_raw'  : r['reported_raw'],
                'reported_parsed': r['reported_parsed'],
            })

    row_str  = f"{participant:<20} "
    row_str += "".join(f"{emotion_marks.get(e, '·'):<{col_w}}" for e in EMOTIONS)
    n_match  = sub['match'].sum()
    row_str += f"{n_match}/{len(sub)}"
    print(row_str)

print(sep)
total   = len(res_df)
matched = int(res_df['match'].sum())
print(f"Total: {total}  |  Match: {matched} ({100*matched/total:.1f}%)  |  Mismatch: {total-matched} ({100*(total-matched)/total:.1f}%)")

# ── Mismatch detail table ─────────────────────────────────────────────────────
mismatch_df = pd.DataFrame(mismatch_records)

print()
print("=" * 80)
print("RECORDS TO REMOVE (actual emotion NOT in self-report)")
print("=" * 80)
print(f"{'Participant':<20} {'Remove Emotion':<16} {'Self-Report (raw)'}")
print("-" * 80)
for _, r in mismatch_df.iterrows():
    print(f"{r['participant']:<20} {r['actual_emotion']:<16} {r['reported_raw']}")

print()
print(f"Total records to remove: {len(mismatch_df)}")
print()
print("By emotion:")
print(mismatch_df.groupby('actual_emotion').size().sort_values(ascending=False).to_string())
print()
print("By participant:")
print(mismatch_df.groupby('participant').size().sort_values(ascending=False).to_string())

# ── Export ────────────────────────────────────────────────────────────────────
out_dir = NEW_DATA_ROOT / "self_report_filter"
out_dir.mkdir(parents=True, exist_ok=True)

res_df.to_csv(out_dir / "all_pairs.csv", index=False)
mismatch_df[['participant', 'actual_emotion', 'reported_raw']].to_csv(
    out_dir / "records_to_remove.csv", index=False
)
print(f"\nSaved to: {out_dir}")
print("  - all_pairs.csv")
print("  - records_to_remove.csv")

# Part 3: Labels, Emotion Classes, and the AU Feature Set

## 3.1 Emotion Classes, Data Sources, and AU Channels

Defines the seven Ekman emotions, the three data sources (our natural recordings, our acted
recordings, EmojiHeroVR), and the label encoder.

`AU_NAMES` is the teacher's 70-channel input: the full Quest Pro face-tracking blendshape set,
including the ten eye AUs. This full set is what makes the teacher signal-rich relative to the
student.

In [ ]:

class Emotion(Enum):
    ANGER = "Anger"
    DISGUST = "Disgust"
    FEAR = "Fear"
    HAPPINESS = "Happiness"
    NEUTRAL = "Neutral"
    SADNESS = "Sadness"
    SURPRISE = "Surprise"

class DataType(Enum):
  OUR_NATURAL= "Our_natural"
  OUR_ACTED= "Our_acted"
  EMOJIHERO= "Emojihero"


le = LabelEncoder().fit([emo.value for emo in Emotion])


AU_NAMES: list[str] = [
    "BrowLowererL",
    "BrowLowererR",
    "CheekPuffL",
    "CheekPuffR",
    "CheekRaiserL",
    "CheekRaiserR",
    "CheekSuckL",
    "CheekSuckR",
    "ChinRaiserB",
    "ChinRaiserT",
    "DimplerL",
    "DimplerR",
    "EyesClosedL",
    "EyesClosedR",
    "EyesLookDownL",
    "EyesLookDownR",
    "EyesLookLeftL",
    "EyesLookLeftR",
    "EyesLookRightL",
    "EyesLookRightR",
    "EyesLookUpL",
    "EyesLookUpR",
    "InnerBrowRaiserL",
    "InnerBrowRaiserR",
    "JawDrop",
    "JawSidewaysLeft",
    "JawSidewaysRight",
    "JawThrust",
    "LidTightenerL",
    "LidTightenerR",
    "LipCornerDepressorL",
    "LipCornerDepressorR",
    "LipCornerPullerL",
    "LipCornerPullerR",
    "LipFunnelerLB",
    "LipFunnelerLT",
    "LipFunnelerRB",
    "LipFunnelerRT",
    "LipPressorL",
    "LipPressorR",
    "LipPuckerL",
    "LipPuckerR",
    "LipStretcherL",
    "LipStretcherR",
    "LipSuckLB",
    "LipSuckLT",
    "LipSuckRB",
    "LipSuckRT",
    "LipTightenerL",
    "LipTightenerR",
    "LipsToward",
    "LowerLipDepressorL",
    "LowerLipDepressorR",
    "MouthLeft",
    "MouthRight",
    "NoseWrinklerL",
    "NoseWrinklerR",
    "OuterBrowRaiserL",
    "OuterBrowRaiserR",
    "UpperLidRaiserL",
    "UpperLidRaiserR",
    "UpperLipRaiserL",
    "UpperLipRaiserR",
    "TongueTipInterdental",
    "TongueTipAlveolar",
    "TongueFrontDorsalPalate",
    "TongueMidDorsalPalate",
    "TongueBackDorsalVelar",
    "TongueOut",
    "TongueRetreat",
]

# EMO_TO_AU: dict[Emotion, list[str]] = {
#     Emotion.ANGER: ['CheekRaiser', 'TongueFrontDorsalPalate', 'MouthRight', 'MouthLeft', 'LipsToward', 'LipCornerDepressor', 'LipStretcher'],
#     Emotion.DISGUST: ['TongueRetreat', 'LipTightener', 'LipFunnelerB', 'EyesLookRight', 'CheekPuff', 'UpperLipRaiser', 'LipPucker'],
#     Emotion.FEAR: ['TongueTipAlveolar', 'EyesLookRight', 'LipStretcher', 'CheekRaiser', 'LipSuckT'],
#     Emotion.HAPPINESS: ['LipSuckB', 'LipStretcher', 'LipPucker', 'LipsToward', 'LipPressor', 'ChinRaiserT', 'LipSuckT', 'JawThrust', 'TongueBackDorsalVelar', 'CheekRaiser'],
#     Emotion.NEUTRAL: [],
#     Emotion.SADNESS: ['UpperLipRaiser', 'TongueFrontDorsalPalate', 'LowerLipDepressor', 'CheekRaiser', 'LipSuckB', 'TongueRetreat'],
#     Emotion.SURPRISE: ['TongueBackDorsalVelar', 'LowerLipDepressor', 'LipTightener', 'LipStretcher', 'TongueTipAlveolar', 'LipSuckB', 'UpperLidRaiser', 'TongueMidDorsalPalate', 'ChinRaiserT'],
# }

### Canonical class order

The label encoder sorts alphabetically, so the class index order is fixed as
Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise. This order is used for every confusion
matrix, classification report, and soft-label column throughout both notebooks.

In [ ]:
print("le.classes_ =", list(le.classes_))

### Window length

`SEQ_LENGTH = 5` frames at 10 Hz, i.e. a 0.5-second temporal window.

In [ ]:
SEQ_LENGTH=5

In [ ]:
print(AU_NAMES)

### Legacy check: 60-AU teacher (eye AUs excluded)

Retained for provenance only. This assertion belongs to the earlier 60-AU teacher, which excluded
the ten eye AUs to avoid overlap with the student's gaze channels. It will **fail** against the
current 70-AU list.

**Do not run this cell.** It is a candidate for the archive notebook; Section 3.2 is the active
check.

In [ ]:
EYE_AU_NAMES = [
    "EyesClosedL",
    "EyesClosedR",
    "EyesLookDownL",
    "EyesLookDownR",
    "EyesLookLeftL",
    "EyesLookLeftR",
    "EyesLookRightL",
    "EyesLookRightR",
    "EyesLookUpL",
    "EyesLookUpR",
]

print("Number of AU features after removing eye AUs:", len(AU_NAMES))
print(AU_NAMES)

for eye_au in EYE_AU_NAMES:
    assert eye_au not in AU_NAMES, f"{eye_au} is still inside AU_NAMES!"

### Active check: 70-AU teacher (eye AUs included)

Confirms that all ten eye AUs are present and that the teacher input width is exactly 70. The teacher
was expanded from 60 to 70 AUs so that eye-correlated AUs could be attributed correctly in the SHAP
gap decomposition (student notebook, Part 6).

In [ ]:
EYE_AU_NAMES = [
    "EyesClosedL",
    "EyesClosedR",
    "EyesLookDownL",
    "EyesLookDownR",
    "EyesLookLeftL",
    "EyesLookLeftR",
    "EyesLookRightL",
    "EyesLookRightR",
    "EyesLookUpL",
    "EyesLookUpR",
]

print("Number of AU features (including eye AUs):", len(AU_NAMES))
print(AU_NAMES)

for eye_au in EYE_AU_NAMES:
    assert eye_au in AU_NAMES, f"{eye_au} is missing from AU_NAMES!"

assert len(AU_NAMES) == 70, f"Expected 70 AU_NAMES, got {len(AU_NAMES)}"

# Part 4: Paths and Data-Loading Helpers

## 4.1 Data and Output Paths

`data_folder_path` points at the filtered per-participant AU recordings; `output_path` is the root
for every artefact produced by this notebook.

In [ ]:
data_folder_path = "/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/vr-face-au-study/filtered-dataset"
output_path = Path("/content/drive/MyDrive/outputs_15_participants_70AU-NOT_normalize_teacher_data-self_filtered_NATURAL_ONLY-EmojiHero_Fixed")

## 4.2 Output Subdirectories

Separate directories for saved model checkpoints and for predictions/metrics. Each downstream
section writes into its own subfolder so that no two experiments overwrite each other.

In [ ]:
models_path = f"{output_path}/models"
predictions_path = f"{output_path}/predictions"

os.makedirs(models_path, exist_ok=True)
os.makedirs(predictions_path, exist_ok=True)

print("Models path:", models_path)
print("Predictions path:", predictions_path)

## 4.4 Core Data-Loading Helpers

Defines the loading and windowing pipeline used for our own recordings:

- `data_files` / `all_files_list` — glob the per-participant CSVs for a given data source
  (`ACT_<emotion>.csv` for acted, `weights_<emotion>.csv` for natural)
- `subject_from_path` — extract the participant ID from the folder name; this is the grouping key
  that guarantees participant-exclusive splits
- `data_frame` — drop the timestamp column and reorder the AU columns to match `AU_NAMES` exactly
- `sep_process_one` — the **windowing rule**: a frame is marked active if any AU crosses the
  threshold (for Neutral, if *all* AUs stay below it); a 5-frame window is emitted only when it
  contains 3 consecutive active frames. Windows are non-overlapping (`step = SEQ_LENGTH`).
- `stack_sequences` / `to_numpy_labels` — convert window lists into `(N, 5, 70)` tensors
- `train_val_test_data_by_subject` — participant-exclusive 70/10/20 split via `StratifiedGroupKFold`
  (10 folds: 0-6 train, 7 validation, 8-9 test), with assertions that no participant appears in more
  than one split
- `mean_centered_max_scale` — normalize each AU independently within the current recording. The recording-level mean is subtracted from each feature, and the centered values are divided by the largest absolute centered value. This produces approximately zero-centered values within `[-1, 1]`.

Mean Centered Max Scale Normalization applied here:

```
Dataset            Function                            Normalization unit
-----------------  ----------------------------------  ---------------------------------------------------------
Original natural   sep_process_one()                   Applied independently to each CSV recording
Original acted     sep_process_one()                   Applied independently to each CSV recording
New participants   make_windows_from_new_recording()   Applied to each CSV recording after 60->10 Hz downsampling
EmojiHero          average_every_3_rows()              Applied independently to each parent_seq after 30->10 Hz downsampling
```

In [ ]:
def mean_centered_max_scale(df):
    """
    Apply mean-centered absolute-max scaling independently to each feature.

    For every feature column:
    1. Subtract the recording-level mean.
    2. Divide by the largest absolute centered value.

    The resulting values are approximately within [-1, 1].
    """
    centered = df - df.mean()

    absolute_max = centered.abs().max()

    scaled = centered / (absolute_max + 1e-8)

    return scaled

In [ ]:
def all_files_list(data_type: DataType):
    # Materialize generator -> list so we can split by files
    return list(data_files(data_type))

def forever(x):
    while True:
        yield x

def data_files_emo(emo: Emotion, data_type:DataType):
  if data_type == DataType.OUR_ACTED:
    yield from zip(
        glob(f"{data_folder_path}/*/ACT_{emo.value}.csv"),
        forever(emo),
    )
  elif data_type == DataType.OUR_NATURAL:
    yield from zip(
        glob(f"{data_folder_path}/*/weights_{emo.value}.csv"),
        forever(emo),
    )

def data_files(data_type:DataType):
    for emo in Emotion:
        yield from data_files_emo(emo, data_type)

# NEW: extract participant id from the folder between .../vr_filtered_clean/*/...
def subject_from_path(file_path: str) -> str:
    return os.path.basename(os.path.dirname(file_path))

def data_frame(file: str):
    df = pd.read_csv(file)

    itimes = df.iloc[:, 0].astype(int)
    df = df.iloc[:, 1:]

    missing = [au for au in AU_NAMES if au not in df.columns]

    if missing:
        raise ValueError(f"Missing AU columns: {missing}")

    df = df[AU_NAMES]

    return df, itimes


# Chunk the data from the file and give them in (chunk, label) pairs
def sep_process_one(file: str, emo: Emotion, thresh: Callable[[float], bool]):

    df, itimes = data_frame(file)                 # raw, for labeling
    #df_scaled = mean_centered_max_scale(df)       # per-recording, model input only

    labels = [0] * df.shape[0]
    try:
      emo_label = le.transform([emo.value])[0]
    except Exception as e:
      print(f"Error occured for file {file}")
      print(f"Emotion: {emo.value}")
      raise e

    for i in range(df.shape[0]):
      if emo == Emotion.NEUTRAL:
          au_values = df.iloc[i].tolist()
          if all(thresh(value) for value in au_values):
            labels[i]=1
      else:
        for au in AU_NAMES:
        # for au in EMO_TO_AU[emo]:
            if thresh(df[au][i]):
                labels[i] = 1
                break

    step = SEQ_LENGTH
    i, n = 0, df.shape[0] - SEQ_LENGTH

    pct = 0.2
    consecutive = 3

    def seq_sum(thresh):
        def fn(x, y):
            if x >= thresh:
                return thresh
            else:
                return 0 if y == 0 else x + y
        return fn

    while i <= n:
        count = reduce(seq_sum(consecutive), labels[i:i + SEQ_LENGTH])
        # if count >= SEQ_LENGTH * pct:
        if count == consecutive:
            yield (df[i:i + SEQ_LENGTH], emo_label)
            i += step
        else:
            i += 1

def sep_process(files: Generator[tuple[str, Emotion], None, None], thresh: dict[Emotion, Callable[[float], bool]]):
    for file, emo in files:
        yield from sep_process_one(file, emo, thresh[emo])


def stack_sequences(seq_list):
    """
    seq_list: list of pandas DataFrames of shape (SEQ_LENGTH, n_features)
    returns: np.ndarray of shape (N, SEQ_LENGTH, n_features)
    """
    return np.stack([df.values.astype(np.float32) for df in seq_list], axis=0)

def to_numpy_labels(labels_list):
    return np.asarray(labels_list, dtype=np.int64)

# NEW: subject-exclusive 70/10/20 split using StratifiedGroupKFold
def train_val_test_data_by_subject(
    thresh: dict[Emotion, Callable[[float], bool]],
    data_type: DataType,
    random_state: int = 42,
    n_splits: int = 10
):
    """
    Strict subject-exclusive split (participant in exactly one split),
    â‰ˆ70/10/20 by combining 10 StratifiedGroupKFold folds:
      - train: folds 0..6
      - val:   fold 7
      - test:  folds 8..9
    Stratifies by emotion at the file level while grouping by participant.
    Returns: X_train, X_val, X_test, y_train, y_val, y_test (lists)
    """
    all_files = all_files_list(data_type)
    if not all_files:
        raise RuntimeError("No files found. Check your glob path.")

    files, emos = zip(*all_files)

    # y = emotion labels for stratification (use your LabelEncoder if available)
    try:
        y = np.array([le.transform([e.value])[0] for e in emos], dtype=int)
    except Exception:
        # fallback to raw string labels if le not fitted here
        y = np.array([str(e.value) for e in emos])

    # groups = subject ids extracted from folder name
    groups = np.array([subject_from_path(p) for p in files])

    # build 10 folds stratified by emotion and grouped by subject
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = []
    dummy_X = np.zeros((len(y), 1))
    for _, test_idx in sgkf.split(dummy_X, y, groups=groups):
        folds.append(test_idx)
    if len(folds) != n_splits:
        raise RuntimeError("Unexpected number of folds produced.")

    # combine folds: 7 train, 1 val, 2 test
    train_idx = np.concatenate(folds[:7])
    val_idx   = folds[7]
    test_idx  = np.concatenate(folds[8:])

    # sanity: ensure subject exclusivity
    train_subjects = set(groups[train_idx])
    val_subjects   = set(groups[val_idx])
    test_subjects  = set(groups[test_idx])
    assert train_subjects.isdisjoint(val_subjects)
    assert train_subjects.isdisjoint(test_subjects)
    assert val_subjects.isdisjoint(test_subjects)

    # build split-specific file lists (tuples of (file, emo))
    train_files = [all_files[i] for i in train_idx]
    val_files   = [all_files[i] for i in val_idx]
    test_files  = [all_files[i] for i in test_idx]

    # chunk per split (no leakage)
    train_samples = list(sep_process(iter(train_files), thresh))
    val_samples   = list(sep_process(iter(val_files),   thresh))
    test_samples  = list(sep_process(iter(test_files),  thresh))

    if not train_samples:
        raise RuntimeError("No training sequences created. Adjust thresholds/windowing or verify data.")
    if not val_samples:
        raise RuntimeError("No validation sequences created. Adjust thresholds/windowing or verify data.")
    if not test_samples:
        raise RuntimeError("No test sequences created. Adjust thresholds/windowing or verify data.")

    X_train, y_train = zip(*train_samples)
    X_val,   y_val   = zip(*val_samples)
    X_test,  y_test  = zip(*test_samples)

    return list(X_train), list(X_val), list(X_test), list(y_train), list(y_val), list(y_test)

## 4.5 Helpers for the New Participant Recordings

The new participant recordings are stored in a different folder layout and were captured at 60 Hz.
These helpers bring them into the same format the teacher already expects: keep only the 70 AU
columns in `AU_NAMES`, downsample 60 Hz to 10 Hz, and emit `(SEQ_LENGTH, 70)` windows that can be
concatenated directly onto the existing data.

The teacher architecture and input shape are unchanged by this integration.

### Helper definitions

- `downsample_60hz_to_10hz_face_au` — coerces the AU columns to numeric, fills gaps, and averages
  every 6 frames (60 Hz -> 10 Hz)
- `new_collection_records` — walks `ActingRecordings/` and `VideoRecordings/` and returns one row per
  recording with participant, emotion, and domain (1 = acted, 0 = natural)
- `make_windows_from_new_recording` — applies the same threshold and 3-consecutive-frame rule as the
  original pipeline, so window selection is identical across data sources
- `load_new_collection_split` — expands a set of records into windows, emotion labels, and domain
  labels

These helpers keep the teacher model unchanged. They read the new participant CSVs, keep only the same 60 face AU columns in AU_NAMES, downsample 60 Hz to 10 Hz, create (SEQ_LENGTH, 60) windows, and return lists that can be appended to the previous data.

In [ ]:
from pathlib import Path

DOWNSAMPLE_FACTOR = 6  # 60 Hz -> 10 Hz


def downsample_60hz_to_10hz_face_au(df, au_names=AU_NAMES):
    """Keep only teacher face-AU features and convert new recordings from 60 Hz to 10 Hz."""
    missing = [c for c in au_names if c not in df.columns]
    if missing:
        raise ValueError(f"Missing face AU columns: {missing[:10]} ... total={len(missing)}")

    df = df[au_names].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.ffill().bfill().fillna(0.0)

    usable_rows = (len(df) // DOWNSAMPLE_FACTOR) * DOWNSAMPLE_FACTOR
    df = df.iloc[:usable_rows]

    return (
        df.groupby(np.arange(len(df)) // DOWNSAMPLE_FACTOR)
          .mean()
          .reset_index(drop=True)
    )


def new_collection_records(root=NEW_DATA_ROOT):
    """Return one row per new recording file with participant, emotion, and domain metadata."""
    rows = []
    valid_emotions = {e.value for e in Emotion}

    for recording_type in ["ActingRecordings", "VideoRecordings"]:
        base = root / recording_type
        if not base.exists():
            print(f"Warning: folder not found: {base}")
            continue

        for csv_path in base.glob("*/*/*.csv"):
            participant = csv_path.parents[1].name
            emotion_name = csv_path.parent.name

            if emotion_name not in valid_emotions:
                continue

            rows.append({
                "path": str(csv_path),
                "participant": participant,
                "emotion": Emotion(emotion_name),
                "recording_type": recording_type,
                "domain": 1 if recording_type == "ActingRecordings" else 0,
            })

    records = pd.DataFrame(rows)
    if records.empty:
        raise RuntimeError(
            f"No new participant CSV files found under {root}. "
            "Check NEW_DATA_ROOT and folder structure."
        )

    return records


def make_windows_from_new_recording(csv_path, emotion, thresh):
    """Create teacher-compatible windows from one new participant recording."""
    raw = pd.read_csv(csv_path)

    df = downsample_60hz_to_10hz_face_au(raw, AU_NAMES)   # raw downsampled
    #df_scaled = mean_centered_max_scale(df)               # model input only

    labels = [0] * len(df)

    for i in range(len(df)):
        if emotion == Emotion.NEUTRAL:
            if all(thresh[emotion](value) for value in df.iloc[i].tolist()):
                labels[i] = 1
        else:
            if any(thresh[emotion](df[au].iloc[i]) for au in AU_NAMES):
                labels[i] = 1

    windows, y = [], []
    emo_label = le.transform([emotion.value])[0]

    i = 0
    n = len(df) - SEQ_LENGTH
    consecutive = 3

    while i < n:
        has_consecutive = any(
            sum(labels[i + j:i + j + consecutive]) == consecutive
            for j in range(SEQ_LENGTH - consecutive + 1)
        )

        if has_consecutive:
            windows.append(df.iloc[i:i + SEQ_LENGTH][AU_NAMES])   #
            y.append(emo_label)
            i += SEQ_LENGTH
        else:
            i += 1

    return windows, y


def load_new_collection_split(record_df, thresh):
    """Load a split of new participant recordings into X, emotion labels, and domain labels."""
    X, y, domains = [], [], []

    for _, row in record_df.iterrows():
        windows, labels = make_windows_from_new_recording(
            row["path"],
            row["emotion"],
            thresh,
        )

        X.extend(windows)
        y.extend(labels)
        domains.extend([row["domain"]] * len(labels))

    return X, y, domains


# Part 5: Dataset Construction

## 5.1 Load EmojiHeroVR

Loads the public EmojiHeroVR (DFEA) train/validation/test CSVs. Any AU channel that the teacher
expects but EmojiHeroVR does not provide is added as a constant zero column, so that all three data
sources share the same 70-channel input layout.

### Load the EmojiHeroVR CSVs

**Note:** EmojiHeroVR does not expose every channel the Quest Pro provides (e.g. the tongue AUs).
Those columns are filled with zeros so the input width matches. This is a modelling assumption worth
stating explicitly in the paper: for EmojiHeroVR windows, the missing channels carry no information
rather than being genuinely inactive.

In [ ]:
train_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/training_set.csv")
valid_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/validation_set.csv")
test_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/test_set.csv")

#train_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/training_set.csv")
#valid_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/validation_set.csv")
#test_em_hero_df = pd.read_csv("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/VR Emotions (K-Arts, Semi)/EmojiHeroDB/emoji-hero-vr-db-dfea-as-csv/test_set.csv")

#for au in AU_NAMES[63:]:
  #train_em_hero_df[au]= 0
  #valid_em_hero_df[au]= 0
  #test_em_hero_df[au]= 0

for df in [train_em_hero_df, valid_em_hero_df, test_em_hero_df]:
    for au in AU_NAMES:
        if au not in df.columns:
            df[au] = 0.0

print("Train data shape: ", train_em_hero_df.shape)
print("Validation data shape: ", valid_em_hero_df.shape)
print("Test data shape: ", test_em_hero_df.shape)

In [ ]:
emojihero_splits = {
    "train": train_em_hero_df,
    "validation": valid_em_hero_df,
    "test": test_em_hero_df,
}

for split_name, df in emojihero_splits.items():
    # Preserve the complete sequence identifier.
    df["sequence_id"] = df["sequence_id"].astype(str)
    df["parent_seq"] = df["sequence_id"]

    # Extract the emotion ID from the final part of sequence_id.
    df["emotion"] = (
        df["sequence_id"]
        .str.rsplit("-", n=1)
        .str[-1]
        .astype(int)
    )

    # Verify that the extracted emotion matches the provided label.
    if "Label" in df.columns:
        label_values = pd.to_numeric(
            df["Label"],
            errors="raise",
        ).astype(int)

        if not (df["emotion"] == label_values).all():
            raise ValueError(
                f"{split_name}: sequence_id emotion IDs "
                "do not match the Label column."
            )

    # Sort frames chronologically within each sequence.
    df.sort_values(
        ["parent_seq", "timestamp"],
        kind="stable",
        inplace=True,
    )

    df.reset_index(
        drop=True,
        inplace=True,
    )

    # Verify that every sequence contains exactly 30 frames.
    frame_counts = df.groupby(
        "parent_seq",
        sort=False,
    ).size()

    invalid_frame_counts = frame_counts[
        frame_counts != 30
    ]

    if not invalid_frame_counts.empty:
        raise ValueError(
            f"{split_name}: some sequences do not contain "
            f"exactly 30 frames:\n"
            f"{invalid_frame_counts.head()}"
        )

    # Verify that every sequence contains one emotion only.
    emotion_counts = df.groupby(
        "parent_seq",
        sort=False,
    )["emotion"].nunique(dropna=False)

    mixed_sequences = emotion_counts[
        emotion_counts != 1
    ]

    if not mixed_sequences.empty:
        raise ValueError(
            f"{split_name}: some sequences contain multiple "
            f"emotion labels:\n"
            f"{mixed_sequences.head()}"
        )

    print(
        f"{split_name:10s}: "
        f"{len(df):6d} frames, "
        f"{df['parent_seq'].nunique():4d} sequences"
    )

## 5.3 Match the EmojiHeroVR Frame Rate to 10 Hz

### Average every 3 frames

EmojiHeroVR is recorded at 30 Hz. Averaging every 3 consecutive frames within a `parent_seq` group
brings it to the same 10 Hz sampling rate as our own downsampled recordings, so that a 5-frame window
spans 0.5 s in every data source.

In [ ]:
import pandas as pd

import pandas as pd
import numpy as np

def average_every_3_rows(df: pd.DataFrame, AU_NAMES: list) -> pd.DataFrame:
    """
    Average every 3 consecutive rows within each parent_seq group.
    - Only averages the numeric columns listed in AU_NAMES.
    - Keeps 'parent_seq' and 'emotion' from any of the 3 rows (they're identical).
    - Expects groups of 30 rows per parent_seq (produces 10 rows per group).

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with string columns 'parent_seq' and 'emotion', and numeric AU columns.
    AU_NAMES : list
        List of column names that should be averaged (numeric columns).

    Returns
    -------
    pd.DataFrame
        Aggregated dataframe with columns: ['parent_seq', 'emotion'] + AU_NAMES
    """
    d = df.copy()

    # Ensure we only try to average columns that actually exist
    num_cols = [c for c in AU_NAMES if c in d.columns]

    # Create chunk index for each group of 3 within each parent_seq (0..9 for 30-row groups)
    d["_chunk"] = d.groupby("parent_seq").cumcount() // 3

    # Build aggregation spec: mean for AU columns, first for emotion
    agg_spec = {c: "mean" for c in num_cols}
    agg_spec["emotion"] = "first"

    out = (
        d.groupby(["parent_seq", "_chunk"], as_index=False)
         .agg(agg_spec)
         .drop(columns=["_chunk"])
    )

    # Mean-centered max scaling per recording (parent_seq)
    #out[num_cols] = (
     #   out.groupby("parent_seq", group_keys=False)[num_cols]
      #     .apply(mean_centered_max_scale)
    #)

    # Reorder columns
    ordered_cols = ["parent_seq", "emotion"] + num_cols
    return out[ordered_cols].reset_index(drop=True)

train_em_hero_df = average_every_3_rows(train_em_hero_df, AU_NAMES)
valid_em_hero_df = average_every_3_rows(valid_em_hero_df, AU_NAMES)
test_em_hero_df = average_every_3_rows(test_em_hero_df, AU_NAMES)

print("EmojiHero train columns:", train_em_hero_df.columns.tolist())
print("Number of EmojiHero AU features:", len([c for c in train_em_hero_df.columns if c in AU_NAMES]))

for eye_au in EYE_AU_NAMES:
    assert eye_au in train_em_hero_df.columns, f"{eye_au} missing from EmojiHero train!"
    assert eye_au in valid_em_hero_df.columns, f"{eye_au} missing from EmojiHero valid!"
    assert eye_au in test_em_hero_df.columns, f"{eye_au} missing from EmojiHero test!"

assert len([c for c in train_em_hero_df.columns if c in AU_NAMES]) == 70


In [ ]:
print(train_em_hero_df.shape)
print(valid_em_hero_df.shape)
print(test_em_hero_df.shape)

## 5.4 Build 5-Frame EmojiHeroVR Windows

### Sliding 5-frame windows

Builds 5-frame windows with stride 1 and shuffles them. The window label is taken from the emotion
column of the frames it contains.

**Note:** unlike our own recordings, these windows overlap and are not constrained to stay inside a
single `parent_seq`. Because train/validation/test come from EmojiHeroVR's own predefined splits,
this does not leak across splits, but it does mean EmojiHeroVR contributes more (and more correlated)
windows than a non-overlapping scheme would.

In [ ]:
def create_emojihero_windows(
    df,
    feature_cols,
    window_size=5,
    stride=1,
):
    """
    Create sliding windows within each EmojiHero sequence.

    A window is never allowed to cross a sequence or emotion
    boundary.
    """
    X = []
    y = []

    for sequence_id, sequence_df in df.groupby(
        "parent_seq",
        sort=False,
    ):
        sequence_df = sequence_df.reset_index(drop=True)

        labels = sequence_df["emotion"].unique()

        if len(labels) != 1:
            raise ValueError(
                f"Sequence {sequence_id} contains multiple "
                f"emotion labels: {labels.tolist()}"
            )

        if len(sequence_df) != 10:
            raise ValueError(
                f"Sequence {sequence_id} contains "
                f"{len(sequence_df)} downsampled frames; "
                "expected 10."
            )

        label = int(labels[0])

        for start in range(
            0,
            len(sequence_df) - window_size + 1,
            stride,
        ):
            end = start + window_size

            window = (
                sequence_df
                .iloc[start:end][feature_cols]
                .copy()
            )

            if window.shape != (
                window_size,
                len(feature_cols),
            ):
                raise ValueError(
                    f"Invalid window shape for sequence "
                    f"{sequence_id}: {window.shape}"
                )

            X.append(window)
            y.append(label)

    return X, y


X_train_em_hero, Y_train_em_hero = (
    create_emojihero_windows(
        train_em_hero_df,
        AU_NAMES,
        window_size=SEQ_LENGTH,
        stride=1,
    )
)

X_val_em_hero, Y_val_em_hero = (
    create_emojihero_windows(
        valid_em_hero_df,
        AU_NAMES,
        window_size=SEQ_LENGTH,
        stride=1,
    )
)

X_test_em_hero, Y_test_em_hero = (
    create_emojihero_windows(
        test_em_hero_df,
        AU_NAMES,
        window_size=SEQ_LENGTH,
        stride=1,
    )
)


# Shuffle windows and labels together to preserve alignment.
X_train_em_hero, Y_train_em_hero = sk_shuffle(
    X_train_em_hero,
    Y_train_em_hero,
    random_state=42,
)

X_val_em_hero, Y_val_em_hero = sk_shuffle(
    X_val_em_hero,
    Y_val_em_hero,
    random_state=42,
)

X_test_em_hero, Y_test_em_hero = sk_shuffle(
    X_test_em_hero,
    Y_test_em_hero,
    random_state=42,
)


print("EmojiHero window counts:")
print("Train:", len(X_train_em_hero))
print("Validation:", len(X_val_em_hero))
print("Test:", len(X_test_em_hero))


# Final consistency checks.
assert len(X_train_em_hero) == len(Y_train_em_hero)
assert len(X_val_em_hero) == len(Y_val_em_hero)
assert len(X_test_em_hero) == len(Y_test_em_hero)

assert all(
    window.shape == (SEQ_LENGTH, len(AU_NAMES))
    for window in X_train_em_hero
)

assert all(
    window.shape == (SEQ_LENGTH, len(AU_NAMES))
    for window in X_val_em_hero
)

assert all(
    window.shape == (SEQ_LENGTH, len(AU_NAMES))
    for window in X_test_em_hero
)

## 5.5 Build the Final Training Corpus (with the new participants)



### 5.5.1 Full dataset build

This single cell performs the complete corpus construction:

1. **Thresholds** — an AU is active above 0.25; a Neutral frame requires all AUs at or below 0.25.
2. **Our data** — natural and acted recordings are split participant-exclusively (70/10/20).
3. **Domain labels** — 0 = natural (video-elicited), 1 = acted. EmojiHeroVR is merged into the acted
   domain.
4. **Self-report filter** — two participants are dropped entirely (0/7 and 1/7 self-report matches),
   plus eight specific (participant, emotion) pairs where the intended emotion was never reported.
   These lists come from Part 2.
5. **New participant split** — `StratifiedGroupKFold` with one fold per remaining participant, then
   allocated to train/validation/test by a 70/10/20 participant ratio, with disjointness assertions.
6. **Combination and shuffling** — natural + acted/EmojiHeroVR + new participants, shuffled with a
   fixed seed.
7. **Tensors** — every split is stacked into `(N, 5, 70)` float arrays.
8. **Class weights** — balanced weights, renormalised to average 1.0, to counter the class imbalance
   across the seven emotions.
9. **Assertions** — every tensor must still have shape `(SEQ_LENGTH, 70)`.

#### With Self-Filtering

In [ ]:
thresh = {emo: lambda x: x > 0.25 for emo in Emotion}
thresh[Emotion.NEUTRAL] = lambda x: x <= 0.25

# Previous natural and acted data
X_train_nat, X_val_nat, X_test_nat, Y_train_nat, Y_val_nat, Y_test_nat = train_val_test_data_by_subject(thresh, DataType.OUR_NATURAL)
X_train_act, X_val_act, X_test_act, Y_train_act, Y_val_act, Y_test_act = train_val_test_data_by_subject(thresh, DataType.OUR_ACTED)

Y_train_domain_nat = [0 for _ in range(len(Y_train_nat))]
Y_train_domain_act = [1 for _ in range(len(Y_train_act))]
Y_train_domain_em_hero = [1 for _ in range(len(Y_train_em_hero))]

Y_val_domain_nat = [0 for _ in range(len(Y_val_nat))]
Y_val_domain_act = [1 for _ in range(len(Y_val_act))]
Y_val_domain_em_hero = [1 for _ in range(len(Y_val_em_hero))]

Y_test_domain_nat = [0 for _ in range(len(Y_test_nat))]
Y_test_domain_act = [1 for _ in range(len(Y_test_act))]
Y_test_domain_em_hero = [1 for _ in range(len(Y_test_em_hero))]

# Previous acted-style set includes old acted data + EmojiHero.
X_train_act_comb = X_train_act + X_train_em_hero
Y_train_act_comb = Y_train_act + Y_train_em_hero
Y_train_domain_act_comb = Y_train_domain_act + Y_train_domain_em_hero

X_val_act_comb = X_val_act + X_val_em_hero
Y_val_act_comb = Y_val_act + Y_val_em_hero
Y_val_domain_act_comb = Y_val_domain_act + Y_val_domain_em_hero

X_test_act_comb = X_test_act + X_test_em_hero
Y_test_act_comb = Y_test_act + Y_test_em_hero
Y_test_domain_act_comb = Y_test_domain_act + Y_test_domain_em_hero

records_new = new_collection_records()

# ── Self-report filter (NATURAL / VideoRecordings ONLY) ──────────────────────
# Self-report validity only speaks to whether the video elicitation worked.
# Acted recordings are instructed poses, so they are kept unfiltered.
EXCLUDE_PARTICIPANTS_ENTIRELY = {
    'P41',  # 0/7 match — no reliable self-report
    'P37',  # 1/7 match — effectively no usable data
}

EXCLUDE_PARTICIPANT_EMOTION = {
    ('P35', 'Neutral'),
    ('P36', 'Surprise'),
    ('P39', 'Anger'),
    ('P40', 'Anger'),
    ('P40', 'Sadness'),
    ('P40', 'Surprise'),
    ('P42', 'Happiness'),
    ('P43', 'Surprise'),
    ('P46', 'Anger'),
    ('P46', 'Happiness'),
    ('P47', 'Neutral'),
    ('P47', 'Happiness'),
    ('P48', 'Neutral'),
    ('P48', 'Anger'),
    ('P49', 'Fear'),
    ('P49', 'Anger'),
}

is_natural = records_new["recording_type"] == "VideoRecordings"

drop_participant = is_natural & records_new["participant"].isin(EXCLUDE_PARTICIPANTS_ENTIRELY)

drop_pair = is_natural & records_new.apply(
    lambda r: (r["participant"], r["emotion"].value) in EXCLUDE_PARTICIPANT_EMOTION,
    axis=1,
)

before = len(records_new)
before_nat = int(is_natural.sum())
before_act = before - before_nat

records_new = records_new[~(drop_participant | drop_pair)].reset_index(drop=True)

after = len(records_new)
after_nat = int((records_new["recording_type"] == "VideoRecordings").sum())
after_act = after - after_nat

print("Self-report filter (natural only):")
print(f"  natural : {before_nat} → {after_nat} ({before_nat - after_nat} removed)")
print(f"  acted   : {before_act} → {after_act} ({before_act - after_act} removed, must be 0)")
print(f"  total   : {before} → {after}")
assert before_act == after_act, "Acted recordings must not be filtered."
print(f"Remaining participants: {sorted(records_new['participant'].unique())}")
print("Records per emotion:")
print(records_new['emotion'].apply(lambda e: e.value).value_counts().sort_index())
print("Records per recording_type:")
print(records_new['recording_type'].value_counts())
# ── End self-report filter ───────────────────────────────────────────────────

records_new["Yemo"] = records_new["emotion"].apply(lambda e: le.transform([e.value])[0])
records_new["fold"] = -1
# ... rest of cell 30 continues unchanged

# NEW: Load 10-participant face-AU-only data.
#records_new = new_collection_records()
#records_new["Yemo"] = records_new["emotion"].apply(lambda e: le.transform([e.value])[0])
#records_new["fold"] = -1

n_participants = records_new["participant"].nunique()
print(f"New participants: {n_participants}")

sgkf_new = StratifiedGroupKFold(n_splits=n_participants, shuffle=True, random_state=42)
for fold_id, (_, heldout_idx) in enumerate(
    sgkf_new.split(
        records_new["path"],
        records_new["Yemo"],
        groups=records_new["participant"],
    )
):
    records_new.loc[heldout_idx, "fold"] = fold_id

VAL_RATIO = 0.10
TEST_RATIO = 0.20

n_test = max(1, round(TEST_RATIO * n_participants))
n_val = max(1, round(VAL_RATIO * n_participants))
n_train = n_participants - n_val - n_test

assert n_train > 0
assert n_val > 0
assert n_test > 0

train_folds = list(range(0, n_train))
val_folds   = list(range(n_train, n_train + n_val))
test_folds  = list(range(n_train + n_val, n_participants))

#new_train_records = records_new[records_new["fold"].between(0, 6)].reset_index(drop=True)
#new_val_records = records_new[records_new["fold"] == 7].reset_index(drop=True)
#new_test_records = records_new[records_new["fold"].between(8, 9)].reset_index(drop=True)

new_train_records = records_new[records_new["fold"].isin(train_folds)].reset_index(drop=True)
new_val_records   = records_new[records_new["fold"].isin(val_folds)].reset_index(drop=True)
new_test_records  = records_new[records_new["fold"].isin(test_folds)].reset_index(drop=True)


assert set(new_train_records["participant"]).isdisjoint(new_val_records["participant"])
assert set(new_train_records["participant"]).isdisjoint(new_test_records["participant"])
assert set(new_val_records["participant"]).isdisjoint(new_test_records["participant"])

X_train_new, Y_train_new, Y_train_domain_new = load_new_collection_split(new_train_records, thresh)
X_val_new, Y_val_new, Y_val_domain_new = load_new_collection_split(new_val_records, thresh)
X_test_new, Y_test_new, Y_test_domain_new = load_new_collection_split(new_test_records, thresh)

print("Previous natural train windows:", len(X_train_nat))
print("Previous acted+EmojiHero train windows:", len(X_train_act_comb))
print("NEW 10-participant train windows:", len(X_train_new))

# Final combined set: previous data + new 10-participant AU-only data.
X_train_comb = X_train_nat + X_train_act_comb + X_train_new
Y_train_comb = Y_train_nat + Y_train_act_comb + Y_train_new
Y_train_domain_comb = Y_train_domain_nat + Y_train_domain_act_comb + Y_train_domain_new

X_val_comb = X_val_nat + X_val_act_comb + X_val_new
Y_val_comb = Y_val_nat + Y_val_act_comb + Y_val_new
Y_val_domain_comb = Y_val_domain_nat + Y_val_domain_act_comb + Y_val_domain_new

X_test_comb = X_test_nat + X_test_act_comb + X_test_new
Y_test_comb = Y_test_nat + Y_test_act_comb + Y_test_new
Y_test_domain_comb = Y_test_domain_nat + Y_test_domain_act_comb + Y_test_domain_new

# Shuffle combined train/validation/test sets.
X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb = sk_shuffle(X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb, random_state=42)
X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb = sk_shuffle(X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb, random_state=42)
X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb = sk_shuffle(X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb, random_state=42)

X_train_comb, Y_train_comb, Y_train_domain_comb = sk_shuffle(X_train_comb, Y_train_comb, Y_train_domain_comb, random_state=42)
X_val_comb, Y_val_comb, Y_val_domain_comb = sk_shuffle(X_val_comb, Y_val_comb, Y_val_domain_comb, random_state=42)
X_test_comb, Y_test_comb, Y_test_domain_comb = sk_shuffle(X_test_comb, Y_test_comb, Y_test_domain_comb, random_state=42)

# Convert lists of DataFrames to numpy tensors.
X_train_nat_np = stack_sequences(X_train_nat)
Y_train_nat_np = to_numpy_labels(Y_train_nat)
Y_train_nat_domain_np = to_numpy_labels(Y_train_domain_nat)

X_train_act_comb_np = stack_sequences(X_train_act_comb)
Y_train_act_comb_np = to_numpy_labels(Y_train_act_comb)
Y_train_act_domain_comb_np = to_numpy_labels(Y_train_domain_act_comb)

X_val_act_comb_np = stack_sequences(X_val_act_comb)
Y_val_act_comb_np = to_numpy_labels(Y_val_act_comb)
Y_val_domain_act_comb_np = to_numpy_labels(Y_val_domain_act_comb)

X_test_act_comb_np = stack_sequences(X_test_act_comb)
Y_test_act_comb_np = to_numpy_labels(Y_test_act_comb)
Y_test_domain_act_comb_np = to_numpy_labels(Y_test_domain_act_comb)

X_train_np = stack_sequences(X_train_comb)
Y_train_np = to_numpy_labels(Y_train_comb)
Y_train_domain_np = to_numpy_labels(Y_train_domain_comb)

X_val_np = stack_sequences(X_val_comb)
Y_val_np = to_numpy_labels(Y_val_comb)
Y_val_domain_np = to_numpy_labels(Y_val_domain_comb)

X_test_nat_np = stack_sequences(X_test_nat)
Y_test_nat_np = to_numpy_labels(Y_test_nat)
Y_test_nat_domain_np = to_numpy_labels(Y_test_domain_nat)

X_test_comb_np = stack_sequences(X_test_comb)
Y_test_comb_np = to_numpy_labels(Y_test_comb)
Y_test_domain_comb_np = to_numpy_labels(Y_test_domain_comb)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(Y_train_comb),
    y=Y_train_comb,
)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights * len(class_weights)

assert X_train_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_val_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_test_comb_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))


In [ ]:
print("Split assignment:")
for name, recs in [("train", new_train_records),
                   ("val",   new_val_records),
                   ("test",  new_test_records)]:
    parts = sorted(recs["participant"].unique())
    emos  = sorted(recs["emotion"].apply(lambda e: e.value).unique())
    missing = sorted(set(e.value for e in Emotion) - set(emos))
    print(f"  {name:5s} ({len(parts)}p): {parts}")
    print(f"        emotions present: {emos}")
    if missing:
        print(f"        MISSING: {missing}")

In [ ]:
print("EmojiHero windows:")
print("Train:", len(X_train_em_hero))
print("Validation:", len(X_val_em_hero))
print("Test:", len(X_test_em_hero))

assert len(X_train_em_hero) == 5784
assert len(X_val_em_hero) == 2310
assert len(X_test_em_hero) == 2268

print("\nFinal tensors:")
print("Train:", X_train_np.shape)
print("Validation:", X_val_np.shape)
print("Test:", X_test_comb_np.shape)

assert X_train_np.shape[1:] == (5, 70)
assert X_val_np.shape[1:] == (5, 70)
assert X_test_comb_np.shape[1:] == (5, 70)

assert len(X_train_np) == len(Y_train_np)
assert len(X_val_np) == len(Y_val_np)
assert len(X_test_comb_np) == len(Y_test_comb_np)

print("\nAll Part 5 checks passed.")

#### Without Self-Filtering

In [ ]:
thresh = {emo: lambda x: x > 0.25 for emo in Emotion}
thresh[Emotion.NEUTRAL] = lambda x: x <= 0.25

# Previous natural and acted data
X_train_nat, X_val_nat, X_test_nat, Y_train_nat, Y_val_nat, Y_test_nat = train_val_test_data_by_subject(thresh, DataType.OUR_NATURAL)
X_train_act, X_val_act, X_test_act, Y_train_act, Y_val_act, Y_test_act = train_val_test_data_by_subject(thresh, DataType.OUR_ACTED)

Y_train_domain_nat = [0 for _ in range(len(Y_train_nat))]
Y_train_domain_act = [1 for _ in range(len(Y_train_act))]
Y_train_domain_em_hero = [1 for _ in range(len(Y_train_em_hero))]

Y_val_domain_nat = [0 for _ in range(len(Y_val_nat))]
Y_val_domain_act = [1 for _ in range(len(Y_val_act))]
Y_val_domain_em_hero = [1 for _ in range(len(Y_val_em_hero))]

Y_test_domain_nat = [0 for _ in range(len(Y_test_nat))]
Y_test_domain_act = [1 for _ in range(len(Y_test_act))]
Y_test_domain_em_hero = [1 for _ in range(len(Y_test_em_hero))]

# Previous acted-style set includes old acted data + EmojiHero.
X_train_act_comb = X_train_act + X_train_em_hero
Y_train_act_comb = Y_train_act + Y_train_em_hero
Y_train_domain_act_comb = Y_train_domain_act + Y_train_domain_em_hero

X_val_act_comb = X_val_act + X_val_em_hero
Y_val_act_comb = Y_val_act + Y_val_em_hero
Y_val_domain_act_comb = Y_val_domain_act + Y_val_domain_em_hero

X_test_act_comb = X_test_act + X_test_em_hero
Y_test_act_comb = Y_test_act + Y_test_em_hero
Y_test_domain_act_comb = Y_test_domain_act + Y_test_domain_em_hero

records_new = new_collection_records()

# ── Self-report filter DISABLED — using all participants / all emotions ───────
print(f"Self-report filter OFF: {len(records_new)} records kept "
      f"(no participants or emotion pairs excluded)")
print(f"Participants: {sorted(records_new['participant'].unique())}")
print(f"Records per emotion:\n{records_new['emotion'].apply(lambda e: e.value).value_counts().sort_index()}")
# ── End (no filter) ───────────────────────────────────────────────────────────

records_new["Yemo"] = records_new["emotion"].apply(lambda e: le.transform([e.value])[0])
records_new["fold"] = -1

n_participants = records_new["participant"].nunique()
print(f"New participants: {n_participants}")

sgkf_new = StratifiedGroupKFold(n_splits=n_participants, shuffle=True, random_state=42)
for fold_id, (_, heldout_idx) in enumerate(
    sgkf_new.split(
        records_new["path"],
        records_new["Yemo"],
        groups=records_new["participant"],
    )
):
    records_new.loc[heldout_idx, "fold"] = fold_id

VAL_RATIO = 0.10
TEST_RATIO = 0.20

n_test = max(1, round(TEST_RATIO * n_participants))
n_val = max(1, round(VAL_RATIO * n_participants))
n_train = n_participants - n_val - n_test

assert n_train > 0
assert n_val > 0
assert n_test > 0

train_folds = list(range(0, n_train))
val_folds   = list(range(n_train, n_train + n_val))
test_folds  = list(range(n_train + n_val, n_participants))

new_train_records = records_new[records_new["fold"].isin(train_folds)].reset_index(drop=True)
new_val_records   = records_new[records_new["fold"].isin(val_folds)].reset_index(drop=True)
new_test_records  = records_new[records_new["fold"].isin(test_folds)].reset_index(drop=True)

assert set(new_train_records["participant"]).isdisjoint(new_val_records["participant"])
assert set(new_train_records["participant"]).isdisjoint(new_test_records["participant"])
assert set(new_val_records["participant"]).isdisjoint(new_test_records["participant"])

X_train_new, Y_train_new, Y_train_domain_new = load_new_collection_split(new_train_records, thresh)
X_val_new, Y_val_new, Y_val_domain_new = load_new_collection_split(new_val_records, thresh)
X_test_new, Y_test_new, Y_test_domain_new = load_new_collection_split(new_test_records, thresh)

print("Previous natural train windows:", len(X_train_nat))
print("Previous acted+EmojiHero train windows:", len(X_train_act_comb))
print("NEW participant train windows:", len(X_train_new))

# Final combined set: previous data + new AU-only data.
X_train_comb = X_train_nat + X_train_act_comb + X_train_new
Y_train_comb = Y_train_nat + Y_train_act_comb + Y_train_new
Y_train_domain_comb = Y_train_domain_nat + Y_train_domain_act_comb + Y_train_domain_new

X_val_comb = X_val_nat + X_val_act_comb + X_val_new
Y_val_comb = Y_val_nat + Y_val_act_comb + Y_val_new
Y_val_domain_comb = Y_val_domain_nat + Y_val_domain_act_comb + Y_val_domain_new

X_test_comb = X_test_nat + X_test_act_comb + X_test_new
Y_test_comb = Y_test_nat + Y_test_act_comb + Y_test_new
Y_test_domain_comb = Y_test_domain_nat + Y_test_domain_act_comb + Y_test_domain_new

# Shuffle combined train/validation/test sets.
X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb = sk_shuffle(X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb, random_state=42)
X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb = sk_shuffle(X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb, random_state=42)
X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb = sk_shuffle(X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb, random_state=42)

X_train_comb, Y_train_comb, Y_train_domain_comb = sk_shuffle(X_train_comb, Y_train_comb, Y_train_domain_comb, random_state=42)
X_val_comb, Y_val_comb, Y_val_domain_comb = sk_shuffle(X_val_comb, Y_val_comb, Y_val_domain_comb, random_state=42)
X_test_comb, Y_test_comb, Y_test_domain_comb = sk_shuffle(X_test_comb, Y_test_comb, Y_test_domain_comb, random_state=42)

# Convert lists of DataFrames to numpy tensors.
X_train_nat_np = stack_sequences(X_train_nat)
Y_train_nat_np = to_numpy_labels(Y_train_nat)
Y_train_nat_domain_np = to_numpy_labels(Y_train_domain_nat)

X_train_act_comb_np = stack_sequences(X_train_act_comb)
Y_train_act_comb_np = to_numpy_labels(Y_train_act_comb)
Y_train_act_domain_comb_np = to_numpy_labels(Y_train_domain_act_comb)

X_val_act_comb_np = stack_sequences(X_val_act_comb)
Y_val_act_comb_np = to_numpy_labels(Y_val_act_comb)
Y_val_domain_act_comb_np = to_numpy_labels(Y_val_domain_act_comb)

X_test_act_comb_np = stack_sequences(X_test_act_comb)
Y_test_act_comb_np = to_numpy_labels(Y_test_act_comb)
Y_test_domain_act_comb_np = to_numpy_labels(Y_test_domain_act_comb)

X_train_np = stack_sequences(X_train_comb)
Y_train_np = to_numpy_labels(Y_train_comb)
Y_train_domain_np = to_numpy_labels(Y_train_domain_comb)

X_val_np = stack_sequences(X_val_comb)
Y_val_np = to_numpy_labels(Y_val_comb)
Y_val_domain_np = to_numpy_labels(Y_val_domain_comb)

X_test_nat_np = stack_sequences(X_test_nat)
Y_test_nat_np = to_numpy_labels(Y_test_nat)
Y_test_nat_domain_np = to_numpy_labels(Y_test_domain_nat)

X_test_comb_np = stack_sequences(X_test_comb)
Y_test_comb_np = to_numpy_labels(Y_test_comb)
Y_test_domain_comb_np = to_numpy_labels(Y_test_domain_comb)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(Y_train_comb),
    y=Y_train_comb,
)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights * len(class_weights)

assert X_train_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_val_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_test_comb_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))

In [ ]:
print("Split assignment:")
for name, recs in [("train", new_train_records),
                   ("val",   new_val_records),
                   ("test",  new_test_records)]:
    parts = sorted(recs["participant"].unique())
    emos  = sorted(recs["emotion"].apply(lambda e: e.value).unique())
    missing = sorted(set(e.value for e in Emotion) - set(emos))
    print(f"  {name:5s} ({len(parts)}p): {parts}")
    print(f"        emotions present: {emos}")
    if missing:
        print(f"        MISSING: {missing}")

## 5.6 Alternative Build: Original Teacher (without the new participants)

Baseline ablation. Same pipeline, same thresholds, same split logic — the new participant data is
simply omitted.

**Note:** this cell overwrites the arrays produced by Section 5.5. Run either 5.5 or 5.6, never both,
before training.

### 5.6.1 Original teacher dataset build

Identical to Section 5.5 except that `X_train_new` / `X_val_new` / `X_test_new` are never appended.
The self-report filter is therefore irrelevant here — there is no new-participant data to filter.

In [ ]:
thresh = {emo: lambda x: x > 0.25 for emo in Emotion}
thresh[Emotion.NEUTRAL] = lambda x: x <= 0.25

# Previous natural and acted data
X_train_nat, X_val_nat, X_test_nat, Y_train_nat, Y_val_nat, Y_test_nat = train_val_test_data_by_subject(thresh, DataType.OUR_NATURAL)
X_train_act, X_val_act, X_test_act, Y_train_act, Y_val_act, Y_test_act = train_val_test_data_by_subject(thresh, DataType.OUR_ACTED)

Y_train_domain_nat = [0 for _ in range(len(Y_train_nat))]
Y_train_domain_act = [1 for _ in range(len(Y_train_act))]
Y_train_domain_em_hero = [1 for _ in range(len(Y_train_em_hero))]

Y_val_domain_nat = [0 for _ in range(len(Y_val_nat))]
Y_val_domain_act = [1 for _ in range(len(Y_val_act))]
Y_val_domain_em_hero = [1 for _ in range(len(Y_val_em_hero))]

Y_test_domain_nat = [0 for _ in range(len(Y_test_nat))]
Y_test_domain_act = [1 for _ in range(len(Y_test_act))]
Y_test_domain_em_hero = [1 for _ in range(len(Y_test_em_hero))]

# Acted-style set includes acted data + EmojiHero.
X_train_act_comb = X_train_act + X_train_em_hero
Y_train_act_comb = Y_train_act + Y_train_em_hero
Y_train_domain_act_comb = Y_train_domain_act + Y_train_domain_em_hero

X_val_act_comb = X_val_act + X_val_em_hero
Y_val_act_comb = Y_val_act + Y_val_em_hero
Y_val_domain_act_comb = Y_val_domain_act + Y_val_domain_em_hero

X_test_act_comb = X_test_act + X_test_em_hero
Y_test_act_comb = Y_test_act + Y_test_em_hero
Y_test_domain_act_comb = Y_test_domain_act + Y_test_domain_em_hero

print("Natural train windows:", len(X_train_nat))
print("Acted+EmojiHero train windows:", len(X_train_act_comb))

# Final combined set: natural + acted/EmojiHero only (no new participant data).
X_train_comb = X_train_nat + X_train_act_comb
Y_train_comb = Y_train_nat + Y_train_act_comb
Y_train_domain_comb = Y_train_domain_nat + Y_train_domain_act_comb

X_val_comb = X_val_nat + X_val_act_comb
Y_val_comb = Y_val_nat + Y_val_act_comb
Y_val_domain_comb = Y_val_domain_nat + Y_val_domain_act_comb

X_test_comb = X_test_nat + X_test_act_comb
Y_test_comb = Y_test_nat + Y_test_act_comb
Y_test_domain_comb = Y_test_domain_nat + Y_test_domain_act_comb

# Shuffle combined train/validation/test sets.
X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb = sk_shuffle(X_train_act_comb, Y_train_act_comb, Y_train_domain_act_comb, random_state=42)
X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb = sk_shuffle(X_val_act_comb, Y_val_act_comb, Y_val_domain_act_comb, random_state=42)
X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb = sk_shuffle(X_test_act_comb, Y_test_act_comb, Y_test_domain_act_comb, random_state=42)

X_train_comb, Y_train_comb, Y_train_domain_comb = sk_shuffle(X_train_comb, Y_train_comb, Y_train_domain_comb, random_state=42)
X_val_comb, Y_val_comb, Y_val_domain_comb = sk_shuffle(X_val_comb, Y_val_comb, Y_val_domain_comb, random_state=42)
X_test_comb, Y_test_comb, Y_test_domain_comb = sk_shuffle(X_test_comb, Y_test_comb, Y_test_domain_comb, random_state=42)

# Convert lists of DataFrames to numpy tensors.
X_train_nat_np = stack_sequences(X_train_nat)
Y_train_nat_np = to_numpy_labels(Y_train_nat)
Y_train_nat_domain_np = to_numpy_labels(Y_train_domain_nat)

X_train_act_comb_np = stack_sequences(X_train_act_comb)
Y_train_act_comb_np = to_numpy_labels(Y_train_act_comb)
Y_train_act_domain_comb_np = to_numpy_labels(Y_train_domain_act_comb)

X_val_act_comb_np = stack_sequences(X_val_act_comb)
Y_val_act_comb_np = to_numpy_labels(Y_val_act_comb)
Y_val_domain_act_comb_np = to_numpy_labels(Y_val_domain_act_comb)

X_test_act_comb_np = stack_sequences(X_test_act_comb)
Y_test_act_comb_np = to_numpy_labels(Y_test_act_comb)
Y_test_domain_act_comb_np = to_numpy_labels(Y_test_domain_act_comb)

X_train_np = stack_sequences(X_train_comb)
Y_train_np = to_numpy_labels(Y_train_comb)
Y_train_domain_np = to_numpy_labels(Y_train_domain_comb)

X_val_np = stack_sequences(X_val_comb)
Y_val_np = to_numpy_labels(Y_val_comb)
Y_val_domain_np = to_numpy_labels(Y_val_domain_comb)

X_test_nat_np = stack_sequences(X_test_nat)
Y_test_nat_np = to_numpy_labels(Y_test_nat)
Y_test_nat_domain_np = to_numpy_labels(Y_test_domain_nat)

X_test_comb_np = stack_sequences(X_test_comb)
Y_test_comb_np = to_numpy_labels(Y_test_comb)
Y_test_domain_comb_np = to_numpy_labels(Y_test_domain_comb)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(Y_train_comb),
    y=Y_train_comb,
)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights * len(class_weights)

assert X_train_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_val_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_test_comb_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))

# Save the Splits

In [ ]:
from pathlib import Path

SAVE_DIR = Path(output_path) / 'teacher_15_participants-self_filtered_ONLY_NATURAL-NOT-normalized_every_teacher_dataset'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    SAVE_DIR / 'natural_splits.npz',
    X_train=X_train_nat, X_val=X_val_nat, X_test=X_test_nat,
    Y_train=Y_train_nat, Y_val=Y_val_nat, Y_test=Y_test_nat,
)
np.savez_compressed(
    SAVE_DIR / 'acted_splits.npz',
    X_train=X_train_act, X_val=X_val_act, X_test=X_test_act,
    Y_train=Y_train_act, Y_val=Y_val_act, Y_test=Y_test_act,
)

print('Saved to:', SAVE_DIR)
for f in sorted(SAVE_DIR.iterdir()):
    print(f'  {f.name:24s} {f.stat().st_size/1e6:.2f} MB')

## Test 1
Test 1 — recording level (original + new participant). Verifies that the function behaves correctly on an actual recording: after centering, each column has a mean ≈ 0 and a max ≈ 1, with no values ​​> 1.

In [ ]:
def check_recording_scaling(df_raw, name):
    sc = mean_centered_max_scale(df_raw)

    # Determine which columns contain actual variation.
    centered = df_raw - df_raw.mean()
    active = centered.abs().max() > 1e-6

    # After mean centering, every active feature should have mean approximately zero.
    mean_abs = sc.mean().abs().max()

    # After absolute-max scaling, every active feature should have
    # maximum absolute magnitude approximately one.
    abs_max_active = sc.abs().max()[active]

    # No value should be outside approximately [-1, 1].
    outside_range = (np.abs(sc.values) > 1 + 1e-6).any()

    global_min = sc.min().min()
    global_max = sc.max().max()

    print(
        f"[{name}] "
        f"|mean| max={mean_abs:.2e}  "
        f"active abs-max=({abs_max_active.min():.4f}, "
        f"{abs_max_active.max():.4f})  "
        f"range=({global_min:.4f}, {global_max:.4f})  "
        f"active={int(active.sum())}/{len(active)}  "
        f"outside [-1,1]? {bool(outside_range)}"
    )

    assert mean_abs < 1e-5, (
        f"{name}: centering broken"
    )

    assert abs_max_active.min() >= 0.99, (
        f"{name}: absolute-max scaling broken; "
        f"minimum abs-max={abs_max_active.min():.4f}"
    )

    assert not outside_range, (
        f"{name}: value outside [-1,1]"
    )


f_nat, _ = all_files_list(DataType.OUR_NATURAL)[0]
df_nat, _ = data_frame(f_nat)
check_recording_scaling(df_nat, "OUR_NATURAL")


f_act, _ = all_files_list(DataType.OUR_ACTED)[0]
df_act, _ = data_frame(f_act)
check_recording_scaling(df_act, "OUR_ACTED")


rec = new_collection_records().iloc[0]
df_new = downsample_60hz_to_10hz_face_au(
    pd.read_csv(rec["path"]),
    AU_NAMES
)
check_recording_scaling(df_new, "NEW_PARTICIPANT")

print("Test 1 passed.")

## Test 2
Test 2 — EmojiHero (per parent_seq). After Cell 42 runs, `train_em_hero_df` should already be scaled. Check whether each `parent_seq` is normalized independently.

In [ ]:
def check_emojihero_scaling(df, name):
    num_cols = [
        c for c in AU_NAMES
        if c in df.columns
    ]

    grouped = df.groupby("parent_seq")[num_cols]

    mean_abs = grouped.mean().abs().to_numpy().max()
    global_abs_max = df[num_cols].abs().to_numpy().max()
    global_min = df[num_cols].to_numpy().min()
    global_max = df[num_cols].to_numpy().max()

    print(
        f"[{name}] "
        f"max |per-seq mean|={mean_abs:.2e}  "
        f"range=({global_min:.4f}, {global_max:.4f})  "
        f"global |max|={global_abs_max:.4f}"
    )

    assert mean_abs < 1e-4, (
        f"{name}: per-sequence centering broken"
    )

    assert global_abs_max <= 1 + 1e-6, (
        f"{name}: value outside [-1,1]"
    )


check_emojihero_scaling(
    train_em_hero_df,
    "EmojiHero train"
)

check_emojihero_scaling(
    valid_em_hero_df,
    "EmojiHero validation"
)

check_emojihero_scaling(
    test_em_hero_df,
    "EmojiHero test"
)

print("Test 2 passed.")

## Test 3
Test 3 — final tensors + corpus regression. After windowing: no values ​​> 1, negatives present. And B's key point — the number of windows must remain the same as it was before normalization.

In [ ]:
def tensor_check(name, X):
    X = np.asarray(X, dtype=np.float64)
    print(f"[{name}] shape={X.shape} min={X.min():.4f} max={X.max():.4f} "
          f"mean={X.mean():.4f} has_neg={(X<0).any()}")
    assert X.max() <= 1 + 1e-4, f"{name}: value greater than 1"
    assert (X < 0).any(),       f"{name}: no negatives -> centering did not run"

tensor_check("X_train_np", X_train_np)
tensor_check("X_train_em_hero", np.stack([d.values for d in X_train_em_hero]))

print("\nWindow counts:", len(X_train_np), len(X_val_np), len(X_test_comb_np))
# Compare these against the run from BEFORE normalization was added.
# If B was applied correctly they must be IDENTICAL (labeling runs on raw values).

## 5.7 Integration Review Checks

Run this before training. It confirms that the new 10-participant data is participant-exclusive
across the train/validation/test splits, prints the final tensor shapes, and asserts that the teacher
input shape is still `(SEQ_LENGTH, 70)`.

**Note:** the final `print` message still reads "(SEQ_LENGTH, 60)" — a leftover from the earlier
60-AU teacher. The assertion above it correctly checks 70.

### 5.7.1 Split exclusivity and tensor shapes

Run this cell before training. It confirms the new 10-participant data is participant-exclusive across splits and that all final tensors still use the teacher input shape (5, 60).

In [ ]:
print("NEW participant split counts")
print("Train participants:", new_train_records["participant"].nunique(), sorted(new_train_records["participant"].unique()))
print("Val participants:", new_val_records["participant"].nunique(), sorted(new_val_records["participant"].unique()))
print("Test participants:", new_test_records["participant"].nunique(), sorted(new_test_records["participant"].unique()))
print()

print("NEW recording distribution")
display(pd.crosstab([records_new["participant"], records_new["recording_type"]], records_new["emotion"].apply(lambda e: e.value)))
print()

print("Final tensor shapes")
print("X_train_np:", X_train_np.shape)
print("Y_train_np:", Y_train_np.shape)
print("Y_train_domain_np:", Y_train_domain_np.shape)
print("X_val_np:", X_val_np.shape)
print("Y_val_np:", Y_val_np.shape)
print("X_test_comb_np:", X_test_comb_np.shape)
print("Y_test_comb_np:", Y_test_comb_np.shape)
print()

print("Window counts")
print("Old natural train:", len(X_train_nat))
print("Old acted + EmojiHero train:", len(X_train_act_comb))
print("New 10-participant train:", len(X_train_new))
print("Final combined train:", len(X_train_comb))

assert X_train_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_val_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert X_test_comb_np.shape[1:] == (SEQ_LENGTH, len(AU_NAMES))
assert len(AU_NAMES) == 70


### 5.7.2 Final tensor shapes

In [ ]:

print(f"X_train shape: {X_train_np.shape}")
print(f"Y_train shape: {Y_train_np.shape}")
print(f"Y_train_domain shape: {Y_train_domain_np.shape}")

print()
print(f"X_val shape: {X_val_np.shape}")
print(f"Y_val shape: {Y_val_np.shape}")
print(f"Y_val_domain shape: {Y_val_domain_np.shape}")
print()
print(f"X_test_nat shape: {X_test_nat_np.shape}")
print(f"Y_test_nat shape: {Y_test_nat_np.shape}")
print()

### 5.7.3 Window-count arithmetic

Confirms that the combined training tensor contains exactly the natural plus acted/EmojiHeroVR
windows.

**Note:** this check corresponds to the Section 5.6 build. After the Section 5.5 build it will not
balance, because the new-participant windows are also included.

In [ ]:
print(len(X_train_nat), "+", len(X_train_act_comb), "=", len(X_train_nat) + len(X_train_act_comb))
print("X_train_np actual:", X_train_np.shape[0])

# Part 6: Training and Evaluation Helpers

## 6.1 Scoring and Training Wrappers

- `print_scores` — accuracy, confusion matrix, per-class classification report, and a heatmap for a
  given test set
- `train_then_test` — fits the model on the module-level `X_train_np` / `Y_train_np`, validates on
  `X_val_np` / `Y_val_np`, applies the balanced class weights, and reports macro-F1 on the combined
  test set

**Note:** `train_then_test` reads the training tensors from the global namespace. Whichever dataset
build ran last (Section 5.5 or 5.6) is the one that gets trained on.

In [ ]:
def print_scores(model, X_test, Y_test):
    Y_pred = model.predict(X_test)
    Y_pred = np.argmax(Y_pred, axis=1)
    print('shape', np.shape(Y_test), np.shape(Y_pred))

    Y_test = np.reshape(Y_test, -1)
    Y_pred = np.reshape(Y_pred, -1)

    acc = accuracy_score(Y_test, Y_pred)
    cm = confusion_matrix(Y_test, Y_pred)

    print(f"Accuracy: {acc}")
    print(f"Confusion matrix:\n{cm}")

    report = classification_report(Y_test, Y_pred, labels=range(len(Emotion)), target_names=[emo.value for emo in Emotion])
    print("\nClassification Report:\n", report)

    sns.heatmap(cm, fmt='d', cmap='Blues', xticklabels=[emo.value for emo in Emotion], yticklabels=[emo.value for emo in Emotion])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


def train_then_test(model, window_size, *args, **kws):
    # opt = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.95 )
    # model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

    history = model.fit(
        X_train_np, Y_train_np,
        validation_data=(X_val_np, Y_val_np),
        class_weight=dict(enumerate(class_weights)),
        **kws,
    )
    print(history.params)

    Y_pred = model.predict(X_test_comb_np)
    Y_pred = np.argmax(Y_pred, axis=1)
    score = f1_score(Y_test_comb_np, Y_pred, average='macro')


    print(f"F1 score: {score}")

    return model


# # @tf.keras.saving.register_keras_serializable()
# def loss(Y_true, Y_pred):
#     weights_per_sample = tf.gather(
#         tf.constant(class_weights, dtype=tf.float32),
#         tf.cast(Y_true, tf.int32),
#     )
#     score = tf.keras.losses.sparse_categorical_crossentropy(Y_true, Y_pred)
#     # Apply class weights to the losses
#     return tf.reduce_mean(score * weights_per_sample)

# Part 7: Teacher Model — MoE-TCN

## 7.1 Hyperparameters

Input width 70 (AU channels), 7 output classes, batch size 10, 30 epochs. Early stopping with
`restore_best_weights=True` usually halts training before epoch 30.

In [ ]:
input_size = len(AU_NAMES)
num_classes = len(Emotion)
batch_size = 10
epochs = 30

### Input-width guard

Asserts that all three tensors have the expected 70-channel width before a model is built.

In [ ]:
print("Expected input size:", input_size)
print("X_train_np shape:", X_train_np.shape)
print("X_val_np shape:", X_val_np.shape)
print("X_test_comb_np shape:", X_test_comb_np.shape)

assert X_train_np.shape[-1] == input_size
assert X_val_np.shape[-1] == input_size
assert X_test_comb_np.shape[-1] == input_size

## 7.2 Architecture

### MoE-TCN: 3 TCN experts + Top-2 gate

### 7.2.1 Architecture definition

**`_tcn_block`** — dilated causal 1-D convolution -> batch norm -> ReLU -> dropout, twice, with a
residual connection (1x1 projection when the channel count changes).

**`build_expert_tcn`** — three stacked TCN blocks with dilation rates 1, 2, 4 and 128 filters each,
followed by global average pooling. With kernel size 3, this gives a receptive field that covers the
full 5-frame window.

**`Top2Gate`** — a small MLP produces one logit per expert. Logits are divided by a temperature
`tau` and softmaxed; the top 2 experts are kept, their weights renormalised, and scattered back into
a sparse `(B, 3)` weight vector. Two auxiliary losses are added:

- a **load-balancing loss** (`lb_coef`) penalising deviation from uniform expert usage, so no expert
  is starved
- an **entropy regulariser** (`ent_coef`) encouraging exploration of all experts early in training

**`GateTemperatureScheduler`** — anneals `tau` from 2.0 to 0.7 over 20 epochs: soft, exploratory
routing at the start, sharper and more specialised routing later.

**`build_moe_single_input`** — assembles three experts (`exp_nat`, `exp_act`, `exp_shared`), gates
them on the pooled input sequence, takes the gate-weighted sum, and passes it through a 256-unit
dense classifier to a 7-way softmax. Optimised with SGD (lr = 1e-3) and sparse categorical
cross-entropy.

**Note:** the expert-stacking and fusion steps use `Lambda` layers. Keras cannot deserialise these
from a saved `.keras` file, which is why every reload in this notebook rebuilds the architecture and
calls `load_weights()` (Section 8.3) rather than `load_model()`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K

# ---- TCN expert blocks ----
def _tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2, name=None):
    res = x
    y = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate, name=None if name is None else name+"_c1")(x)
    y = layers.BatchNormalization(name=None if name is None else name+"_bn1")(y)
    y = layers.ReLU(name=None if name is None else name+"_relu1")(y)
    y = layers.Dropout(dropout, name=None if name is None else name+"_do1")(y)
    y = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate, name=None if name is None else name+"_c2")(y)
    y = layers.BatchNormalization(name=None if name is None else name+"_bn2")(y)
    if res.shape[-1] != filters:
        res = layers.Conv1D(filters, 1, padding="same", name=None if name is None else name+"_proj")(res)
        res = layers.BatchNormalization(name=None if name is None else name+"_proj_bn")(res)
    y = layers.Add(name=None if name is None else name+"_add")([y, res])
    y = layers.ReLU(name=None if name is None else name+"_relu2")(y)
    return y

def build_expert_tcn(x, name_prefix="exp"):
    y = _tcn_block(x, 128, 3, 1, 0.2, name=name_prefix+"_tcn1")
    y = _tcn_block(y, 128, 3, 2, 0.2, name=name_prefix+"_tcn2")
    y = _tcn_block(y, 128, 3, 4, 0.2, name=name_prefix+"_tcn3")
    y = layers.GlobalAveragePooling1D(name=name_prefix+"_gap")(y)
    return y  # (B, F)

# ---- Top-2 gate as a Keras Layer ----
class Top2Gate(layers.Layer):
    def __init__(self, num_experts=3, gate_hidden=128, tau_init=2.0,
                 lb_coef=0.02, ent_coef=0.01, name="gate", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_experts = num_experts
        self.gate_hidden = gate_hidden
        self.tau = tf.Variable(tau_init, dtype=tf.float32, trainable=False, name=f"{name}_tau")
        self.lb_coef = lb_coef
        self.ent_coef = ent_coef
        self.d1 = layers.Dense(gate_hidden, activation="relu", name=f"{name}_dense1")
        self.d2 = layers.Dense(num_experts, activation=None, name=f"{name}_logits")

    def call(self, gate_in, training=None):
        logits = self.d2(self.d1(gate_in))           # (B, E)
        w_soft = tf.nn.softmax(logits / self.tau, -1)  # (B, E)

        top2 = tf.math.top_k(w_soft, k=2)
        idx = top2.indices            # (B, 2)
        vals = tf.nn.softmax(top2.values, -1)  # re-norm top-2

        B = tf.shape(w_soft)[0]; E = tf.shape(w_soft)[1]
        row_ids = tf.repeat(tf.range(B), repeats=2)
        col_ids = tf.reshape(idx, (-1,))
        scatter_idx = tf.stack([row_ids, col_ids], axis=1)   # (B*2, 2)
        scatter_vals = tf.reshape(vals, (-1,))
        w_sparse = tf.scatter_nd(scatter_idx, scatter_vals, (B, self.num_experts))  # (B,E)

        # Load-balance loss (uniform usage)
        mean_usage = tf.reduce_mean(w_sparse, axis=0)
        uniform = tf.fill([self.num_experts], 1.0 / tf.cast(self.num_experts, tf.float32))
        lb_loss = tf.reduce_sum(tf.square(mean_usage - uniform))
        self.add_loss(self.lb_coef * lb_loss)

        # Entropy reg on soft weights (encourage exploration early)
        ent = -tf.reduce_mean(tf.reduce_sum(w_soft * tf.math.log(tf.clip_by_value(w_soft, 1e-8, 1.0)), axis=-1))
        self.add_loss(self.ent_coef * (-ent))  # minimize negative entropy

        return w_sparse, w_soft  # sparse mixture, full mixture (for diagnostics)

# ---- Temperature scheduler ----
class GateTemperatureScheduler(callbacks.Callback):
    def __init__(self, gate_layer, start_tau=2.0, end_tau=0.7, anneal_epochs=20):
        super().__init__()
        self.gate = gate_layer
        self.start_tau = float(start_tau)
        self.end_tau = float(end_tau)
        self.anneal_epochs = max(1, int(anneal_epochs))
    def on_epoch_begin(self, epoch, logs=None):
        p = min(1.0, epoch / self.anneal_epochs)
        tau = self.start_tau + p * (self.end_tau - self.start_tau)
        self.gate.tau.assign(tau)
        K.set_value(self.gate.tau, tau)
        if epoch % 5 == 0:
            print(f"[GateTemperatureScheduler] epoch={epoch} tau={tau:.3f}")

# ---- Single-input MoE builder (no domain hint) ----
def build_moe_single_input(
    input_shape, num_classes,
    tau_init=2.0, tau_final=0.7,
    gate_hidden=128, classifier_hidden=256,
    lb_coef=0.02, ent_coef=0.01,
    name="MoE_SingleInput"
):
    seq_in = layers.Input(shape=input_shape, name="seq")

    # Experts
    exp_nat = build_expert_tcn(seq_in, "exp_nat")
    exp_act = build_expert_tcn(seq_in, "exp_act")
    exp_shared = build_expert_tcn(seq_in, "exp_shared")

    # Stack experts into (B, E=3, F) using Lambda + Concatenate (avoid tf.stack on KerasTensors)
    e1 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_nat_expand")(exp_nat)       # (B,1,F)
    e2 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_act_expand")(exp_act)       # (B,1,F)
    e3 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_share_expand")(exp_shared)  # (B,1,F)
    E_stack = layers.Concatenate(axis=1, name="experts_stack")([e1, e2, e3])  # (B,3,F)

    # Gate input: pooled sequence only (single input mode)
    gate_in = layers.GlobalAveragePooling1D(name="gate_pool")(seq_in)

    gate_layer = Top2Gate(num_experts=3, gate_hidden=gate_hidden,
                          tau_init=tau_init, lb_coef=lb_coef, ent_coef=ent_coef, name="gate")
    w_sparse, w_soft = gate_layer(gate_in)   # (B,3), (B,3)

    # Weighted sum: (B,3,1) * (B,3,F) -> sum over experts -> (B,F)
    w_expanded = layers.Lambda(lambda z: tf.expand_dims(z, -1), name="w_expand")(w_sparse)  # (B,3,1)
    fused = layers.Lambda(lambda zs: tf.reduce_sum(zs[0] * zs[1], axis=1), name="fuse")([w_expanded, E_stack])  # (B,F)

    x = layers.Dense(classifier_hidden, activation="relu", name="clf_dense")(fused)
    x = layers.Dropout(0.3, name="clf_do")(x)
    y_out = layers.Dense(num_classes, activation="softmax", name="y")(x)

    model = models.Model(seq_in, y_out, name=name)
    model.compile(optimizer=tf.keras.optimizers.SGD(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Provide a temperature scheduler that targets this gate instance
    anneal_cb = GateTemperatureScheduler(gate_layer, start_tau=tau_init, end_tau=tau_final, anneal_epochs=20)
    return model, anneal_cb

## 7.3 Train the Teacher

Builds the MoE, attaches the temperature scheduler and early stopping (monitoring validation
accuracy, patience 8, restoring the best weights), trains, saves the checkpoint, and reports scores
on all three test sets: natural only, acted + EmojiHeroVR, and combined.

In [ ]:
for name, X, y in [
    ("train", X_train_np, Y_train_np),
    ("validation", X_val_np, Y_val_np),
    ("test", X_test_comb_np, Y_test_comb_np),
]:
    print(
        f"{name:10s}: "
        f"X={X.shape}, "
        f"y={y.shape}, "
        f"finite={np.isfinite(X).all()}, "
        f"classes={np.unique(y).tolist()}"
    )

    assert len(X) == len(y)
    assert X.shape[1:] == (5, 70)
    assert np.isfinite(X).all()
    assert set(np.unique(y)) == set(range(7))

print("Dataset is ready for training.")

In [ ]:
TEACHER_TAG = "teacher-70AU-MoE_3experts_2gate-with_15_participants-NOT-normalized_every_teacher_dataset-self_filtered_ONLY_NATURAL-EmojiHero_fixed"

TEACHER_MODEL_NAME = f"{TEACHER_TAG}.keras"
TEACHER_RUN_DIR    = Path(predictions_path) / TEACHER_TAG

### post-hoc logit adjustment for anger

In [ ]:
!pip -q install onnxruntime

In [ ]:
import json
import numpy as np
import pandas as pd
import onnxruntime as ort

from pathlib import Path
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

ONNX_PATH = Path(
    "/content/sparseVR_teacher_model.onnx"
)

assert ONNX_PATH.exists(), f"ONNX bulunamadı: {ONNX_PATH}"

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Input:", input_name, session.get_inputs()[0].shape)
print("Output:", output_name, session.get_outputs()[0].shape)

In [ ]:
def predict_onnx(session, X):
    X = np.asarray(X, dtype=np.float32)
    outputs = []

    # Even if the batch size is fixed, we pass the samples one by one for it to work.
    for i in range(len(X)):
        result = session.run(
            [output_name],
            {input_name: X[i:i+1]},
        )[0]

        outputs.append(np.asarray(result).reshape(-1, 7)[0])

    probabilities = np.asarray(outputs, dtype=np.float32)

    assert probabilities.shape == (len(X), 7)
    assert np.all(np.isfinite(probabilities))

    probability_sums = probabilities.sum(axis=1)

    print(
        "Probability sums:",
        probability_sums.min(),
        probability_sums.max(),
    )

    assert np.allclose(
        probability_sums,
        1.0,
        atol=1e-4,
    ), "The ONNX outputs are not probability distributions."

    return probabilities

In [ ]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
)

CLASS_NAMES = list(le.classes_)

EXPECTED_CLASS_ORDER = [
    "Anger",
    "Disgust",
    "Fear",
    "Happiness",
    "Neutral",
    "Sadness",
    "Surprise",
]

assert CLASS_NAMES == EXPECTED_CLASS_ORDER, (
    f"Unexpected class order: {CLASS_NAMES}"
)

ANGER_INDEX = CLASS_NAMES.index("Anger")


def to_arrays(X, y):
    X_np = stack_sequences(list(X))
    y_np = np.asarray(list(y), dtype=np.int64)

    assert len(X_np) == len(y_np)
    assert X_np.shape[1:] == (5, 70)

    return X_np, y_np


def select_domain(X, y, domains, wanted_domain):
    selected_X = []
    selected_y = []

    for window, label, domain in zip(X, y, domains):
        if domain == wanted_domain:
            selected_X.append(window)
            selected_y.append(label)

    return selected_X, selected_y


def print_class_counts(name, y):
    counts = np.bincount(
        np.asarray(y, dtype=np.int64),
        minlength=len(CLASS_NAMES),
    )

    print(f"\n{name}: total={len(y)}")

    for class_name, count in zip(CLASS_NAMES, counts):
        print(f"  {class_name:<10}: {count}")


# Eski own-acted
X_val_old_acted_np, Y_val_old_acted_np = to_arrays(
    X_val_act,
    Y_val_act,
)

X_test_old_acted_np, Y_test_old_acted_np = to_arrays(
    X_test_act,
    Y_test_act,
)

# Yeni katılımcılardan yalnızca acted.
# Notebook'ta domain 1 = acted.
X_val_new_acted, Y_val_new_acted = select_domain(
    X_val_new,
    Y_val_new,
    Y_val_domain_new,
    wanted_domain=1,
)

X_test_new_acted, Y_test_new_acted = select_domain(
    X_test_new,
    Y_test_new,
    Y_test_domain_new,
    wanted_domain=1,
)

X_val_new_acted_np, Y_val_new_acted_np = to_arrays(
    X_val_new_acted,
    Y_val_new_acted,
)

X_test_new_acted_np, Y_test_new_acted_np = to_arrays(
    X_test_new_acted,
    Y_test_new_acted,
)

# Birleşik own-acted
X_val_own_acted_np = np.concatenate(
    [
        X_val_old_acted_np,
        X_val_new_acted_np,
    ],
    axis=0,
)

Y_val_own_acted_np = np.concatenate(
    [
        Y_val_old_acted_np,
        Y_val_new_acted_np,
    ],
    axis=0,
)

X_test_own_acted_np = np.concatenate(
    [
        X_test_old_acted_np,
        X_test_new_acted_np,
    ],
    axis=0,
)

Y_test_own_acted_np = np.concatenate(
    [
        Y_test_old_acted_np,
        Y_test_new_acted_np,
    ],
    axis=0,
)

print_class_counts(
    "Old acted validation",
    Y_val_old_acted_np,
)

print_class_counts(
    "New acted validation",
    Y_val_new_acted_np,
)

print_class_counts(
    "Combined own-acted validation",
    Y_val_own_acted_np,
)

print_class_counts(
    "Combined own-acted test",
    Y_test_own_acted_np,
)

In [ ]:
def reproduce_old_acted_subject_split():
    all_files = all_files_list(DataType.OUR_ACTED)

    files, emotions = zip(*all_files)

    labels = np.asarray([
        le.transform([emotion.value])[0]
        for emotion in emotions
    ])

    groups = np.asarray([
        subject_from_path(path)
        for path in files
    ])

    splitter = StratifiedGroupKFold(
        n_splits=10,
        shuffle=True,
        random_state=42,
    )

    dummy_X = np.zeros((len(labels), 1))

    folds = [
        heldout_indices
        for _, heldout_indices in splitter.split(
            dummy_X,
            labels,
            groups=groups,
        )
    ]

    val_indices = folds[7]
    test_indices = np.concatenate(folds[8:])

    old_val_subjects = sorted(
        set(groups[val_indices])
    )

    old_test_subjects = sorted(
        set(groups[test_indices])
    )

    return old_val_subjects, old_test_subjects


old_val_subjects, old_test_subjects = (
    reproduce_old_acted_subject_split()
)

new_val_acted_records = new_val_records[
    new_val_records["recording_type"] == "ActingRecordings"
]

new_test_acted_records = new_test_records[
    new_test_records["recording_type"] == "ActingRecordings"
]

new_val_subjects = sorted(
    new_val_acted_records["participant"].unique()
)

new_test_subjects = sorted(
    new_test_acted_records["participant"].unique()
)

print("Old acted validation subjects:", old_val_subjects)
print("New acted validation subjects:", new_val_subjects)

print("Old acted test subjects:", old_test_subjects)
print("New acted test subjects:", new_test_subjects)

all_val_subjects = sorted(
    set(old_val_subjects) |
    set(new_val_subjects)
)

all_test_subjects = sorted(
    set(old_test_subjects) |
    set(new_test_subjects)
)

print("\nTotal own-acted validation participants:",
      len(all_val_subjects))

print("Total own-acted test participants:",
      len(all_test_subjects))

print("Validation participants:", all_val_subjects)
print("Test participants:", all_test_subjects)

assert set(all_val_subjects).isdisjoint(all_test_subjects)

In [ ]:
validation_probabilities = {
    "old_acted": predict_onnx(
        session,
        X_val_old_acted_np,
    ),

    "new_acted": predict_onnx(
        session,
        X_val_new_acted_np,
    ),

    "own_acted_combined": predict_onnx(
        session,
        X_val_own_acted_np,
    ),

    "emojihero": predict_onnx(
        session,
        stack_sequences(X_val_em_hero),
    ),
}

validation_labels = {
    "old_acted": Y_val_old_acted_np,
    "new_acted": Y_val_new_acted_np,
    "own_acted_combined": Y_val_own_acted_np,
    "emojihero": np.asarray(
        Y_val_em_hero,
        dtype=np.int64,
    ),
}

test_probabilities = {
    "old_acted": predict_onnx(
        session,
        X_test_old_acted_np,
    ),

    "new_acted": predict_onnx(
        session,
        X_test_new_acted_np,
    ),

    "own_acted_combined": predict_onnx(
        session,
        X_test_own_acted_np,
    ),

    "emojihero": predict_onnx(
        session,
        stack_sequences(X_test_em_hero),
    ),
}

test_labels = {
    "old_acted": Y_test_old_acted_np,
    "new_acted": Y_test_new_acted_np,
    "own_acted_combined": Y_test_own_acted_np,
    "emojihero": np.asarray(
        Y_test_em_hero,
        dtype=np.int64,
    ),
}

# Own acted + EmojiHero, yalnızca ikincil raporlama için
test_probabilities["all_acted"] = np.concatenate(
    [
        test_probabilities["own_acted_combined"],
        test_probabilities["emojihero"],
    ],
    axis=0,
)

test_labels["all_acted"] = np.concatenate(
    [
        test_labels["own_acted_combined"],
        test_labels["emojihero"],
    ],
    axis=0,
)

print("Probability generation complete.")

In [ ]:
def apply_anger_weight(probabilities, weight):
    raw_scores = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    decision_scores = raw_scores.copy()

    decision_scores[:, ANGER_INDEX] *= weight

    predictions = decision_scores.argmax(axis=1)

    adjusted_probabilities = (
        decision_scores /
        decision_scores.sum(axis=1, keepdims=True)
    )

    return predictions, adjusted_probabilities


def evaluate_weight(y_true, probabilities, weight):
    predictions, _ = apply_anger_weight(
        probabilities,
        weight,
    )

    report = classification_report(
        y_true,
        predictions,
        labels=range(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    result = {
        "weight": float(weight),

        "macro_f1": f1_score(
            y_true,
            predictions,
            labels=range(len(CLASS_NAMES)),
            average="macro",
            zero_division=0,
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions,
        ),

        "anger_precision": report["Anger"]["precision"],
        "anger_recall": report["Anger"]["recall"],
        "anger_f1": report["Anger"]["f1-score"],
    }

    for class_name in CLASS_NAMES:
        result[f"{class_name.lower()}_recall"] = (
            report[class_name]["recall"]
        )

    return result


def scan_weights(y_true, probabilities):
    weights = np.round(
        np.arange(1.0, 4.001, 0.01),
        2,
    )

    rows = [
        evaluate_weight(
            y_true,
            probabilities,
            weight,
        )
        for weight in weights
    ]

    results = pd.DataFrame(rows)

    best_macro_f1 = results["macro_f1"].max()

    candidates = results[
        np.isclose(
            results["macro_f1"],
            best_macro_f1,
            atol=1e-12,
        )
    ]
    best = candidates.sort_values("weight").iloc[0]

    return results, best

In [ ]:
validation_scans = {}
validation_best_rows = {}

for source_name in [
    "old_acted",
    "new_acted",
    "own_acted_combined",
]:
    scan, best = scan_weights(
        validation_labels[source_name],
        validation_probabilities[source_name],
    )

    validation_scans[source_name] = scan
    validation_best_rows[source_name] = best

best_summary = pd.DataFrame([
    {
        "source": source_name,
        **best.to_dict(),
    }
    for source_name, best
    in validation_best_rows.items()
])

display(
    best_summary[
        [
            "source",
            "weight",
            "macro_f1",
            "balanced_accuracy",
            "anger_precision",
            "anger_recall",
            "anger_f1",
        ]
    ].round(4)
)

selected_weight = float(
    validation_best_rows[
        "own_acted_combined"
    ]["weight"]
)

print("Candidate weight:", selected_weight)

In [ ]:
validation_comparison_rows = []

for source_name, probabilities in (
    validation_probabilities.items()
):
    y_true = validation_labels[source_name]

    baseline = evaluate_weight(
        y_true,
        probabilities,
        1.0,
    )

    adjusted = evaluate_weight(
        y_true,
        probabilities,
        selected_weight,
    )

    validation_comparison_rows.append({
        "source": source_name,
        "version": "baseline",
        **baseline,
    })

    validation_comparison_rows.append({
        "source": source_name,
        "version": "adjusted",
        **adjusted,
    })

validation_comparison = pd.DataFrame(
    validation_comparison_rows
)

display(
    validation_comparison[
        [
            "source",
            "version",
            "weight",
            "macro_f1",
            "balanced_accuracy",
            "anger_precision",
            "anger_recall",
            "anger_f1",
            "neutral_recall",
            "disgust_recall",
        ]
    ].round(4)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_confusion(
    source_name,
    weight,
    title,
):
    y_true = validation_labels[source_name]
    probabilities = validation_probabilities[source_name]

    predictions, _ = apply_anger_weight(
        probabilities,
        weight,
    )

    matrix = confusion_matrix(
        y_true,
        predictions,
        labels=range(len(CLASS_NAMES)),
    )

    plt.figure(figsize=(9, 7))

    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_confusion(
    "own_acted_combined",
    1.0,
    "Own-acted validation — baseline",
)

plot_confusion(
    "own_acted_combined",
    selected_weight,
    f"Own-acted validation — Anger weight={selected_weight}",
)

plot_confusion(
    "emojihero",
    selected_weight,
    f"EmojiHero validation — Anger weight={selected_weight}",
)

In [ ]:
print("Guarded candidate:", guarded_weight)

available_columns = guarded_row.index.tolist()
print("Available columns:")
print(available_columns)

display(
    guarded_row.to_frame().T[
        [
            "weight",
            "macro_f1_own_acted",
            "anger_precision",
            "anger_recall",
            "anger_f1_own_acted",
            "macro_f1_emojihero",
            "anger_f1_emojihero",
            "disgust_recall_emojihero",
            "neutral_recall_emojihero",
        ]
    ].round(4)
)

In [ ]:
# Baseline values
own_baseline = evaluate_weight(
    validation_labels["own_acted_combined"],
    validation_probabilities["own_acted_combined"],
    1.0,
)

emojihero_baseline = evaluate_weight(
    validation_labels["emojihero"],
    validation_probabilities["emojihero"],
    1.0,
)

# For a coefficient to be accepted:
# 1. Own-acted macro-F1 must not decrease
# 2. Own-acted Anger recall must increase
# 3. Own-acted Anger F1 must increase
# 4. EmojiHero macro-F1 must not decrease
acceptable_weights = guardrail_table[
    (
        guardrail_table["macro_f1_own_acted"]
        >= own_baseline["macro_f1"]
    )
    &
    (
        guardrail_table["anger_recall"]
        > own_baseline["anger_recall"]
    )
    &
    (
        guardrail_table["anger_f1_own_acted"]
        > own_baseline["anger_f1"]
    )
    &
    (
        guardrail_table["macro_f1_emojihero"]
        >= emojihero_baseline["macro_f1"]
    )
].copy()

print("Own-acted baseline:")
print({
    "macro_f1": own_baseline["macro_f1"],
    "anger_precision": own_baseline["anger_precision"],
    "anger_recall": own_baseline["anger_recall"],
    "anger_f1": own_baseline["anger_f1"],
})

print("\nEmojiHero baseline:")
print({
    "macro_f1": emojihero_baseline["macro_f1"],
    "anger_precision": emojihero_baseline["anger_precision"],
    "anger_recall": emojihero_baseline["anger_recall"],
    "anger_f1": emojihero_baseline["anger_f1"],
})

print(
    "\nNumber of weights satisfying all criteria:",
    len(acceptable_weights),
)

if acceptable_weights.empty:
    print("\nDECISION: REJECTED")
    print(
        "No Anger weight improves own-acted Anger "
        "while preserving overall performance on both domains."
    )
    final_validation_decision = "rejected"
    final_weight = 1.0
else:
    display(
        acceptable_weights[
            [
                "weight",
                "macro_f1_own_acted",
                "anger_precision",
                "anger_recall",
                "anger_f1_own_acted",
                "macro_f1_emojihero",
                "anger_f1_emojihero",
            ]
        ].round(4)
    )

    final_validation_decision = "candidate_found"
    final_weight = None

In [ ]:
adjustment_path = (
    Path(predictions_path) /
    "anger_decision_adjustment"
)

adjustment_path.mkdir(
    parents=True,
    exist_ok=True,
)

guardrail_table.to_csv(
    adjustment_path /
    "validation_guardrail_scan.csv",
    index=False,
)

validation_comparison.to_csv(
    adjustment_path /
    "validation_baseline_vs_candidates.csv",
    index=False,
)

validation_decision = {
    "status": final_validation_decision,
    "deployed_weight": 1.0,
    "tested_candidate_own_acted": 1.55,
    "tested_candidate_with_emojihero_guardrail": 1.14,
    "reason": (
        "No candidate increased own-acted Anger recall and F1 "
        "while preserving macro-F1 on own-acted and EmojiHero validation."
    ),
    "test_set_opened": False,
    "next_step": (
        "Investigate offline-to-live domain and preprocessing shift."
    ),
    "model": ONNX_PATH.name,
}

with open(
    adjustment_path /
    "validation_decision.json",
    "w",
) as file:
    json.dump(
        validation_decision,
        file,
        indent=2,
    )

print(json.dumps(validation_decision, indent=2))
print("\nSaved to:", adjustment_path)

In [ ]:
import numpy as np


def roughness(W, min_std=1e-3):
    """Temporal roughness of a window set.

    W : array (n_windows, n_frames, n_channels)

    For each channel, computes std(frame-to-frame difference) / std(signal)
    within each window, then averages over windows. Smoothed (time-averaged)
    input gives a low ratio; independently sampled input gives a high one.
    White noise tends to sqrt(2) ~ 1.41.
    """
    W = np.asarray(W, dtype=np.float64)

    ratios = []
    for channel in range(W.shape[2]):
        x = W[:, :, channel]

        signal_std = x.std(axis=1)
        diff_std = np.diff(x, axis=1).std(axis=1)

        active = signal_std > min_std
        if active.sum() >= 5:
            ratios.append((diff_std[active] / signal_std[active]).mean())

    return np.array(ratios)


sources = {
    "old acted    (unknown)":       X_val_old_acted_np,
    "new acted    (mean of 6)":     X_val_new_acted_np,
    "EmojiHeroVR  (mean of 3)":     stack_sequences(X_val_em_hero),
    "old natural  (unknown)":       stack_sequences(list(X_val_nat)),
}

print(f"{'source':<28} {'windows':>8} {'channels':>9} {'mean r':>8} {'median r':>10}")
print("-" * 67)

for name, windows in sources.items():
    r = roughness(windows)
    print(f"{name:<28} {len(windows):>8} {len(r):>9} "
          f"{r.mean():>8.3f} {np.median(r):>10.3f}")

print()
print("Live headset reference (measured on logged sessions): mean r = 1.18 - 1.25")
print("White noise ~ 1.41   |   heavily smoothed << 1")


In [ ]:
import numpy as np
import pandas as pd
from glob import glob

print(f"{'participant':<18} {'file':<22} {'rows':>6} "
      f"{'median step':>12} {'unique steps (first 5)'}")
print("-" * 90)

for path in sorted(glob(f"{data_folder_path}/*/ACT_*.csv"))[:5]:
    parts = path.replace("\\", "/").split("/")
    participant, filename = parts[-2], parts[-1]

    timestamps = pd.read_csv(path).iloc[:, 0].astype(float).to_numpy()
    steps = np.diff(timestamps)

    print(f"{participant:<18} {filename:<22} {len(timestamps):>6} "
          f"{np.median(steps):>12.3f} {np.unique(steps)[:5]}")

print()
print("Median step ~100  -> already 10 Hz")
print("Median step ~16.7 -> 60 Hz, and the notebook never downsamples it")


----------------------------------

In [ ]:
# ============================================================
# COMPARE 3 ONNX MODELS ON ACTED-ONLY HELD-OUT TEST DATA
#
# Acted domain includes:
#   - old acted test data
#   - EmojiHeroVR test data
#   - held-out new-participant ActingRecordings
#
# Natural / VideoRecordings are excluded.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)

try:
    import onnxruntime as ort
except ImportError:
    raise ImportError(
        "onnxruntime is not installed. Run this once in a separate cell:\n"
        "!pip install -q onnxruntime"
    )


# ------------------------------------------------------------
# 1. PUT YOUR THREE ONNX MODEL PATHS HERE
# ------------------------------------------------------------

MODEL_PATHS = {
    "model_1": Path(
        "/content/act_trial_92.onnx"
    ),
    "model_2": Path(
        "/content/natural_trial_96.onnx"
    ),
    "model_3": Path(
        "/content/sparseVR_teacher_model.onnx"
    ),
}


# ------------------------------------------------------------
# 2. CLASS ORDER
#
# This must match the Keras/ONNX/Unity output order.
# ------------------------------------------------------------

CLASS_NAMES = [
    "Anger",
    "Disgust",
    "Fear",
    "Happiness",
    "Neutral",
    "Sadness",
    "Surprise",
]

CLASS_LABELS = np.arange(len(CLASS_NAMES))


# ------------------------------------------------------------
# 3. BUILD THE ACTED-ONLY TEST SET
# ------------------------------------------------------------

required_variables = [
    "X_test_comb_np",
    "Y_test_comb_np",
    "Y_test_domain_comb_np",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the dataset-construction cells first. "
        f"Missing variables: {missing_variables}"
    )


X_test_all = np.asarray(
    X_test_comb_np,
    dtype=np.float32,
)

y_test_all = np.asarray(
    Y_test_comb_np,
    dtype=np.int64,
).reshape(-1)

domain_test_all = np.asarray(
    Y_test_domain_comb_np,
    dtype=np.int64,
).reshape(-1)


assert len(X_test_all) == len(y_test_all)
assert len(y_test_all) == len(domain_test_all)
assert X_test_all.shape[1:] == (5, 70)
assert set(np.unique(domain_test_all)).issubset({0, 1})


acted_mask = domain_test_all == 1

X_acted = np.ascontiguousarray(
    X_test_all[acted_mask],
    dtype=np.float32,
)

y_acted = y_test_all[acted_mask]


assert len(X_acted) > 0
assert set(np.unique(y_acted)) == set(CLASS_LABELS)

print("Acted-only test set")
print("--------------------")
print("X shape:", X_acted.shape)
print("y shape:", y_acted.shape)

print("\nClass distribution:")
for class_index, class_name in enumerate(CLASS_NAMES):
    count = int(np.sum(y_acted == class_index))
    print(f"  {class_name:12s}: {count}")


# ------------------------------------------------------------
# 4. HELPER FUNCTIONS
# ------------------------------------------------------------

def softmax(logits):
    """Numerically stable softmax."""
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)

    return (
        exp_values
        / np.sum(exp_values, axis=1, keepdims=True)
    ).astype(np.float32)


def convert_to_probabilities(raw_output):
    """
    Keep valid probability outputs unchanged.

    If the model returns logits, apply softmax for probability-based
    metrics. Argmax is unchanged by this conversion.
    """
    raw_output = np.asarray(raw_output, dtype=np.float32)

    if raw_output.ndim > 2:
        raw_output = np.squeeze(raw_output)

    if raw_output.ndim == 1:
        raw_output = raw_output.reshape(1, -1)

    if raw_output.shape[1] != len(CLASS_NAMES):
        raise ValueError(
            "Expected 7 model outputs, "
            f"but received shape {raw_output.shape}"
        )

    row_sums = raw_output.sum(axis=1)

    already_probabilities = (
        np.all(np.isfinite(raw_output))
        and np.all(raw_output >= -1e-6)
        and np.all(raw_output <= 1.0 + 1e-6)
        and np.allclose(
            row_sums,
            1.0,
            atol=1e-4,
            rtol=1e-4,
        )
    )

    if already_probabilities:
        return raw_output, "probabilities"

    return softmax(raw_output), "logits → softmax"


def run_onnx_model(model_path, X):
    """
    Run an ONNX model.

    First attempts batched inference. If the model has a fixed batch
    size of 1, it automatically falls back to one window at a time.
    """
    if not model_path.exists():
        raise FileNotFoundError(
            f"ONNX model not found:\n{model_path}"
        )

    session = ort.InferenceSession(
        str(model_path),
        providers=["CPUExecutionProvider"],
    )

    inputs = session.get_inputs()
    outputs = session.get_outputs()

    if len(inputs) != 1:
        raise ValueError(
            f"{model_path.name} has {len(inputs)} inputs; expected 1."
        )

    input_meta = inputs[0]
    output_meta = outputs[0]

    input_name = input_meta.name
    output_name = output_meta.name

    print(f"\nLoading: {model_path.name}")
    print("  input name: ", input_name)
    print("  input shape:", input_meta.shape)
    print("  output name:", output_name)
    print("  output shape:", output_meta.shape)

    try:
        raw_output = session.run(
            [output_name],
            {input_name: X},
        )[0]

        inference_mode = "full batch"

    except Exception as batch_error:
        print(
            "  Full-batch inference was unavailable; "
            "falling back to batch size 1."
        )

        predictions = []

        for index in range(len(X)):
            sample = X[index:index + 1]

            sample_output = session.run(
                [output_name],
                {input_name: sample},
            )[0]

            predictions.append(
                np.asarray(sample_output).reshape(1, -1)
            )

        raw_output = np.concatenate(
            predictions,
            axis=0,
        )

        inference_mode = "batch size 1"

    raw_output = np.asarray(raw_output)

    if raw_output.ndim > 2:
        raw_output = np.squeeze(raw_output)

    if raw_output.ndim == 1:
        raw_output = raw_output.reshape(1, -1)

    probabilities, output_type = convert_to_probabilities(
        raw_output
    )

    return {
        "session": session,
        "raw_output": raw_output,
        "probabilities": probabilities,
        "output_type": output_type,
        "inference_mode": inference_mode,
        "input_name": input_name,
        "output_name": output_name,
    }


# ------------------------------------------------------------
# 5. RUN AND EVALUATE ALL THREE MODELS
# ------------------------------------------------------------

summary_rows = []
per_class_rows = []
all_results = {}

for model_name, model_path in MODEL_PATHS.items():

    inference_result = run_onnx_model(
        model_path,
        X_acted,
    )

    probabilities = inference_result["probabilities"]

    if len(probabilities) != len(y_acted):
        raise ValueError(
            f"{model_name}: expected {len(y_acted)} predictions, "
            f"received {len(probabilities)}."
        )

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    report = classification_report(
        y_acted,
        predictions,
        labels=CLASS_LABELS,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_acted,
        predictions,
        labels=CLASS_LABELS,
    )

    accuracy = accuracy_score(
        y_acted,
        predictions,
    )

    balanced_accuracy = balanced_accuracy_score(
        y_acted,
        predictions,
    )

    macro_f1 = f1_score(
        y_acted,
        predictions,
        labels=CLASS_LABELS,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_acted,
        predictions,
        labels=CLASS_LABELS,
        average="weighted",
        zero_division=0,
    )

    model_log_loss = log_loss(
        y_acted,
        probabilities,
        labels=CLASS_LABELS,
    )

    summary_rows.append({
        "model": model_name,
        "file": model_path.name,
        "n_samples": len(y_acted),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "log_loss": model_log_loss,
        "output_type": inference_result["output_type"],
        "inference_mode": inference_result["inference_mode"],
    })

    for class_name in CLASS_NAMES:
        class_result = report[class_name]

        per_class_rows.append({
            "model": model_name,
            "class": class_name,
            "precision": class_result["precision"],
            "recall": class_result["recall"],
            "f1": class_result["f1-score"],
            "support": int(class_result["support"]),
        })

    all_results[model_name] = {
        "model_path": model_path,
        "probabilities": probabilities,
        "predictions": predictions,
        "classification_report": report,
        "confusion_matrix": cm,
    }

    print("\n" + "=" * 72)
    print(f"{model_name}: {model_path.name}")
    print("=" * 72)

    print(
        classification_report(
            y_acted,
            predictions,
            labels=CLASS_LABELS,
            target_names=CLASS_NAMES,
            zero_division=0,
        )
    )


# ------------------------------------------------------------
# 6. OVERALL COMPARISON
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows).sort_values(
    "macro_f1",
    ascending=False,
).reset_index(drop=True)

print("\n\nOVERALL COMPARISON")
display(
    summary_df.style.format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "weighted_f1": "{:.4f}",
        "log_loss": "{:.4f}",
    })
)


# ------------------------------------------------------------
# 7. PER-CLASS F1 COMPARISON
# ------------------------------------------------------------

per_class_df = pd.DataFrame(per_class_rows)

f1_comparison = (
    per_class_df
    .pivot(
        index="class",
        columns="model",
        values="f1",
    )
    .reindex(CLASS_NAMES)
)

recall_comparison = (
    per_class_df
    .pivot(
        index="class",
        columns="model",
        values="recall",
    )
    .reindex(CLASS_NAMES)
)

precision_comparison = (
    per_class_df
    .pivot(
        index="class",
        columns="model",
        values="precision",
    )
    .reindex(CLASS_NAMES)
)

print("\nPER-CLASS F1")
display(f1_comparison.style.format("{:.4f}"))

print("\nPER-CLASS RECALL")
display(recall_comparison.style.format("{:.4f}"))

print("\nPER-CLASS PRECISION")
display(precision_comparison.style.format("{:.4f}"))


# ------------------------------------------------------------
# 8. CONFUSION MATRICES
# ------------------------------------------------------------

number_of_models = len(MODEL_PATHS)

fig, axes = plt.subplots(
    1,
    number_of_models,
    figsize=(7 * number_of_models, 6),
)

if number_of_models == 1:
    axes = [axes]

for axis, model_name in zip(axes, MODEL_PATHS.keys()):
    cm = all_results[model_name]["confusion_matrix"]

    cm_normalized = (
        cm.astype(np.float64)
        / np.maximum(
            cm.sum(axis=1, keepdims=True),
            1,
        )
    )

    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )

    axis.set_title(
        f"{model_name}\nActed-only test"
    )
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. F1 BAR CHART
# ------------------------------------------------------------

ax = f1_comparison.plot(
    kind="bar",
    figsize=(13, 6),
    ylim=(0, 1),
)

ax.set_title(
    "ONNX models — per-class F1 on acted-only held-out test data"
)
ax.set_xlabel("Emotion")
ax.set_ylabel("F1 score")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. OPTIONAL: PAIRWISE PREDICTION AGREEMENT
# ------------------------------------------------------------

model_names = list(MODEL_PATHS.keys())

agreement_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float,
)

for model_a in model_names:
    for model_b in model_names:
        prediction_a = all_results[model_a]["predictions"]
        prediction_b = all_results[model_b]["predictions"]

        agreement_matrix.loc[model_a, model_b] = np.mean(
            prediction_a == prediction_b
        )

print("\nPAIRWISE ARGMAX AGREEMENT")
display(agreement_matrix.style.format("{:.2%}"))

In [ ]:
# ============================================================
# 11. SAVE ALL COMPARISON RESULTS TO GOOGLE DRIVE
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import json
import re


BASE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "outputs_15_participants_70AU-NOT_normalize_teacher_data-"
    "self_filtered_NATURAL_ONLY-EmojiHero_Fixed"
)

RUN_TIMESTAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

RESULTS_DIR = (
    BASE_OUTPUT_DIR
    / "onnx_acted_only_comparison"
    / f"run_{RUN_TIMESTAMP}"
)

RAW_RESULTS_DIR = RESULTS_DIR / "raw_predictions"
CONFUSION_DIR = RESULTS_DIR / "confusion_matrices"
REPORTS_DIR = RESULTS_DIR / "classification_reports"
FIGURES_DIR = RESULTS_DIR / "figures"

for directory in [
    RESULTS_DIR,
    RAW_RESULTS_DIR,
    CONFUSION_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def safe_filename(value):
    """Convert model names into filesystem-safe names."""
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


# ------------------------------------------------------------
# Save overall and per-class tables
# ------------------------------------------------------------

summary_df.to_csv(
    RESULTS_DIR / "overall_metrics.csv",
    index=False,
)

per_class_df.to_csv(
    RESULTS_DIR / "per_class_metrics.csv",
    index=False,
)

f1_comparison.to_csv(
    RESULTS_DIR / "per_class_f1_comparison.csv"
)

recall_comparison.to_csv(
    RESULTS_DIR / "per_class_recall_comparison.csv"
)

precision_comparison.to_csv(
    RESULTS_DIR / "per_class_precision_comparison.csv"
)

agreement_matrix.to_csv(
    RESULTS_DIR / "pairwise_argmax_agreement.csv"
)


# ------------------------------------------------------------
# Save results separately for every model
# ------------------------------------------------------------

models_metadata = {}

for model_name, result in all_results.items():

    safe_name = safe_filename(model_name)
    model_path = Path(result["model_path"])

    probabilities = np.asarray(
        result["probabilities"],
        dtype=np.float32,
    )

    predictions = np.asarray(
        result["predictions"],
        dtype=np.int64,
    )

    cm = np.asarray(
        result["confusion_matrix"],
        dtype=np.int64,
    )

    cm_normalized = (
        cm.astype(np.float64)
        / np.maximum(
            cm.sum(axis=1, keepdims=True),
            1,
        )
    )

    # Raw labels, predictions and probabilities
    np.savez_compressed(
        RAW_RESULTS_DIR / f"{safe_name}.npz",
        y_true=np.asarray(y_acted, dtype=np.int64),
        y_pred=predictions,
        y_proba=probabilities,
        confusion_matrix=cm,
        class_names=np.asarray(CLASS_NAMES, dtype=str),
    )

    # Classification report as JSON
    with open(
        REPORTS_DIR / f"{safe_name}.json",
        mode="w",
        encoding="utf-8",
    ) as file:
        json.dump(
            result["classification_report"],
            file,
            indent=2,
            default=float,
        )

    # Classification report as CSV
    report_df = pd.DataFrame(
        result["classification_report"]
    ).T

    report_df.to_csv(
        REPORTS_DIR / f"{safe_name}.csv"
    )

    # Raw confusion matrix
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{name}" for name in CLASS_NAMES],
        columns=[f"pred_{name}" for name in CLASS_NAMES],
    )

    cm_df.to_csv(
        CONFUSION_DIR / f"{safe_name}_raw.csv"
    )

    # Normalized confusion matrix
    cm_normalized_df = pd.DataFrame(
        cm_normalized,
        index=[f"true_{name}" for name in CLASS_NAMES],
        columns=[f"pred_{name}" for name in CLASS_NAMES],
    )

    cm_normalized_df.to_csv(
        CONFUSION_DIR / f"{safe_name}_normalized.csv"
    )

    models_metadata[model_name] = {
        "onnx_path": str(model_path),
        "onnx_filename": model_path.name,
        "raw_results_file": (
            f"raw_predictions/{safe_name}.npz"
        ),
        "classification_report_json": (
            f"classification_reports/{safe_name}.json"
        ),
        "classification_report_csv": (
            f"classification_reports/{safe_name}.csv"
        ),
        "confusion_matrix_raw": (
            f"confusion_matrices/{safe_name}_raw.csv"
        ),
        "confusion_matrix_normalized": (
            f"confusion_matrices/{safe_name}_normalized.csv"
        ),
    }


# ------------------------------------------------------------
# Save normalized confusion-matrix figure
# ------------------------------------------------------------

number_of_models = len(MODEL_PATHS)

fig, axes = plt.subplots(
    1,
    number_of_models,
    figsize=(7 * number_of_models, 6),
)

if number_of_models == 1:
    axes = [axes]

for axis, model_name in zip(axes, MODEL_PATHS.keys()):

    cm = np.asarray(
        all_results[model_name]["confusion_matrix"],
        dtype=np.float64,
    )

    cm_normalized = (
        cm
        / np.maximum(
            cm.sum(axis=1, keepdims=True),
            1,
        )
    )

    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )

    axis.set_title(
        f"{model_name}\nActed-only held-out test"
    )
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")

plt.tight_layout()

confusion_figure_path = (
    FIGURES_DIR
    / "normalized_confusion_matrices.png"
)

fig.savefig(
    confusion_figure_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Save per-class F1 figure
# ------------------------------------------------------------

fig, axis = plt.subplots(figsize=(13, 6))

f1_comparison.plot(
    kind="bar",
    ylim=(0, 1),
    ax=axis,
)

axis.set_title(
    "ONNX models — per-class F1 on acted-only held-out test"
)
axis.set_xlabel("Emotion")
axis.set_ylabel("F1 score")
axis.grid(axis="y", alpha=0.3)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()

f1_figure_path = (
    FIGURES_DIR
    / "per_class_f1_comparison.png"
)

fig.savefig(
    f1_figure_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Save run metadata
# ------------------------------------------------------------

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "test_set": {
        "name": "acted_only_held_out_test",
        "description": (
            "Windows selected from the combined held-out test "
            "corpus where domain == 1. This includes old acted, "
            "EmojiHeroVR, and held-out new-participant acted "
            "recordings. Natural recordings are excluded."
        ),
        "n_samples": int(len(y_acted)),
        "input_shape": list(X_acted.shape),
        "single_window_shape": list(X_acted.shape[1:]),
    },

    "class_order": CLASS_NAMES,

    "class_distribution": {
        class_name: int(
            np.sum(y_acted == class_index)
        )
        for class_index, class_name
        in enumerate(CLASS_NAMES)
    },

    "models": models_metadata,

    "primary_metric": "macro_f1",

    "focus_classes": [
        "Fear",
        "Anger",
    ],

    "generated_files": {
        "overall_metrics": "overall_metrics.csv",
        "per_class_metrics": "per_class_metrics.csv",
        "per_class_f1": "per_class_f1_comparison.csv",
        "per_class_recall": (
            "per_class_recall_comparison.csv"
        ),
        "per_class_precision": (
            "per_class_precision_comparison.csv"
        ),
        "pairwise_agreement": (
            "pairwise_argmax_agreement.csv"
        ),
        "normalized_confusion_figure": (
            "figures/normalized_confusion_matrices.png"
        ),
        "per_class_f1_figure": (
            "figures/per_class_f1_comparison.png"
        ),
    },
}

with open(
    RESULTS_DIR / "run_metadata.json",
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        default=float,
    )


# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("RESULTS SAVED SUCCESSFULLY")
print("=" * 72)
print(RESULTS_DIR)

print("\nSaved files:")

for saved_file in sorted(
    RESULTS_DIR.rglob("*")
):
    if saved_file.is_file():
        print(
            " -",
            saved_file.relative_to(RESULTS_DIR),
        )

In [ ]:
# ============================================================================
# LOAD THE SAVED TEACHER AND TEST ON ACTED TEST DATA ONLY
#
# Domain convention:
#   0 = natural
#   1 = acted
#
# EmojiHeroVR is classified as acted in this notebook.
# This cell performs inference only—no training or tuning.
# ============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ---------------------------------------------------------------------------
# 1. Select only acted windows from the complete held-out test corpus
# ---------------------------------------------------------------------------

X_test_all = np.asarray(
    X_test_comb_np,
    dtype=np.float32,
)

y_test_all = np.asarray(
    Y_test_comb_np,
    dtype=np.int64,
).reshape(-1)

domain_test_all = np.asarray(
    Y_test_domain_comb_np,
    dtype=np.int64,
).reshape(-1)


assert len(X_test_all) == len(y_test_all) == len(domain_test_all)

assert set(np.unique(domain_test_all)).issubset({0, 1}), (
    f"Unexpected domain labels: {np.unique(domain_test_all)}"
)


acted_mask = domain_test_all == 1

X_test_acted_only = X_test_all[acted_mask]
Y_test_acted_only = y_test_all[acted_mask]


assert len(Y_test_acted_only) > 0, (
    "No acted test windows were found."
)

assert X_test_acted_only.shape[1:] == (
    SEQ_LENGTH,
    len(AU_NAMES),
), X_test_acted_only.shape


# ---------------------------------------------------------------------------
# 2. Load the saved teacher model
# ---------------------------------------------------------------------------

teacher_model_path = Path(
    "/content/drive/MyDrive/"
    "outputs_15_participants_70AU-NOT_normalize_teacher_data-"
    "self_filtered_NATURAL_ONLY-EmojiHero_Fixed/"
    "models/"
    "teacher-70AU-MoE_3experts_2gate-with_15_participants-"
    "NOT-normalized_every_teacher_dataset-"
    "self_filtered_ONLY_NATURAL-EmojiHero_fixed.keras"
)


if not teacher_model_path.exists():
    raise FileNotFoundError(
        f"Teacher model was not found:\n{teacher_model_path}"
    )


# Rebuild the exact teacher architecture.
acted_teacher_model, _ = build_moe_single_input(
    input_shape=(
        SEQ_LENGTH,
        input_size,
    ),
    num_classes=num_classes,
    tau_init=2.0,
    tau_final=0.7,
    gate_hidden=128,
    classifier_hidden=256,
    lb_coef=0.02,
    ent_coef=0.01,
)


# Restore the saved teacher weights.
acted_teacher_model.load_weights(
    teacher_model_path
)

print("Loaded teacher model:")
print(teacher_model_path)


# ---------------------------------------------------------------------------
# 3. Evaluate the teacher on acted test data only
# ---------------------------------------------------------------------------

acted_test_loss, _ = acted_teacher_model.evaluate(
    X_test_acted_only,
    Y_test_acted_only,
    batch_size=64,
    verbose=0,
)


acted_probabilities = acted_teacher_model.predict(
    X_test_acted_only,
    batch_size=64,
    verbose=0,
)

acted_predictions = np.argmax(
    acted_probabilities,
    axis=1,
)


# ---------------------------------------------------------------------------
# 4. Calculate evaluation metrics
# ---------------------------------------------------------------------------

class_names = [
    emotion.value
    for emotion in Emotion
]

class_labels = np.arange(
    len(class_names)
)


acted_summary = pd.DataFrame([
    {
        "model": teacher_model_path.name,
        "test_set": "acted_only_including_EmojiHero",
        "n_samples": int(
            len(Y_test_acted_only)
        ),
        "loss": float(
            acted_test_loss
        ),
        "accuracy": accuracy_score(
            Y_test_acted_only,
            acted_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            Y_test_acted_only,
            acted_predictions,
        ),
        "precision_macro": precision_score(
            Y_test_acted_only,
            acted_predictions,
            labels=class_labels,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            Y_test_acted_only,
            acted_predictions,
            labels=class_labels,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            Y_test_acted_only,
            acted_predictions,
            labels=class_labels,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            Y_test_acted_only,
            acted_predictions,
            labels=class_labels,
            average="weighted",
            zero_division=0,
        ),
    }
])


print(
    f"\nActed-only test windows: "
    f"{len(Y_test_acted_only):,}"
)

display(acted_summary)


# ---------------------------------------------------------------------------
# 5. Per-class results
# ---------------------------------------------------------------------------

print("\nClassification report:")

print(
    classification_report(
        Y_test_acted_only,
        acted_predictions,
        labels=class_labels,
        target_names=class_names,
        zero_division=0,
    )
)


# ---------------------------------------------------------------------------
# 6. Confusion matrix
# ---------------------------------------------------------------------------

acted_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        Y_test_acted_only,
        acted_predictions,
        labels=class_labels,
    ),
    index=[
        f"true_{name}"
        for name in class_names
    ],
    columns=[
        f"pred_{name}"
        for name in class_names
    ],
)

display(acted_confusion_matrix)

In [ ]:
# ============================================================================
# SAVE ACTED-ONLY TEACHER TEST RESULTS
# ============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix


acted_results_path = Path(
    "/content/drive/MyDrive/"
    "outputs_15_participants_70AU-NOT_normalize_teacher_data-"
    "self_filtered_NATURAL_ONLY-EmojiHero_Fixed/"
    "predictions/"
    "teacher-70AU-MoE_3experts_2gate-with_15_participants-"
    "NOT-normalized_every_teacher_dataset-"
    "self_filtered_ONLY_NATURAL-EmojiHero_fixed/"
    "results_full_metrics"
)

acted_results_path.mkdir(
    parents=True,
    exist_ok=True,
)


# Confirm that the previous evaluation cell was run.
required_variables = [
    "acted_summary",
    "Y_test_acted_only",
    "acted_predictions",
    "acted_probabilities",
    "class_names",
    "class_labels",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the acted-only evaluation cell first. "
        f"Missing variables: {missing_variables}"
    )


# ---------------------------------------------------------------------------
# Per-class metrics
# ---------------------------------------------------------------------------

acted_report_dict = classification_report(
    Y_test_acted_only,
    acted_predictions,
    labels=class_labels,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

acted_per_class_df = (
    pd.DataFrame(acted_report_dict)
    .T
)


# ---------------------------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------------------------

acted_cm_array = confusion_matrix(
    Y_test_acted_only,
    acted_predictions,
    labels=class_labels,
)

acted_cm_df = pd.DataFrame(
    acted_cm_array,
    index=[
        f"true_{name}"
        for name in class_names
    ],
    columns=[
        f"pred_{name}"
        for name in class_names
    ],
)


# ---------------------------------------------------------------------------
# Save scalar metrics
# ---------------------------------------------------------------------------

acted_summary.to_csv(
    acted_results_path
    / "acted_only_scalar_metrics.csv",
    index=False,
)

with open(
    acted_results_path
    / "acted_only_scalar_metrics.json",
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        acted_summary.iloc[0].to_dict(),
        file,
        indent=2,
        default=float,
    )


# ---------------------------------------------------------------------------
# Save per-class metrics and confusion matrix
# ---------------------------------------------------------------------------

acted_per_class_df.to_csv(
    acted_results_path
    / "acted_only_classification_report.csv"
)

with open(
    acted_results_path
    / "acted_only_classification_report.json",
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        acted_report_dict,
        file,
        indent=2,
        default=float,
    )

acted_cm_df.to_csv(
    acted_results_path
    / "acted_only_confusion_matrix.csv"
)


# ---------------------------------------------------------------------------
# Save labels, predictions, probabilities, and confusion matrix
# ---------------------------------------------------------------------------

np.savez_compressed(
    acted_results_path
    / "acted_only_raw_predictions.npz",
    y_true=np.asarray(
        Y_test_acted_only,
        dtype=np.int64,
    ),
    y_pred=np.asarray(
        acted_predictions,
        dtype=np.int64,
    ),
    y_proba=np.asarray(
        acted_probabilities,
        dtype=np.float32,
    ),
    confusion_matrix=acted_cm_array,
    class_names=np.asarray(
        class_names,
        dtype=str,
    ),
)


print("Acted-only teacher results saved to:")
print(acted_results_path)

print("\nSaved files:")
for saved_file in sorted(
    acted_results_path.glob("acted_only_*")
):
    print(" -", saved_file.name)

In [ ]:
!pip install onnxruntime -q

In [ ]:
# ============================================================
# ONNX EXPORT VALIDATION — must run after cell 92
# Checks that sparseVR_teacher_model.onnx matches Keras output
# ============================================================
import onnxruntime as ort
import numpy as np
from sklearn.metrics import accuracy_score

onnx_path = "/content/sparseVR_teacher_model.onnx"  # adjust if needed

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name  = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Same data the Keras model used
X_input = X_test_acted_only.astype(np.float32)  # shape (N, 5, 70)
onnx_proba = session.run([output_name], {input_name: X_input})[0]
onnx_pred  = np.argmax(onnx_proba, axis=1)

keras_acc  = accuracy_score(Y_test_acted_only, acted_predictions)
onnx_acc   = accuracy_score(Y_test_acted_only, onnx_pred)
agreement  = np.mean(onnx_pred == acted_predictions)
max_prob_diff = np.abs(onnx_proba - acted_probabilities).max()

print(f"Keras accuracy : {keras_acc:.4f}")
print(f"ONNX  accuracy : {onnx_acc:.4f}")
print(f"Prediction agreement : {agreement:.4f}  (should be ~1.0)")
print(f"Max probability diff : {max_prob_diff:.6f}  (should be < 1e-4)")

if agreement < 0.99:
    mismatch = np.where(onnx_pred != acted_predictions)[0]
    print(f"\nMismatches ({len(mismatch)} samples):")
    for i in mismatch[:10]:
        print(f"  sample {i}: keras={acted_predictions[i]}  onnx={onnx_pred[i]}")


In [ ]:
from tensorflow.keras.callbacks import CSVLogger

# Build MoE
model, anneal_cb = build_moe_single_input(
    input_shape=(SEQ_LENGTH, input_size), num_classes=num_classes,
    tau_init=2.0, tau_final=0.7,
    gate_hidden=128, classifier_hidden=256,
    lb_coef=0.02, ent_coef=0.01
)

train_log_path = f"{models_path}/{TEACHER_TAG}_training_log.csv"
csv_logger = CSVLogger(train_log_path, append=False)

es_cb = callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)

# anneal = GateTemperatureScheduler(gate, start_tau=2.0, end_tau=0.7, anneal_epochs=20)

trained_model = train_then_test(
    model,
    window_size=SEQ_LENGTH,
    epochs=epochs,
    batch_size=batch_size,
    verbose=True,
    callbacks=[anneal_cb, es_cb, csv_logger]
    )

trained_model.save(f"{models_path}/{TEACHER_MODEL_NAME}")

print("Natural Test Data:")
print_scores(trained_model, X_test_nat_np, Y_test_nat_np)


print("Acted Test Data:")
print_scores(trained_model, X_test_act_comb_np, Y_test_act_comb_np)

print("Combined test data: ")
print_scores(trained_model, X_test_comb_np, Y_test_comb_np)


# # Callbacks: temperature annealing + early stopping (optional)
# anneal_cb = GateTemperatureScheduler(gate_layer, start_tau=2.0, end_tau=0.7, anneal_epochs=20)
# es_cb = callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)

# # Train on combined train set, validate on combined val
# history = model.fit(
#     x=[X_train_np, h_train_np],
#     y=y_train_np,
#     validation_data=([X_val_np, h_val_np], y_val_np),
#     epochs=50,
#     batch_size=32,
#     callbacks=[anneal_cb, es_cb],
#     verbose=1
# )


## 7.4 Result Export

In [ ]:
import json
from pathlib import Path


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    top_k_accuracy_score,
)
from sklearn.preprocessing import label_binarize


def compute_full_metrics(model, X, y_true, class_labels, class_names, model_name="model", test_name="test"):
    """
    Run a model on (X, y_true) and compute the full metric suite.
    Returns a dict with scalar metrics, per-class metrics, confusion matrices,
    and the raw predictions/probabilities (so plots can reuse them without re-predicting).
    """
    y_true = np.reshape(np.asarray(y_true), -1)
    y_proba = model.predict(X, batch_size=64, verbose=0)
    y_proba = np.asarray(y_proba)
    y_pred = np.argmax(y_proba, axis=1)

    n_classes = len(class_labels)

    # --- Basic ---
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # --- Precision/Recall/F1 (aggregate) ---
    prec_macro = precision_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)

    prec_weighted = precision_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)

    prec_micro = precision_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)
    rec_micro = recall_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)

    # --- Agreement metrics ---
    kappa = cohen_kappa_score(y_true, y_pred, labels=class_labels)
    mcc = matthews_corrcoef(y_true, y_pred)

    # --- Top-2 accuracy ---
    present_labels = np.unique(y_true)
    try:
        top2_acc = top_k_accuracy_score(y_true, y_proba, k=2, labels=class_labels)
    except Exception as e:
        top2_acc = np.nan
        print(f"[{model_name} | {test_name}] top-2 accuracy skipped: {e}")

    # --- Log loss ---
    try:
        logloss = log_loss(y_true, y_proba, labels=class_labels)
    except Exception as e:
        logloss = np.nan
        print(f"[{model_name} | {test_name}] log loss skipped: {e}")

    # --- ROC-AUC and PR-AUC (One-vs-Rest); only well-defined if >=2 classes present in y_true ---
    roc_auc_macro = roc_auc_score_safe = np.nan
    roc_auc_weighted = np.nan
    pr_auc_macro = np.nan
    pr_auc_weighted = np.nan
    if len(present_labels) >= 2:
        y_true_bin = label_binarize(y_true, classes=class_labels)
        try:
            roc_auc_macro = roc_auc_score(y_true_bin, y_proba, average="macro", multi_class="ovr")
            roc_auc_weighted = roc_auc_score(y_true_bin, y_proba, average="weighted", multi_class="ovr")
        except Exception as e:
            print(f"[{model_name} | {test_name}] ROC-AUC skipped: {e}")
        try:
            pr_auc_macro = average_precision_score(y_true_bin, y_proba, average="macro")
            pr_auc_weighted = average_precision_score(y_true_bin, y_proba, average="weighted")
        except Exception as e:
            print(f"[{model_name} | {test_name}] PR-AUC skipped: {e}")
    else:
        print(f"[{model_name} | {test_name}] Only one class present in y_true; ROC-AUC/PR-AUC undefined, skipped.")

    # --- Per-class report ---
    report_dict = classification_report(
        y_true, y_pred, labels=class_labels, target_names=class_names,
        output_dict=True, zero_division=0,
    )
    per_class_df = pd.DataFrame(report_dict).T.loc[class_names]
    per_class_df.index.name = "class"

    # --- Confusion matrices ---
    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    with np.errstate(all="ignore"):
        cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    scalar_metrics = {
        "model": model_name,
        "test_set": test_name,
        "n_samples": int(len(y_true)),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": prec_micro,
        "recall_micro": rec_micro,
        "f1_micro": f1_micro,
        "cohen_kappa": kappa,
        "matthews_corrcoef": mcc,
        "top2_accuracy": top2_acc,
        "log_loss": logloss,
        "roc_auc_macro_ovr": roc_auc_macro,
        "roc_auc_weighted_ovr": roc_auc_weighted,
        "pr_auc_macro_ovr": pr_auc_macro,
        "pr_auc_weighted_ovr": pr_auc_weighted,
    }

    return {
        "scalar_metrics": scalar_metrics,
        "per_class_df": per_class_df,
        "confusion_matrix": cm,
        "confusion_matrix_normalized": cm_norm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


print("compute_full_metrics() defined.")


### 7.4.1 Compact per-test-set results

`eval_and_pack` runs the trained model on one test set and packages accuracy, macro-F1, the confusion
matrix, and the full classification report. Raw predictions are written as `.npz` and the whole set
is summarised in `summary.json`.

In [ ]:
# ---- Save teacher_plus_10participants (70-AU) results: natural / acted / combined test sets ----
teacher10_dir = TEACHER_RUN_DIR / "results_eval_and_pack"
teacher10_dir.mkdir(parents=True, exist_ok=True)

def eval_and_pack(model, X_test, Y_test, name):
    Y_pred = model.predict(X_test)
    Y_pred = np.argmax(Y_pred, axis=1)
    Y_test_flat = np.reshape(Y_test, -1)

    acc = accuracy_score(Y_test_flat, Y_pred)
    f1_macro = f1_score(Y_test_flat, Y_pred, average="macro")
    cm = confusion_matrix(Y_test_flat, Y_pred)
    report = classification_report(
        Y_test_flat, Y_pred,
        labels=range(len(Emotion)),
        target_names=[emo.value for emo in Emotion],
        output_dict=True,
    )

    np.savez(
        teacher10_dir / f"raw_predictions_{name}.npz",
        y_true=Y_test_flat,
        y_pred=Y_pred,
        confusion_matrix=cm,
    )

    return {
        "test_set": name,
        "n_samples": int(len(Y_test_flat)),
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
    }

results_by_testset = {
    "natural": eval_and_pack(trained_model, X_test_nat_np, Y_test_nat_np, "natural"),
    "acted_plus_emojihero": eval_and_pack(trained_model, X_test_act_comb_np, Y_test_act_comb_np, "acted_plus_emojihero"),
    "combined": eval_and_pack(trained_model, X_test_comb_np, Y_test_comb_np, "combined"),
}

summary = {
    "model_name": TEACHER_MODEL_NAME,
    "input_shape": [SEQ_LENGTH, len(AU_NAMES)],
    "results_by_testset": results_by_testset,
}

with open(teacher10_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

print("Saved teacher_plus_new_participants (70-AU) results to:", teacher10_dir)
for name, r in results_by_testset.items():
    print(f"  {name}: accuracy={r['accuracy']:.4f}  f1_macro={r['f1_macro']:.4f}  n={r['n_samples']}")

## 7.5 Full Metric Suite Export

Re-evaluates the same trained model with `compute_full_metrics()` — the richer metric suite
(balanced accuracy, Cohen's kappa, MCC, top-2 accuracy, log loss, ROC-AUC, PR-AUC) — and saves scalar
metrics, raw predictions, a summary, and a copy of the model checkpoint.

**Note:** `compute_full_metrics()` is defined in Section 11.3. Run that cell before this one.

In [ ]:
from pathlib import Path

teacher_re_dir = TEACHER_RUN_DIR / "results_full_metrics"
teacher_re_dir.mkdir(parents=True, exist_ok=True)

class_labels = list(range(len(Emotion)))
class_names  = [emo.value for emo in Emotion]

re_results = {}
re_scalar_rows = []

for name, X_t, Y_t in [
    ("old_natural",            X_test_nat_np,      Y_test_nat_np),
    ("old_acted_plus_EmojiHero", X_test_act_comb_np, Y_test_act_comb_np),
    ("all_combined",           X_test_comb_np,     Y_test_comb_np),
]:
    result = compute_full_metrics(
        trained_model, X_t, Y_t,
        class_labels=class_labels,
        class_names=class_names,
        model_name=TEACHER_MODEL_NAME,
        test_name=name,
    )
    re_results[name] = result
    re_scalar_rows.append(result["scalar_metrics"])
    print(f"{name}: f1_macro={result['scalar_metrics']['f1_macro']:.4f}  accuracy={result['scalar_metrics']['accuracy']:.4f}")

scalar_df_re = pd.DataFrame(re_scalar_rows)
scalar_df_re.to_csv(teacher_re_dir / "scalar_metrics.csv", index=False)

raw_dir_re = teacher_re_dir / "raw_predictions"
raw_dir_re.mkdir(exist_ok=True)
for name, result in re_results.items():
    np.savez(
        raw_dir_re / f"removed_emotions__{name}.npz",
        y_true=result["y_true"],
        y_pred=result["y_pred"],
        y_proba=result["y_proba"],
        confusion_matrix=result["confusion_matrix"],
    )

summary_re = {
    "model_name": TEACHER_MODEL_NAME,
    "class_names": class_names,
    "results_by_testset": {
        name: result["scalar_metrics"] for name, result in re_results.items()
    },
}
with open(teacher_re_dir / "summary.json", "w") as f:
    json.dump(summary_re, f, indent=2, default=float)

# Part 8: Fine-Tuning Variant

An alternative to retraining from scratch: load the original teacher's weights and fine-tune them on
the new participants only, with a small learning rate. This is compared against joint retraining in
Part 9.

## 8.1 Original Teacher Checkpoint Path

In [ ]:
ORIGINAL_TEACHER_MODEL_PATH = Path("/content/drive/MyDrive/outputs_new_participants_70AU-20260715_1031/models/teacher-70AU-MoE_3experts_2gate-without_new_participants.keras")

In [ ]:
print(ORIGINAL_TEACHER_MODEL_PATH)

## 8.2 Load the Teacher by Rebuilding + Restoring Weights

Rebuilds the exact architecture from `build_moe_single_input()` and then loads only the weights.

This is the required workaround for the `Lambda` layer serialisation problem: calling
`keras.models.load_model()` on the saved file fails because the expert-stacking and fusion `Lambda`
layers cannot be reconstructed from their serialised form.

In [ ]:
def load_teacher_model(model_path, input_shape=(SEQ_LENGTH, input_size), num_classes=num_classes):
    # Rebuild the exact same architecture used at training time
    model, _ = build_moe_single_input(
        input_shape=input_shape, num_classes=num_classes,
        tau_init=2.0, tau_final=0.7,
        gate_hidden=128, classifier_hidden=256,
        lb_coef=0.02, ent_coef=0.01,
    )
    # Load only the weights, skip reconstructing the saved Functional graph
    model.load_weights(model_path)
    return model

### Instantiate the original teacher

In [ ]:
model = load_teacher_model(ORIGINAL_TEACHER_MODEL_PATH, input_shape=(SEQ_LENGTH, 70))

In [ ]:
print(model)
print(len(X_train_new))

## 8.3 Fine-Tuning Tensors

Stacks the new-participant windows only. These are the fine-tuning inputs.

In [ ]:
X_train_new_np = stack_sequences(X_train_new)
Y_train_new_np = to_numpy_labels(Y_train_new)

X_val_new_np = stack_sequences(X_val_new)
Y_val_new_np = to_numpy_labels(Y_val_new)

print("Fine-tune train shape:", X_train_new_np.shape)

## 8.4 Fine-Tune

Recompiles the loaded teacher with a deliberately small learning rate (SGD, 1e-4) so that the
pre-trained representation is nudged rather than overwritten, and trains for at most 15 epochs with
early stopping on validation loss (patience 3).

**Note:** the model is fine-tuned on the new participants only, but validated on the *combined*
validation set. Early stopping therefore selects the checkpoint that best preserves overall
performance rather than the one that best fits the new participants.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import tensorflow as tf
from tensorflow.keras import callbacks

# ---------------------------------------------------------
# 1. Reload the original teacher from disk.
# This guarantees that fine-tuning starts from the original
# teacher rather than an accidentally modified model.
# ---------------------------------------------------------
finetune_model = load_teacher_model(
    ORIGINAL_TEACHER_MODEL_PATH,
    input_shape=(SEQ_LENGTH, 70),
    num_classes=num_classes,
)

# ---------------------------------------------------------
# 2. Prepare the new-participant tensors.
# ---------------------------------------------------------
X_train_new_np = stack_sequences(X_train_new)
Y_train_new_np = to_numpy_labels(Y_train_new)

X_val_new_np = stack_sequences(X_val_new)
Y_val_new_np = to_numpy_labels(Y_val_new)

print("Fine-tune train shape:", X_train_new_np.shape)
print("Fine-tune validation shape:", X_val_new_np.shape)
print("Classes in new train:", np.unique(Y_train_new_np))
print("Classes in new validation:", np.unique(Y_val_new_np))

assert X_train_new_np.shape[1:] == (SEQ_LENGTH, 70)
assert X_val_new_np.shape[1:] == (SEQ_LENGTH, 70)
assert len(X_train_new_np) == len(Y_train_new_np)
assert len(X_val_new_np) == len(Y_val_new_np)

# ---------------------------------------------------------
# 3. Calculate class weights from fine-tuning training data.
# ---------------------------------------------------------
present_classes = np.unique(Y_train_new_np)

ft_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=Y_train_new_np,
)

ft_class_weight_dict = {
    int(class_id): float(weight)
    for class_id, weight in zip(present_classes, ft_class_weights)
}

print("Fine-tune class weights:", ft_class_weight_dict)

# ---------------------------------------------------------
# 4. Compile with a small learning rate.
# ---------------------------------------------------------
finetune_model.compile(
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=1e-4,
        momentum=0.9,
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# ---------------------------------------------------------
# 5. Stop according to validation macro F1 if available.
# Otherwise val_loss is usually safer than val_accuracy for
# imbalanced emotion classes.
# ---------------------------------------------------------
es_cb_finetune = callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True,
)

#automatically makes the learning rate smaller when validation performance stops improving.
reduce_lr_cb = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

# ---------------------------------------------------------
# 6. Fine-tune on new-participant training data.
# ---------------------------------------------------------
history_finetune = finetune_model.fit(
    X_train_new_np,
    Y_train_new_np,
    validation_data=(X_val_new_np, Y_val_new_np),
    epochs=30,
    batch_size=batch_size,
    class_weight=ft_class_weight_dict,
    callbacks=[
        es_cb_finetune,
        reduce_lr_cb,
    ],
    verbose=True,
)

## 8.5 Evaluate and Save the Fine-Tuned Teacher

Same metric suite and output convention as Section 7.5, written to its own directory.

In [ ]:
finetuned_dir = Path(predictions_path) / "teacher_70AU_finetuned_on_new_participants"
finetuned_dir.mkdir(parents=True, exist_ok=True)

class_labels = list(range(len(Emotion)))
class_names = [emo.value for emo in Emotion]

finetuned_results = {}
finetuned_scalar_rows = []

for name, X_t, Y_t in [
    ("old_natural", X_test_nat_np, Y_test_nat_np),
    ("old_acted_plus_EmojiHero", X_test_act_comb_np, Y_test_act_comb_np),
    ("all_combined", X_test_comb_np, Y_test_comb_np),
]:
    result = compute_full_metrics(
        finetune_model, X_t, Y_t,
        class_labels=class_labels,
        class_names=class_names,
        model_name="teacher_finetuned_on_10p",
        test_name=name,
    )
    finetuned_results[name] = result
    finetuned_scalar_rows.append(result["scalar_metrics"])
    print(f"{name}: f1_macro={result['scalar_metrics']['f1_macro']:.4f}  "
          f"accuracy={result['scalar_metrics']['accuracy']:.4f}")

# Save scalar metrics in the same wide format as teacher_model_comparison
scalar_df_finetuned = pd.DataFrame(finetuned_scalar_rows)
scalar_df_finetuned.to_csv(finetuned_dir / "scalar_metrics_finetuned.csv", index=False)

# Save raw predictions/probabilities per test set (same convention as teacher_model_comparison)
raw_dir_finetuned = finetuned_dir / "raw_predictions"
raw_dir_finetuned.mkdir(exist_ok=True)
for name, result in finetuned_results.items():
    np.savez(
        raw_dir_finetuned / f"teacher_finetuned_on_10p__{name}.npz",
        y_true=result["y_true"],
        y_pred=result["y_pred"],
        y_proba=result["y_proba"],
        confusion_matrix=result["confusion_matrix"],
    )

# Save full summary.json (scalar metrics only, easy to diff/reload)
summary_finetuned = {
    "model_name": "teacher_finetuned_on_10p",
    "class_names": class_names,
    "results_by_testset": {
        name: result["scalar_metrics"] for name, result in finetuned_results.items()
    },
}
with open(finetuned_dir / "summary.json", "w") as f:
    json.dump(summary_finetuned, f, indent=2, default=float)

finetune_model.save(finetuned_dir / "teacher-70AU-finetuned-on-new_participants.keras")
print("Saved:", finetuned_dir)
print(" - scalar_metrics_finetuned.csv")
print(" - raw_predictions/*.npz")
print(" - summary.json")
print(" - teacher-70AU-finetuned-on-10participants.keras")

### Fine-tuned performance on the new-participant test set

The most direct measure of whether fine-tuning actually transferred to the new participants.

In [ ]:
X_test_new_np = stack_sequences(X_test_new)
Y_test_new_np = to_numpy_labels(Y_test_new)

Y_pred_new = np.argmax(finetune_model.predict(X_test_new_np), axis=1)
f1m_new = f1_score(Y_test_new_np, Y_pred_new, average="macro")
acc_new = accuracy_score(Y_test_new_np, Y_pred_new)
print(f"new_10participants: f1_macro={f1m_new:.4f}  accuracy={acc_new:.4f}")

## 9.1 Three-Way Comparison: Original vs. Fine-Tuned vs. Jointly Retrained

In [ ]:
import json

### 9.1.1 Assemble the comparison table

Loads the saved `summary.json` files for each teacher variant. The pipelines used different test-set
names over time (`natural` vs `old_natural`, `combined` vs `all_combined`), so `TEST_SET_ALIASES`
maps them onto one canonical set of names before anything is compared.

In [ ]:
import json
import pandas as pd

# Canonical test_set name mapping (different pipelines used different naming)
TEST_SET_ALIASES = {
    "natural": "natural",
    "old_natural": "natural",
    "acted_plus_emojihero": "acted_plus_emojihero",
    "old_acted_plus_EmojiHero": "acted_plus_emojihero",
    "combined": "combined",
    "all_combined": "combined",
}

rows = []

def add_rows_from_results_by_testset(model_name, summary):
    for test_name, metrics in summary["results_by_testset"].items():
        canon = TEST_SET_ALIASES.get(test_name, test_name)
        row = {"model": model_name, "test_set": canon}
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                row[k] = v
        rows.append(row)

def add_rows_from_scalar_metrics_list(summary):
    for entry in summary["scalar_metrics"]:
        canon = TEST_SET_ALIASES.get(entry["test_set"], entry["test_set"])
        row = {"model": entry["model"], "test_set": canon}
        for k, v in entry.items():
            if k in ("model", "test_set"):
                continue
            if isinstance(v, (int, float)):
                row[k] = v
        rows.append(row)

# original_teacher and teacher_plus_10participants both come from the rich,
# up-to-date comparison file (correct test-set sizes, full metric suite).
rich_path = Path(predictions_path) / "teacher_model_comparison" / "summary.json"
if rich_path.exists():
    with open(rich_path) as f:
        rich_summary = json.load(f)
    add_rows_from_scalar_metrics_list(rich_summary)
else:
    print(f"WARNING: {rich_path} not found.")

# Fine-tuned model: separate summary.json using results_by_testset schema
finetuned_path = Path(predictions_path) / "teacher_70AU_finetuned_on_10participants" / "summary.json"
if finetuned_path.exists():
    with open(finetuned_path) as f:
        finetuned_summary = json.load(f)
    add_rows_from_results_by_testset("teacher_finetuned_on_10p", finetuned_summary)
else:
    print(f"WARNING: {finetuned_path} not found.")

comparison_df = pd.DataFrame(rows)
print("Models found:", sorted(comparison_df["model"].unique()))
print("Test sets found:", sorted(comparison_df["test_set"].unique()))
print("Available metric columns:", [c for c in comparison_df.columns if c not in ("model", "test_set")])
print()

comparison_out_dir = Path(predictions_path) / "three_way_teacher_comparison"
comparison_out_dir.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(comparison_out_dir / "comparison_long_all_metrics.csv", index=False)

metric_cols = [c for c in comparison_df.columns if c not in ("model", "test_set", "n_samples")]
for metric in metric_cols:
    pivot = comparison_df.pivot(index="test_set", columns="model", values=metric)
    print(f"=== {metric} ===")
    print(pivot.round(4))
    print()
    pivot.to_csv(comparison_out_dir / f"{metric}_pivot.csv")

print("Saved all comparison files to:", comparison_out_dir)

### 9.1.2 Metric-by-metric comparison with win counts

Pivots every metric across models and test sets, marks the best model per row, and tallies wins.
`log_loss` is the only metric where lower is better.

In [ ]:
def print_three_way_comparison(comparison_df, models_order=None, lower_is_better=("log_loss",)):
    """
    Pretty-print a model x test_set x metric comparison with a winner annotation per row.
    """
    if models_order is None:
        models_order = sorted(comparison_df["model"].unique())

    metric_cols = [c for c in comparison_df.columns if c not in ("model", "test_set", "n_samples")]
    test_sets = sorted(comparison_df["test_set"].unique())

    win_counts = {m: 0 for m in models_order}

    for metric in metric_cols:
        print("=" * 100)
        print(f"METRIC: {metric}")
        print("=" * 100)
        pivot = comparison_df.pivot(index="test_set", columns="model", values=metric)[models_order]

        for test_set in test_sets:
            row = pivot.loc[test_set]
            higher_better = metric not in lower_is_better
            best_model = row.idxmax() if higher_better else row.idxmin()
            win_counts[best_model] += 1

            vals_str = "  ".join(f"{m}={row[m]:.4f}" for m in models_order)
            print(f"  [{test_set:22s}] {vals_str}   -> best: {best_model}")
        print()

    print("=" * 100)
    print("OVERALL WIN COUNT (across all metric x test_set combinations)")
    print("=" * 100)
    total = sum(win_counts.values())
    for m in sorted(win_counts, key=win_counts.get, reverse=True):
        pct = 100 * win_counts[m] / total
        print(f"  {m:35s}: {win_counts[m]:3d} wins  ({pct:.1f}%)")
    print()

    overall_winner = max(win_counts, key=win_counts.get)
    print(f"VERDICT: '{overall_winner}' wins the most metric/test_set combinations overall.")
    return win_counts


win_counts = print_three_way_comparison(
    comparison_df,
    models_order=["original_teacher", "teacher_finetuned_on_10p", "teacher_plus_10participants"],
)

### 9.1.3 Per-class F1 comparison

Recomputes per-class metrics from the saved raw predictions rather than re-running the models, so
the numbers are exactly the ones that were reported.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

class_names = [emo.value for emo in Emotion]
class_labels = list(range(len(Emotion)))

NPZ_PATHS = {
    "original_teacher": Path(predictions_path) / "teacher_model_comparison" / "raw_predictions",
    "teacher_plus_10participants": Path(predictions_path) / "teacher_model_comparison" / "raw_predictions",
    "teacher_finetuned_on_10p": Path(predictions_path) / "teacher_70AU_finetuned_on_10participants" / "raw_predictions",
}

TEST_NAME_FOR_MODEL = {
    "original_teacher": {
        "natural": "original_teacher__old_natural.npz",
        "acted_plus_emojihero": "original_teacher__old_acted_plus_EmojiHero.npz",
        "combined": "original_teacher__all_combined.npz",
    },
    "teacher_plus_10participants": {
        "natural": "teacher_plus_10participants__old_natural.npz",
        "acted_plus_emojihero": "teacher_plus_10participants__old_acted_plus_EmojiHero.npz",
        "combined": "teacher_plus_10participants__all_combined.npz",
    },
    "teacher_finetuned_on_10p": {
        "natural": "teacher_finetuned_on_10p__old_natural.npz",
        "acted_plus_emojihero": "teacher_finetuned_on_10p__old_acted_plus_EmojiHero.npz",
        "combined": "teacher_finetuned_on_10p__all_combined.npz",
    },
}

per_class_rows = []

for model_name, file_map in TEST_NAME_FOR_MODEL.items():
    base_dir = NPZ_PATHS[model_name]
    for test_set, fname in file_map.items():
        fpath = base_dir / fname
        if not fpath.exists():
            print(f"WARNING: {fpath} not found, skipping.")
            continue
        data = np.load(fpath)
        y_true = data["y_true"]
        y_pred = data["y_pred"]

        report = classification_report(
            y_true, y_pred,
            labels=class_labels, target_names=class_names,
            output_dict=True, zero_division=0,
        )
        for cls in class_names:
            per_class_rows.append({
                "model": model_name,
                "test_set": test_set,
                "class": cls,
                "precision": report[cls]["precision"],
                "recall": report[cls]["recall"],
                "f1": report[cls]["f1-score"],
                "support": report[cls]["support"],
            })

per_class_df = pd.DataFrame(per_class_rows)

per_class_out_dir = Path(predictions_path) / "three_way_teacher_comparison"
per_class_out_dir.mkdir(parents=True, exist_ok=True)
per_class_df.to_csv(per_class_out_dir / "per_class_f1_comparison.csv", index=False)

# Per-class F1 pivot, one table per test_set
for test_set in per_class_df["test_set"].unique():
    print("=" * 90)
    print(f"PER-CLASS F1 — test_set: {test_set}")
    print("=" * 90)
    sub = per_class_df[per_class_df["test_set"] == test_set]
    pivot = sub.pivot(index="class", columns="model", values="f1")[
        ["original_teacher", "teacher_finetuned_on_10p", "teacher_plus_10participants"]
    ]
    pivot = pivot.loc[class_names]  # keep emotion order consistent
    print(pivot.round(4))
    print()

print("Saved:", per_class_out_dir / "per_class_f1_comparison.csv")

### 9.1.4 Focused comparison: macro-F1, accuracy, weighted F1

Restricts the comparison to the three headline metrics and prints the original -> fine-tuned delta.

Macro-F1 is treated as the primary metric throughout: accuracy and weighted F1 are sample-weighted
and are inflated by the majority classes, which is misleading under this level of class imbalance.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Load comparison data
with open(Path(predictions_path) / "teacher_model_comparison" / "summary.json") as f:
    rich_summary = json.load(f)

with open(Path(predictions_path) / "teacher_70AU_finetuned_on_10participants-7train_1val" / "summary.json") as f:
    ft_summary = json.load(f)

TEST_SET_ALIASES = {
    "old_natural": "natural",
    "old_acted_plus_EmojiHero": "acted_plus_emojihero",
    "all_combined": "combined",
    "natural": "natural",
    "acted_plus_emojihero": "acted_plus_emojihero",
    "combined": "combined",
}

# Build unified dict: metrics[metric][test_set][model] = value
rows = []

for entry in rich_summary["scalar_metrics"]:
    ts = TEST_SET_ALIASES.get(entry["test_set"], entry["test_set"])
    model = entry["model"]  # "original_teacher" or "teacher_plus_10participants"
    for k, v in entry.items():
        if isinstance(v, (int, float)) and k not in ("n_samples",):
            rows.append({"model": model, "test_set": ts, "metric": k, "value": v})

for ts_raw, metrics in ft_summary["results_by_testset"].items():
    ts = TEST_SET_ALIASES.get(ts_raw, ts_raw)
    for k, v in metrics.items():
        if isinstance(v, (int, float)) and k not in ("n_samples",):
            rows.append({"model": "teacher_finetuned_on_10p", "test_set": ts, "metric": k, "value": v})

df = pd.DataFrame(rows)

# Focus metrics and model order
focus_metrics = ["f1_macro", "accuracy", "f1_weighted"]
model_order   = ["original_teacher", "teacher_finetuned_on_10p", "teacher_plus_10participants"]
model_short   = {"original_teacher": "OT", "teacher_finetuned_on_10p": "FT", "teacher_plus_10participants": "P10"}
test_sets     = ["natural", "acted_plus_emojihero", "combined"]

win_counts = {m: 0 for m in model_order}
total_comparisons = 0

for metric in focus_metrics:
    print("=" * 78)
    print(f"METRIC: {metric}")
    print("=" * 78)
    print(f"  {'test_set':<25} {'OT':>7} {'FT':>7} {'P10':>7}  {'best':>5}  {'OT→FT':>8}")
    print("-" * 78)

    for ts in test_sets:
        vals = {}
        for m in model_order:
            row = df[(df["model"] == m) & (df["test_set"] == ts) & (df["metric"] == metric)]
            vals[m] = row["value"].values[0] if len(row) > 0 else float("nan")

        best_model = max(model_order, key=lambda m: vals[m])
        win_counts[best_model] += 1
        total_comparisons += 1

        ot_ft_diff = vals["teacher_finetuned_on_10p"] - vals["original_teacher"]
        best_short = model_short[best_model]

        print(f"  {ts:<25} "
              f"{vals['original_teacher']:>7.4f} "
              f"{vals['teacher_finetuned_on_10p']:>7.4f} "
              f"{vals['teacher_plus_10participants']:>7.4f}  "
              f"{best_short:>5}  "
              f"{ot_ft_diff:>+8.4f}")
    print()

print("=" * 78)
print(f"WIN COUNT ({total_comparisons} comparisons: {len(focus_metrics)} metrics x {len(test_sets)} test sets)")
print("=" * 78)
for m in sorted(model_order, key=lambda x: win_counts[x], reverse=True):
    pct = 100 * win_counts[m] / total_comparisons
    print(f"  {model_short[m]} ({m:<35}): {win_counts[m]:2d} wins ({pct:.0f}%)")

print()
print("=" * 78)
print("VERDICT")
print("=" * 78)
overall_best = max(model_order, key=lambda m: win_counts[m])
print(f"  Overall best: {overall_best} ({model_short[overall_best]})")
print()
print("  Note: OT=original_teacher, FT=teacher_finetuned_on_10p, P10=teacher_plus_10participants")
print("  f1_macro   = treats all 7 emotions equally (recommended primary metric)")
print("  accuracy   = sample-weighted, favors majority classes (Happiness/Disgust)")
print("  f1_weighted= sample-weighted F1, also biased toward majority classes")
print("  -> f1_macro is the most honest metric for imbalanced emotion recognition.")

# Part 9: Teacher Variant Comparisons

## 9.1 Five-Way Comparison

Extends the comparison to every teacher variant that was trained:

| Key | Variant |
|---|---|
| OT | Original teacher (no new data) |
| P10 | Original + 10 participants, naive joint training |
| FT | Original + 10 participants, fine-tuned |
| RE | Original + 10 participants, self-report filtered |
| RE70T10V20T | Original + 10 participants, self-report filtered, 70/10/20 split |

### 9.1. Configuration and loading

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report

MODEL_ORDER = [
    "original_teacher",
    "plus_10participants",
    "finetuned_on_10p",
    "removed_emotions",
    "removed_emotions_70train_10val_20test_ratio",
]

MODEL_SHORT = {
    "original_teacher":   "OT",
    "plus_10participants": "P10",
    "finetuned_on_10p":   "FT",
    "removed_emotions":   "RE",
    "removed_emotions_70train_10val_20test_ratio": "RE70T10V20T",
}

MODEL_LABELS = {
    "original_teacher":   "Original Teacher (no new data)",
    "plus_10participants": "Original + 10p naive joint training",
    "finetuned_on_10p":   "Original + 10p fine-tuned",
    "removed_emotions":   "Original + 10p self-report filtered",
    "removed_emotions_70train_10val_20test_ratio": "Original + 10p self-report filtered (70:10:20)",
}

TEST_SET_ALIASES = {
    "old_natural":              "natural",
    "old_acted_plus_EmojiHero": "acted_plus_emojihero",
    "all_combined":             "combined",
    "natural":                  "natural",
    "acted_plus_emojihero":     "acted_plus_emojihero",
    "combined":                 "combined",
}

LOWER_IS_BETTER = {"log_loss"}
TEST_SETS     = ["natural", "acted_plus_emojihero", "combined"]
EMOTIONS      = ['Anger','Disgust','Fear','Happiness','Neutral','Sadness','Surprise']
FOCUS_METRICS = [
    "f1_macro", "accuracy", "f1_weighted", "balanced_accuracy",
    "roc_auc_macro_ovr", "log_loss", "matthews_corrcoef", "top2_accuracy",
]

PRED = Path(predictions_path)

rows = []

def add_from_scalar_metrics_list(data):
    for entry in data["scalar_metrics"]:
        ts = TEST_SET_ALIASES.get(entry["test_set"], entry["test_set"])
        m  = "plus_10participants" if entry["model"] == "teacher_plus_10participants" else entry["model"]
        row = {"model": m, "test_set": ts}
        for k, v in entry.items():
            if isinstance(v, (int, float)) and k != "n_samples":
                row[k] = v
        rows.append(row)

def add_from_results_by_testset(data, model_key):
    for ts_raw, metrics in data["results_by_testset"].items():
        ts = TEST_SET_ALIASES.get(ts_raw, ts_raw)
        row = {"model": model_key, "test_set": ts}
        for k, v in metrics.items():
            if isinstance(v, (int, float)) and k != "n_samples":
                row[k] = v
        rows.append(row)

# original_teacher + plus_10participants
p = Path("/content/drive/MyDrive/outputs_10participants_70AU/predictions/teacher_model_comparison/summary.json")
if p.exists():
    with open(p) as f:
        add_from_scalar_metrics_list(json.load(f))
else:
    print(f"WARNING: not found: {p}")

# finetuned_on_10p
p = Path("/content/drive/MyDrive/outputs_10participants_70AU/predictions/teacher_70AU_finetuned_on_10participants/summary.json")
if p.exists():
    with open(p) as f:
        add_from_results_by_testset(json.load(f), "finetuned_on_10p")
else:
    print(f"WARNING: not found: {p}")

p = Path("/content/drive/MyDrive/outputs_10participants_70AU-removed_emotions/predictions/teacher_with_10participants_70AU-removed_emotions-7train_1val/results_all_combined/summary.json")
if p.exists():
    with open(p) as f:
        add_from_results_by_testset(json.load(f), "removed_emotions")
else:
    print(f"WARNING: not found: {p}")

p = Path("/content/drive/MyDrive/outputs_10participants_70AU-removed_emotions/predictions/teacher_with_10participants_70AU-removed_emotions-5train_1val_2test/results_all_combined/summary.json")
if p.exists():
    with open(p) as f:
        add_from_results_by_testset(json.load(f), "removed_emotions_70train_10val_20test_ratio")
else:
    print(f"WARNING: not found: {p}")


compare_df = pd.DataFrame(rows)
print("Models loaded:", sorted(compare_df["model"].unique()))
print("Test sets:    ", sorted(compare_df["test_set"].unique()))
print("Metrics:      ", [c for c in compare_df.columns if c not in ("model", "test_set")])

## 9.2 Scalar metric comparison and win counts

In [ ]:
win_counts = {m: 0 for m in MODEL_ORDER}
total_comparisons = 0

print("=" * 115)
print("TEACHER MODELS COMPARISON — SCALAR METRICS")
print("=" * 115)
for m in MODEL_ORDER:
    print(f"  {MODEL_SHORT[m]:<6} = {MODEL_LABELS[m]}")
print("=" * 115)

for metric in FOCUS_METRICS:
    if metric not in compare_df.columns:
        continue
    arrow = " ↓ lower=better" if metric in LOWER_IS_BETTER else ""
    print(f"\n── {metric}{arrow}")
    print(f"  {'test_set':<25} " + "  ".join(f"{MODEL_SHORT[m]:>8}" for m in MODEL_ORDER) + "   best")
    print(f"  {'-'*25} " + "  ".join("--------" for _ in MODEL_ORDER))

    for ts in TEST_SETS:
        vals = {}
        for m in MODEL_ORDER:
            row = compare_df[(compare_df["model"] == m) & (compare_df["test_set"] == ts)]
            vals[m] = float(row[metric].values[0]) if len(row) > 0 and not row[metric].isna().all() else float("nan")

        valid = {m: v for m, v in vals.items() if not np.isnan(v)}
        best  = min(valid, key=valid.get) if metric in LOWER_IS_BETTER else max(valid, key=valid.get)
        win_counts[best] += 1
        total_comparisons += 1

        val_str = "  ".join(f"{vals[m]:>8.4f}" if not np.isnan(vals[m]) else f"{'N/A':>8}" for m in MODEL_ORDER)
        print(f"  {ts:<25} {val_str}   {MODEL_SHORT[best]}")

print()
print("=" * 115)
print(f"WIN COUNT ({total_comparisons} comparisons: {len(FOCUS_METRICS)} metrics × {len(TEST_SETS)} test sets)")
print("=" * 115)
for m in sorted(MODEL_ORDER, key=lambda x: win_counts[x], reverse=True):
    pct = 100 * win_counts[m] / total_comparisons if total_comparisons > 0 else 0
    bar = "█" * win_counts[m]
    print(f"  {MODEL_SHORT[m]:<6} ({MODEL_LABELS[m]:<45}): {win_counts[m]:3d} wins ({pct:.0f}%)  {bar}")

best_overall = max(MODEL_ORDER, key=lambda m: win_counts[m])
print(f"\n  VERDICT: {MODEL_SHORT[best_overall]} — {MODEL_LABELS[best_overall]}")

In [ ]:
out_dir = Path(predictions_path) / "five_way_teacher_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

compare_df.to_csv(out_dir / "scalar_metrics_all.csv", index=False)
print("Saved to:", out_dir)
print("  - scalar_metrics_all.csv")

## 9.3 Per-class F1 across all five variants

Rebuilds per-class F1 from each variant's saved raw predictions and prints one table per test set,
with the best variant marked per emotion.

In [ ]:
RAW_NPZ = {
    "original_teacher":   Path("/content/drive/MyDrive/outputs_10participants_70AU/predictions") / "teacher_model_comparison" / "raw_predictions",
    "plus_10participants": Path("/content/drive/MyDrive/outputs_10participants_70AU/predictions") / "teacher_model_comparison" / "raw_predictions",
    "finetuned_on_10p":   Path("/content/drive/MyDrive/outputs_10participants_70AU/predictions") / "teacher_70AU_finetuned_on_10participants" / "raw_predictions",
    "removed_emotions":   Path("/content/drive/MyDrive/outputs_10participants_70AU-removed_emotions/predictions/teacher_with_10participants_70AU-removed_emotions-7train_1val/results_all_combined/raw_predictions"),
    "removed_emotions_70train_10val_20test_ratio": Path("/content/drive/MyDrive/outputs_10participants_70AU-removed_emotions/predictions/teacher_with_10participants_70AU-removed_emotions-5train_1val_2test/results_all_combined/raw_predictions")
}

NPZ_FILES = {
    "original_teacher":   {
        "natural":              "original_teacher__old_natural.npz",
        "acted_plus_emojihero": "original_teacher__old_acted_plus_EmojiHero.npz",
        "combined":             "original_teacher__all_combined.npz",
    },
    "plus_10participants": {
        "natural":              "teacher_plus_10participants__old_natural.npz",
        "acted_plus_emojihero": "teacher_plus_10participants__old_acted_plus_EmojiHero.npz",
        "combined":             "teacher_plus_10participants__all_combined.npz",
    },
    "finetuned_on_10p": {
        "natural":              "teacher_finetuned_on_10p__old_natural.npz",
        "acted_plus_emojihero": "teacher_finetuned_on_10p__old_acted_plus_EmojiHero.npz",
        "combined":             "teacher_finetuned_on_10p__all_combined.npz",
    },
    "removed_emotions": {
        "natural":              "removed_emotions__old_natural.npz",
        "acted_plus_emojihero": "removed_emotions__old_acted_plus_EmojiHero.npz",
        "combined":             "removed_emotions__all_combined.npz",
    },
    "removed_emotions_70train_10val_20test_ratio": {
        "natural":              "removed_emotions__old_natural.npz",
        "acted_plus_emojihero": "removed_emotions__old_acted_plus_EmojiHero.npz",
        "combined":             "removed_emotions__all_combined.npz",
    },
}

pc_rows = []
for model_key, file_map in NPZ_FILES.items():
    base = RAW_NPZ[model_key]
    for ts, fname in file_map.items():
        fpath = base / fname
        if not fpath.exists():
            print(f"WARNING: {fpath} not found, skipping.")
            continue
        data = np.load(fpath)
        report = classification_report(
            data["y_true"], data["y_pred"],
            labels=list(range(len(EMOTIONS))),
            target_names=EMOTIONS,
            output_dict=True, zero_division=0,
        )
        for cls in EMOTIONS:
            pc_rows.append({
                "model": model_key, "test_set": ts,
                "class": cls, "f1": report[cls]["f1-score"],
            })

pc_df = pd.DataFrame(pc_rows)

for ts in TEST_SETS:
    print("=" * 95)
    print(f"PER-CLASS F1 — {ts.upper()}")
    print("=" * 95)
    sub = pc_df[pc_df["test_set"] == ts]
    if sub.empty:
        print("  No data.")
        continue
    avail = [m for m in MODEL_ORDER if m in sub["model"].unique()]
    pivot = sub.pivot(index="class", columns="model", values="f1")[avail]
    pivot = pivot.reindex(EMOTIONS)
    pivot.columns = [MODEL_SHORT[m] for m in avail]
    pivot["best"] = pivot.idxmax(axis=1)
    print(pivot.round(4).to_string())
    print()

## 9.4 Export the scalar comparison report

In [ ]:
# Save the printed comparison output as a text file
out_dir = Path(predictions_path) / "five_way_teacher_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

LOWER_IS_BETTER = {"log_loss"}

lines = []
lines.append("=" * 115)
lines.append("FOUR-WAY TEACHER MODEL COMPARISON — SCALAR METRICS")
lines.append("=" * 115)
for m in MODEL_ORDER:
    lines.append(f"  {MODEL_SHORT[m]:<6} = {MODEL_LABELS[m]}")
lines.append("=" * 115)

win_counts_save = {m: 0 for m in MODEL_ORDER}
total_save = 0

for metric in FOCUS_METRICS:
    if metric not in compare_df.columns:
        continue
    arrow = " ↓ lower=better" if metric in LOWER_IS_BETTER else ""
    lines.append(f"\n── {metric}{arrow}")
    lines.append(f"  {'test_set':<25} " + "  ".join(f"{MODEL_SHORT[m]:>8}" for m in MODEL_ORDER) + "   best")
    lines.append(f"  {'-'*25} " + "  ".join("--------" for _ in MODEL_ORDER))

    for ts in TEST_SETS:
        vals = {}
        for m in MODEL_ORDER:
            row = compare_df[(compare_df["model"] == m) & (compare_df["test_set"] == ts)]
            vals[m] = float(row[metric].values[0]) if len(row) > 0 and not row[metric].isna().all() else float("nan")

        valid = {m: v for m, v in vals.items() if not np.isnan(v)}
        best  = min(valid, key=valid.get) if metric in LOWER_IS_BETTER else max(valid, key=valid.get)
        win_counts_save[best] += 1
        total_save += 1

        val_str = "  ".join(f"{vals[m]:>8.4f}" if not np.isnan(vals[m]) else f"{'N/A':>8}" for m in MODEL_ORDER)
        lines.append(f"  {ts:<25} {val_str}   {MODEL_SHORT[best]}")

lines.append("")
lines.append("=" * 115)
lines.append(f"WIN COUNT ({total_save} comparisons: {len(FOCUS_METRICS)} metrics x {len(TEST_SETS)} test sets)")
lines.append("=" * 115)
for m in sorted(MODEL_ORDER, key=lambda x: win_counts_save[x], reverse=True):
    pct = 100 * win_counts_save[m] / total_save if total_save > 0 else 0
    bar = "█" * win_counts_save[m]
    lines.append(f"  {MODEL_SHORT[m]:<6} ({MODEL_LABELS[m]:<45}): {win_counts_save[m]:3d} wins ({pct:.0f}%)  {bar}")

best_overall = max(MODEL_ORDER, key=lambda m: win_counts_save[m])
lines.append(f"\n  VERDICT: {MODEL_SHORT[best_overall]} — {MODEL_LABELS[best_overall]}")

report_text = "\n".join(lines)
print(report_text)

with open(out_dir / "scalar_metrics_report.txt", "w") as f:
    f.write(report_text)

compare_df.to_csv(out_dir / "scalar_metrics_all.csv", index=False)

print(f"\nSaved to: {out_dir}")
print("  - scalar_metrics_report.txt")
print("  - scalar_metrics_all.csv")

## 9.5 Export the per-class F1 report

In [ ]:
pc_rows = []
for model_key, file_map in NPZ_FILES.items():
    base = RAW_NPZ[model_key]
    for ts, fname in file_map.items():
        fpath = base / fname
        if not fpath.exists():
            print(f"WARNING: {fpath} not found, skipping.")
            continue
        data = np.load(fpath)
        report = classification_report(
            data["y_true"], data["y_pred"],
            labels=list(range(len(EMOTIONS))),
            target_names=EMOTIONS,
            output_dict=True, zero_division=0,
        )
        for cls in EMOTIONS:
            pc_rows.append({
                "model": model_key, "test_set": ts,
                "class": cls, "f1": report[cls]["f1-score"],
            })

pc_df = pd.DataFrame(pc_rows)

lines_pc = []
for ts in TEST_SETS:
    lines_pc.append("=" * 95)
    lines_pc.append(f"PER-CLASS F1 — {ts.upper()}")
    lines_pc.append("=" * 95)
    sub = pc_df[pc_df["test_set"] == ts]
    if sub.empty:
        lines_pc.append("  No data.")
        continue
    avail = [m for m in MODEL_ORDER if m in sub["model"].unique()]
    pivot = sub.pivot(index="class", columns="model", values="f1")[avail]
    pivot = pivot.reindex(EMOTIONS)
    pivot.columns = [MODEL_SHORT[m] for m in avail]
    pivot["best"] = pivot.idxmax(axis=1)
    lines_pc.append(pivot.round(4).to_string())
    lines_pc.append("")

pc_report = "\n".join(lines_pc)
print(pc_report)

pc_df.to_csv(out_dir / "per_class_f1_all.csv", index=False)
with open(out_dir / "per_class_f1_report.txt", "w") as f:
    f.write(pc_report)

print(f"Saved to: {out_dir}")
print("  - per_class_f1_all.csv")
print("  - per_class_f1_report.txt")

# Part 10: Teacher Soft Labels for Distillation

This is the hand-off point to the student notebook. The trained teacher predicts a full
class-probability vector for every train / validation / test window; those vectors are the soft
targets used by the RQ2 distillation objectives.

Soft labels and hard labels are saved together so that student and teacher stay index-aligned.

## 10.1 Generate Teacher Soft Labels

Runs the trained teacher over the train, validation, and test windows and keeps the full softmax
output — not the argmax. The relative probability mass across the non-predicted classes is precisely
the "dark knowledge" the student is meant to inherit.

**Note:** this requires `trained_model` to still be in memory from Section 7.3.

In [ ]:
teacher_soft_train = trained_model.predict(X_train_np, batch_size=64)
teacher_soft_val = trained_model.predict(X_val_np, batch_size=64)
teacher_soft_test = trained_model.predict(X_test_comb_np, batch_size=64)

## 10.2 Save Soft and Hard Labels

Soft labels and their matching hard labels are saved together. The student notebook loads these
`.npy` files directly; index alignment between the two notebooks depends on both building windows
with the same threshold rule, the same seed, and the same participant split.

In [ ]:
np.save(f"{predictions_path}/teacher_soft_train.npy", teacher_soft_train)
np.save(f"{predictions_path}/teacher_soft_val.npy", teacher_soft_val)
np.save(f"{predictions_path}/teacher_soft_test.npy", teacher_soft_test)

np.save(f"{predictions_path}/Y_train_hard.npy", Y_train_np)
np.save(f"{predictions_path}/Y_val_hard.npy", Y_val_np)
np.save(f"{predictions_path}/Y_test_hard.npy", Y_test_comb_np)

## 10.3 Reload from Disk

Lets the distillation work resume without re-training the teacher.

In [ ]:
import numpy as np

teacher_soft_train = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/teacher_soft_train.npy")
teacher_soft_val = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/teacher_soft_val.npy")
teacher_soft_test = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/teacher_soft_test.npy")

Y_train_hard = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/Y_train_hard.npy")
Y_val_hard = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/Y_val_hard.npy")
Y_test_hard = np.load("/content/drive/MyDrive/outputs_10participants_70AU/predictions/Y_test_hard.npy")

### Shape check

Soft labels must be `(N, 7)` and hard labels `(N,)`, with matching `N`.

In [ ]:
print("teacher_soft_train:", teacher_soft_train.shape)
print("teacher_soft_val:", teacher_soft_val.shape)
print("teacher_soft_test:", teacher_soft_test.shape)

print("Y_train_hard:", Y_train_hard.shape)
print("Y_val_hard:", Y_val_hard.shape)
print("Y_test_hard:", Y_test_hard.shape)

In [ ]:
print(teacher_soft_train[0])

In [ ]:
emotion_names = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]

for emotion, prob in zip(emotion_names, teacher_soft_train[0]):
    print(f"{emotion}: {prob:.4f}")

# Part 11: Original vs. +10-Participant Teacher (head-to-head)

A controlled comparison of the two saved teacher checkpoints on the same three test sets, using the
full metric suite. This section also defines `compute_full_metrics()`, which earlier sections reuse.

## 11.1 Checkpoint Paths

The two teacher checkpoints being compared, plus the output directory for this section.

In [ ]:
from pathlib import Path

# Edit these two paths if your saved model filenames are different.
ORIGINAL_TEACHER_MODEL_PATH = Path("/content/drive/MyDrive/outputs_10participants_70AU/models/teacher-70AU-acted-natural-MoE_3experts_2gate-without_10participants.keras")
TEN_PARTICIPANT_TEACHER_MODEL_PATH = Path("/content/drive/MyDrive/outputs_10participants_70AU/models/teacher-70AU-acted-natural-MoE_3experts_2gate_with_10participants_data.keras")

# Save comparison outputs here.
output_path = Path("/content/drive/MyDrive/outputs_10participants_70AU")
predictions_path = output_path / "predictions"
comparison_path = predictions_path / "teacher_model_comparison"
comparison_path.mkdir(parents=True, exist_ok=True)

print("Original model:", ORIGINAL_TEACHER_MODEL_PATH, "exists=", ORIGINAL_TEACHER_MODEL_PATH.exists())
print("10-participant model:", TEN_PARTICIPANT_TEACHER_MODEL_PATH, "exists=", TEN_PARTICIPANT_TEACHER_MODEL_PATH.exists())
print("Saving comparison outputs to:", comparison_path)


## 11.2 Load Both Teachers

Both are loaded by rebuilding the architecture and restoring weights (see the note in Section 8.2).

In [ ]:
def load_teacher_model(model_path, input_shape=(SEQ_LENGTH, input_size), num_classes=num_classes):
    # Rebuild the exact same architecture used at training time
    model, _ = build_moe_single_input(
        input_shape=input_shape, num_classes=num_classes,
        tau_init=2.0, tau_final=0.7,
        gate_hidden=128, classifier_hidden=256,
        lb_coef=0.02, ent_coef=0.01,
    )
    # Load only the weights, skip reconstructing the saved Functional graph
    model.load_weights(model_path)
    return model

original_teacher_model = load_teacher_model(ORIGINAL_TEACHER_MODEL_PATH)
teacher_plus_10_model = load_teacher_model(TEN_PARTICIPANT_TEACHER_MODEL_PATH)

MODELS_TO_COMPARE = {
    "original_teacher": original_teacher_model,
    "teacher_plus_10participants": teacher_plus_10_model,
}

## 11.3 Full Metric Suite

`compute_full_metrics()` runs a model on one test set and returns everything needed downstream
without re-predicting: scalar metrics (accuracy, balanced accuracy, macro/weighted/micro precision,
recall and F1, Cohen's kappa, MCC, top-2 accuracy, log loss, one-vs-rest ROC-AUC and PR-AUC),
a per-class table, raw and row-normalised confusion matrices, and the predictions and probabilities
themselves.

Metrics that are undefined for a given test set (e.g. ROC-AUC with a single class present) are
skipped with a warning rather than raising.

**Note:** Sections 7.5 and 8.5 call this function, so it must be defined before them.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    top_k_accuracy_score,
)
from sklearn.preprocessing import label_binarize


def compute_full_metrics(model, X, y_true, class_labels, class_names, model_name="model", test_name="test"):
    """
    Run a model on (X, y_true) and compute the full metric suite.
    Returns a dict with scalar metrics, per-class metrics, confusion matrices,
    and the raw predictions/probabilities (so plots can reuse them without re-predicting).
    """
    y_true = np.reshape(np.asarray(y_true), -1)
    y_proba = model.predict(X, batch_size=64, verbose=0)
    y_proba = np.asarray(y_proba)
    y_pred = np.argmax(y_proba, axis=1)

    n_classes = len(class_labels)

    # --- Basic ---
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # --- Precision/Recall/F1 (aggregate) ---
    prec_macro = precision_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=class_labels, average="macro", zero_division=0)

    prec_weighted = precision_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, labels=class_labels, average="weighted", zero_division=0)

    prec_micro = precision_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)
    rec_micro = recall_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, labels=class_labels, average="micro", zero_division=0)

    # --- Agreement metrics ---
    kappa = cohen_kappa_score(y_true, y_pred, labels=class_labels)
    mcc = matthews_corrcoef(y_true, y_pred)

    # --- Top-2 accuracy ---
    present_labels = np.unique(y_true)
    try:
        top2_acc = top_k_accuracy_score(y_true, y_proba, k=2, labels=class_labels)
    except Exception as e:
        top2_acc = np.nan
        print(f"[{model_name} | {test_name}] top-2 accuracy skipped: {e}")

    # --- Log loss ---
    try:
        logloss = log_loss(y_true, y_proba, labels=class_labels)
    except Exception as e:
        logloss = np.nan
        print(f"[{model_name} | {test_name}] log loss skipped: {e}")

    # --- ROC-AUC and PR-AUC (One-vs-Rest); only well-defined if >=2 classes present in y_true ---
    roc_auc_macro = roc_auc_score_safe = np.nan
    roc_auc_weighted = np.nan
    pr_auc_macro = np.nan
    pr_auc_weighted = np.nan
    if len(present_labels) >= 2:
        y_true_bin = label_binarize(y_true, classes=class_labels)
        try:
            roc_auc_macro = roc_auc_score(y_true_bin, y_proba, average="macro", multi_class="ovr")
            roc_auc_weighted = roc_auc_score(y_true_bin, y_proba, average="weighted", multi_class="ovr")
        except Exception as e:
            print(f"[{model_name} | {test_name}] ROC-AUC skipped: {e}")
        try:
            pr_auc_macro = average_precision_score(y_true_bin, y_proba, average="macro")
            pr_auc_weighted = average_precision_score(y_true_bin, y_proba, average="weighted")
        except Exception as e:
            print(f"[{model_name} | {test_name}] PR-AUC skipped: {e}")
    else:
        print(f"[{model_name} | {test_name}] Only one class present in y_true; ROC-AUC/PR-AUC undefined, skipped.")

    # --- Per-class report ---
    report_dict = classification_report(
        y_true, y_pred, labels=class_labels, target_names=class_names,
        output_dict=True, zero_division=0,
    )
    per_class_df = pd.DataFrame(report_dict).T.loc[class_names]
    per_class_df.index.name = "class"

    # --- Confusion matrices ---
    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    with np.errstate(all="ignore"):
        cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    scalar_metrics = {
        "model": model_name,
        "test_set": test_name,
        "n_samples": int(len(y_true)),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": prec_micro,
        "recall_micro": rec_micro,
        "f1_micro": f1_micro,
        "cohen_kappa": kappa,
        "matthews_corrcoef": mcc,
        "top2_accuracy": top2_acc,
        "log_loss": logloss,
        "roc_auc_macro_ovr": roc_auc_macro,
        "roc_auc_weighted_ovr": roc_auc_weighted,
        "pr_auc_macro_ovr": pr_auc_macro,
        "pr_auc_weighted_ovr": pr_auc_weighted,
    }

    return {
        "scalar_metrics": scalar_metrics,
        "per_class_df": per_class_df,
        "confusion_matrix": cm,
        "confusion_matrix_normalized": cm_norm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


print("compute_full_metrics() defined.")


## 11.4 Define the Test Sets

Three fixed test sets — natural, acted + EmojiHeroVR, combined — plus the new-participant test set if
it exists in the current session.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

class_names = [emo.value for emo in Emotion]
class_labels = list(range(len(class_names)))

TEST_SETS = {
    "old_natural": (X_test_nat_np, Y_test_nat_np),
    "old_acted_plus_EmojiHero": (X_test_act_comb_np, Y_test_act_comb_np),
    "all_combined": (X_test_comb_np, Y_test_comb_np),
}

if "X_test_new_np" in globals() and "Y_test_new_np" in globals():
    TEST_SETS["new_10participants"] = (X_test_new_np, Y_test_new_np)
elif "X_test_new" in globals() and "Y_test_new" in globals():
    X_test_new_np = stack_sequences(X_test_new)
    Y_test_new_np = to_numpy_labels(Y_test_new)
    TEST_SETS["new_10participants"] = (X_test_new_np, Y_test_new_np)

for test_name, (X, y) in TEST_SETS.items():
    print(test_name, X.shape, y.shape)

## 11.5 Evaluate Every (Model, Test Set) Pair

In [ ]:
all_results = {}  # all_results[(model_name, test_name)] = result dict from compute_full_metrics

for model_name, model in MODELS_TO_COMPARE.items():
    for test_name, (X, y) in TEST_SETS.items():
        print(f"Evaluating {model_name} on {test_name} ...")
        result = compute_full_metrics(
            model, X, y,
            class_labels=class_labels,
            class_names=class_names,
            model_name=model_name,
            test_name=test_name,
        )
        all_results[(model_name, test_name)] = result
        print(
            f"  acc={result['scalar_metrics']['accuracy']:.4f}  "
            f"bal_acc={result['scalar_metrics']['balanced_accuracy']:.4f}  "
            f"f1_macro={result['scalar_metrics']['f1_macro']:.4f}"
        )

print("\nDone. Collected", len(all_results), "(model, test_set) evaluations.")

## 11.6 Scalar Metric Tables

Writes three views of the same results: long format (one row per model/test set), wide format (one
row per metric), and a delta table giving `teacher_plus_10participants - original_teacher` for every
metric on every test set.

In [ ]:
scalar_rows = [res["scalar_metrics"] for res in all_results.values()]
scalar_df = pd.DataFrame(scalar_rows).sort_values(["test_set", "model"]).reset_index(drop=True)

display(scalar_df)

scalar_df.to_csv(comparison_path / "scalar_metrics_all_models_all_testsets.csv", index=False)
print("Saved:", comparison_path / "scalar_metrics_all_models_all_testsets.csv")

# ---- Wide format: one row per metric, one column per (model, test_set) for easy side-by-side reading ----
wide_df = scalar_df.set_index(["test_set", "model"]).drop(columns=["n_samples"]).T
wide_df.to_csv(comparison_path / "scalar_metrics_wide.csv")
display(wide_df)

# ---- Delta table: original vs +10participants, per test set, per metric ----
model_a, model_b = "original_teacher", "teacher_plus_10participants"
metric_cols = [c for c in scalar_df.columns if c not in ("model", "test_set", "n_samples")]

delta_rows = []
for test_name in TEST_SETS.keys():
    row_a = scalar_df[(scalar_df.model == model_a) & (scalar_df.test_set == test_name)]
    row_b = scalar_df[(scalar_df.model == model_b) & (scalar_df.test_set == test_name)]
    if row_a.empty or row_b.empty:
        continue
    row_a = row_a.iloc[0]
    row_b = row_b.iloc[0]
    for metric in metric_cols:
        delta_rows.append({
            "test_set": test_name,
            "metric": metric,
            model_a: row_a[metric],
            model_b: row_b[metric],
            "delta (b - a)": row_b[metric] - row_a[metric],
        })

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(comparison_path / "metric_deltas_original_vs_10participants.csv", index=False)
print("\nSaved:", comparison_path / "metric_deltas_original_vs_10participants.csv")
display(delta_df)

## 11.7 Per-Class Metric Tables

Per-class precision, recall, F1, and support for both models, plus a side-by-side per-class F1 pivot
with deltas. This is where a change in overall macro-F1 can be traced to the specific emotions that
moved.

In [ ]:
per_class_rows = []
for (model_name, test_name), res in all_results.items():
    df = res["per_class_df"].copy()
    df = df.reset_index().rename(columns={"index": "class"})
    df["model"] = model_name
    df["test_set"] = test_name
    per_class_rows.append(df)

per_class_all_df = pd.concat(per_class_rows, ignore_index=True)
per_class_all_df = per_class_all_df[["test_set", "class", "model", "precision", "recall", "f1-score", "support"]]
per_class_all_df = per_class_all_df.sort_values(["test_set", "class", "model"]).reset_index(drop=True)

per_class_all_df.to_csv(comparison_path / "per_class_metrics_all_models_all_testsets.csv", index=False)
print("Saved:", comparison_path / "per_class_metrics_all_models_all_testsets.csv")
display(per_class_all_df)

# Pivot: per-class F1, side by side for the two models, one table per test set
for test_name in TEST_SETS.keys():
    sub = per_class_all_df[per_class_all_df.test_set == test_name]
    pivot = sub.pivot(index="class", columns="model", values="f1-score")
    if model_a in pivot.columns and model_b in pivot.columns:
        pivot["delta (b - a)"] = pivot[model_b] - pivot[model_a]
    print(f"\n--- Per-class F1 | test_set = {test_name} ---")
    display(pivot)

## 11.8 Confusion Matrices

Raw and row-normalised confusion matrices for both models on each test set, plotted side by side and
saved as PNG. Normalised matrices are read row-wise: what fraction of each true emotion went where.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_pair(test_name, normalize=False):
    res_a = all_results[(model_a, test_name)]
    res_b = all_results[(model_b, test_name)]

    cm_a = res_a["confusion_matrix_normalized"] if normalize else res_a["confusion_matrix"]
    cm_b = res_b["confusion_matrix_normalized"] if normalize else res_b["confusion_matrix"]

    fmt = ".2f" if normalize else "d"
    vmax = max(cm_a.max(), cm_b.max())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(cm_a, annot=True, fmt=fmt, cmap="Blues", vmin=0, vmax=vmax,
                xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
    axes[0].set_title(f"{model_a}\n{test_name}")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(cm_b, annot=True, fmt=fmt, cmap="Blues", vmin=0, vmax=vmax,
                xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
    axes[1].set_title(f"{model_b}\n{test_name}")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")

    plt.tight_layout()
    suffix = "normalized" if normalize else "raw"
    fname = comparison_path / f"confusion_matrix_{suffix}_{test_name}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fname)


for test_name in TEST_SETS.keys():
    plot_confusion_pair(test_name, normalize=False)
    plot_confusion_pair(test_name, normalize=True)

## 11.9 Per-Class F1 Bar Charts

In [ ]:
for test_name in TEST_SETS.keys():
    sub = per_class_all_df[per_class_all_df.test_set == test_name]
    pivot = sub.pivot(index="class", columns="model", values="f1-score").reindex(class_names)

    x = np.arange(len(class_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, pivot[model_a], width, label=model_a)
    ax.bar(x + width / 2, pivot[model_b], width, label=model_b)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.set_ylabel("F1 score")
    ax.set_ylim(0, 1)
    ax.set_title(f"Per-class F1: {model_a} vs {model_b}\ntest_set = {test_name}")
    ax.legend()
    plt.tight_layout()

    fname = comparison_path / f"per_class_f1_{test_name}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fname)

## 11.10 Key Metric Bar Charts

Six headline metrics for both models across all test sets in one figure.

In [ ]:
key_metrics = ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted", "cohen_kappa", "matthews_corrcoef"]

n_tests = len(TEST_SETS)
fig, axes = plt.subplots(1, n_tests, figsize=(6 * n_tests, 5), sharey=True)
if n_tests == 1:
    axes = [axes]

for ax, test_name in zip(axes, TEST_SETS.keys()):
    row_a = scalar_df[(scalar_df.model == model_a) & (scalar_df.test_set == test_name)].iloc[0]
    row_b = scalar_df[(scalar_df.model == model_b) & (scalar_df.test_set == test_name)].iloc[0]

    x = np.arange(len(key_metrics))
    width = 0.35
    ax.bar(x - width / 2, [row_a[m] for m in key_metrics], width, label=model_a)
    ax.bar(x + width / 2, [row_b[m] for m in key_metrics], width, label=model_b)
    ax.set_xticks(x)
    ax.set_xticklabels(key_metrics, rotation=45, ha="right")
    ax.set_title(test_name)
    ax.set_ylim(min(0, scalar_df[key_metrics].min().min() - 0.05), 1)
    ax.axhline(0, color="black", linewidth=0.8)

axes[0].set_ylabel("score")
axes[0].legend()
fig.suptitle(f"Key metrics: {model_a} vs {model_b}", y=1.02)
plt.tight_layout()

fname = comparison_path / "key_metrics_comparison_all_testsets.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", fname)

## 11.11 Save Raw Predictions and Final Verdict

Persists every model's predictions and probabilities as `.npz` so that any metric can be recomputed
later without re-running the models, writes the scalar summary as JSON, and prints a per-metric
winner for each test set.

In [ ]:
# ---- Save raw predictions/probabilities for every (model, test_set) so results are reproducible later ----
raw_dir = comparison_path / "raw_predictions"
raw_dir.mkdir(exist_ok=True)

for (model_name, test_name), res in all_results.items():
    np.savez(
        raw_dir / f"{model_name}__{test_name}.npz",
        y_true=res["y_true"],
        y_pred=res["y_pred"],
        y_proba=res["y_proba"],
        confusion_matrix=res["confusion_matrix"],
    )

print("Saved raw predictions/probabilities to:", raw_dir)

# ---- JSON summary (scalar metrics only, easy to diff or re-load) ----
summary_json = {
    "class_names": class_names,
    "models_compared": list(MODELS_TO_COMPARE.keys()),
    "test_sets": {name: {"n_samples": int(len(y))} for name, (X, y) in TEST_SETS.items()},
    "scalar_metrics": scalar_df.to_dict(orient="records"),
}
with open(comparison_path / "summary.json", "w") as f:
    json.dump(summary_json, f, indent=2, default=float)
print("Saved:", comparison_path / "summary.json")

# ---- Final printed summary: which model wins on which metric, per test set ----
print("\n" + "=" * 70)
print("FINAL SUMMARY:", model_a, "vs", model_b)
print("=" * 70)
for test_name in TEST_SETS.keys():
    row_a = scalar_df[(scalar_df.model == model_a) & (scalar_df.test_set == test_name)].iloc[0]
    row_b = scalar_df[(scalar_df.model == model_b) & (scalar_df.test_set == test_name)].iloc[0]
    print(f"\n[{test_name}]  (n={int(row_a['n_samples'])})")
    for metric in key_metrics:
        a_val, b_val = row_a[metric], row_b[metric]
        winner = model_b if b_val > a_val else (model_a if a_val > b_val else "tie")
        print(f"  {metric:22s} {model_a}={a_val:.4f}  {model_b}={b_val:.4f}   -> better: {winner}")

print("\nAll comparison artifacts (CSV/JSON/PNG/raw predictions) saved under:")
print(" ", comparison_path)

## 11.12 Diagnostics

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/outputs_10participants_70AU/models"))

### Prediction distribution check

Counts how many windows each model assigns to each emotion. A model that collapses onto one or two
majority classes will show it immediately here, even when its accuracy looks acceptable.

In [ ]:
import numpy as np

ORIGINAL_NO10_PATH = "/content/drive/MyDrive/outputs_10participants_70AU/models/teacher-70AU-acted-natural-MoE_3experts_2gate-without_10participants.keras"
WITH10_PATH = "/content/drive/MyDrive/outputs_10participants_70AU/models/teacher-70AU-acted-natural-MoE_3experts_2gate_with_10participants_data.keras"

model_no10 = load_teacher_model(ORIGINAL_NO10_PATH, input_shape=(SEQ_LENGTH, 70))
model_with10 = load_teacher_model(WITH10_PATH, input_shape=(SEQ_LENGTH, 70))

emo_names = [emo.value for emo in Emotion]

for name, m in [("original_teacher (no 10p)", model_no10), ("teacher_plus_10participants", model_with10)]:
    preds = m.predict(X_test_comb_np)
    pred_labels = np.argmax(preds, axis=1)
    counts = np.bincount(pred_labels, minlength=len(Emotion))
    print(f"\n{name}")
    for emo, c in zip(emo_names, counts):
        print(f"  {emo:12s}: {c}")
    print(f"  total: {counts.sum()}")